**Notebook 03 — Prediction model development and validation**

- Paper: *Risk prediction of 72-hour fatality in panic-level hyperuricemia using machine learning and logistic regression* (PLOS ONE, under review)
- Authors: Hayashi R, Hayashi S, Sueda K, et al.
- Outputs: S11 Table (Table 4 is its five-model subset), S3 and S12–S20 Tables; Figs 3–8; S3 and S6–S9 Figs
- Environment: Google Colaboratory (Python 3.13.16); library versions are pinned and checked in Cell 0
- Patient data are not included in this repository. Data access requests should be directed to the corresponding author (hsmdfacp@jichi.ac.jp).

# Prediction of 72-h Fatality Using MLRA, 8 Gradient Boosting Models, and Prediction One™
---
**Sections (in execution order) and manuscript outputs**
1. Environment Setup & Imports
2. Reproducibility Settings
3. Data Loading; 3b. Prediction One™ Data Loading
4. Data Preprocessing
5. MLRA Implementation (final four-predictor formula)
6. Gradient Boosting Model Definitions (S10 Table)
7. Model Training
8. Evaluation Metrics with DeLong 95% CIs — S11 Table (Table 4 subset)
9. ROC Curves — Fig 3
10. Calibration (logistic recalibration) — Fig 4, S13 Table, S14 Table
11. Decision Curve Analysis — S16 Table, Fig 6
12. SHAP Analysis — Fig 8 (bar plot + beeswarm, `Fig8.tif`), S9 Fig
17. Subgroup Analysis — S17 Table, S6 Fig
18. Missing Data Analysis — S3 Table, S3 Fig, S7 Fig
20. Fairness Analysis — S20 Table, S8 Fig
22. Outlier Influence Analysis — S19 Table
21. Threshold Variation and Sensitivity Figure — Fig 7, S18 Table
23. IDI & NRI — S15 Table, Fig 5
24. Bootstrap Internal Validation (Harrell's optimism correction) — S12 Table

Final cell: PLOS ONE TIFF export of Figs 3–7 and S3, S6–S9 Figs

In [ ]:
# ============================================================
# Cell 0: Environment setup — pin library versions for reproducibility
# Python itself cannot be pinned in Google Colaboratory;
# the version used for the reported analyses was Python 3.13.16.
# NumPy and pandas are preinstalled Colab dependencies and are reported, not pinned.
# ============================================================
!pip install -q xgboost==3.2.0 lightgbm==4.6.0 catboost==1.2.10 scikit-learn==1.6.1 shap==0.54.1 statsmodels==0.15.0

import sys
import importlib

EXPECTED = {
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "catboost": "1.2.10",
    "sklearn": "1.6.1",
    "shap": "0.54.1",
    "statsmodels": "0.15.0",
}

print("=" * 55)
print("Library versions")
print("=" * 55)
print(f"{'Python':<13}: {sys.version.split()[0]}")
mismatch = []
for mod, ver in EXPECTED.items():
    m = importlib.import_module(mod)
    label = "scikit-learn" if mod == "sklearn" else mod
    print(f"{label:<13}: {m.__version__}")
    if m.__version__ != ver:
        mismatch.append(f"{label} {m.__version__} (expected {ver})")
for mod in ("numpy", "pandas"):
    print(f"{mod:<13}: {importlib.import_module(mod).__version__}  (reported, not pinned)")
print("=" * 55)

if sys.version.split()[0] != "3.13.16":
    print(f"NOTE: Python {sys.version.split()[0]} differs from the reported 3.13.16.")
if mismatch:
    raise RuntimeError("Version mismatch — restart the runtime "
                       "(Runtime > Restart session) and rerun this cell: " + "; ".join(mismatch))

## Section 1 · Environment Setup & Imports

In [ ]:
# ============================================================
# Section 1: Environment Setup & Imports
# Library versions are pinned in Cell 0a and verified in Cell 0b.
# ============================================================

# Packages not included in the standard Colab environment
# (catboost and shap are installed with pinned versions in Cell 0a)
!pip install python-pptx openpyxl -q

import os
import sys
import random
import time
import io
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np

# Scikit-learn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, log_loss, matthews_corrcoef,
    brier_score_loss, auc, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier

# Gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Visualisation
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Rectangle
import seaborn as sns

# Statistics
from scipy import stats
from scipy.interpolate import interp1d
import string

# Progress bar
from tqdm import tqdm

# Google Colab utilities
from google.colab import files
from pptx import Presentation
from pptx.util import Inches

# SHAP and statsmodels (version reporting)
import shap
import statsmodels

# ── Version check ─────────────────────────────────────────
print("=" * 55)
print("Library versions")
print("=" * 55)
print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"XGBoost     : {xgb.__version__}")
print(f"LightGBM    : {lgb.__version__}")
print(f"CatBoost    : {cb.__version__}")
import sklearn; print(f"scikit-learn: {sklearn.__version__}")
print(f"SHAP        : {shap.__version__}")
print(f"statsmodels : {statsmodels.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")
print(f"seaborn     : {sns.__version__}")
print("=" * 55)
print("✓ All imports successful")

## Section 2 · Reproducibility Settings

In [ ]:
# ============================================================
# Section 2: Reproducibility Settings
# All random seeds are preserved exactly as in the original code.
# ============================================================

RANDOM_STATE = 42   # Master seed — do NOT modify

def set_global_seed(seed: int = RANDOM_STATE) -> None:
    """
    Enforce full determinism across Python, NumPy, and all
    gradient-boosting libraries.  Must be called before each
    model fit to guarantee bit-identical results across runs.

    Parameters
    ----------
    seed : int
        Random seed value. Default: 42 (original value preserved).
    """
    os.environ['PYTHONHASHSEED']       = str(seed)
    os.environ['OMP_NUM_THREADS']      = '1'
    os.environ['MKL_NUM_THREADS']      = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    os.environ['NUMEXPR_NUM_THREADS']  = '1'
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)

set_global_seed(RANDOM_STATE)

# ── Consistent colour palette (shared across all figures) ──
MODEL_COLORS = {
    'MLRA'               : '#000000',   # Black
    'XGBoost_Standard'   : '#0072B2',   # Blue (colour-blind safe)
    'XGBoost_DART'       : '#56B4E9',   # Sky blue
    'LightGBM_GBDT'      : '#009E73',   # Green
    'LightGBM_DART'      : '#7FBF7F',   # Light green
    'LightGBM_GOSS'      : '#D55E00',   # Vermillion
    'CatBoost_Standard'  : '#E69F00',   # Orange
    'CatBoost_Ordered'   : '#CC79A7',   # Pink
    'HistGradientBoosting': '#9467BD',  # Purple
    'Prediction_One'     : '#8B4513',   # Saddle brown
}

print(f"✓ Global seed set: RANDOM_STATE = {RANDOM_STATE}")
print(f"✓ MODEL_COLORS defined for {len(MODEL_COLORS)} models")


## Section 3 · Data Loading *(code unchanged)*

In [ ]:
# ============================================================
# Section 3: Data Loading
# ─────────────────────────────────────────────────────────────
# IMPORTANT: This block is preserved exactly as in the original.
# Only English comments have been added; no logic was modified.
# ─────────────────────────────────────────────────────────────

# Upload training data (G1 2019)
print("\nPlease upload: UA G1 2019_Training for Claude.csv")
uploaded = files.upload()
train_df = pd.read_csv(list(uploaded.keys())[0])

# Upload validation data (G2 2018–2017)
print("\nPlease upload: UA G2 2018_2017 Validation for Claude.csv")
uploaded = files.upload()
val_df = pd.read_csv(list(uploaded.keys())[0])

# ── Basic data inspection ──────────────────────────────────
print("\n" + "=" * 50)
print("Dataset overview")
print("=" * 50)
print(f"Training set shape  : {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")

# Column-name inventory
print("\nTraining set columns:")
for i, col in enumerate(train_df.columns):
    print(f"  {i}: '{col}'")
print("\nValidation set columns:")
for i, col in enumerate(val_df.columns):
    print(f"  {i}: '{col}'")

print("\nLast column (training) :", repr(train_df.columns[-1]))
print("Last column (validation):", repr(val_df.columns[-1]))

# Remove columns with blank or NaN names
train_df = train_df.loc[:, train_df.columns.notna() &
                            (train_df.columns != '') &
                            (train_df.columns.str.strip() != '')]
val_df   = val_df.loc[:,   val_df.columns.notna() &
                            (val_df.columns != '') &
                            (val_df.columns.str.strip() != '')]

print(f"\nAfter empty-column removal:")
print(f"  Training  : {train_df.shape}")
print(f"  Validation: {val_df.shape}")

# Column alignment check
if list(train_df.columns) == list(val_df.columns):
    print("\n✓ Column names match between datasets")
else:
    print("\n✗ Column name mismatch — check below:")
    for i, (tc, vc) in enumerate(zip(train_df.columns, val_df.columns)):
        if tc != vc:
            print(f"  Index {i}: training='{tc}' vs validation='{vc}'")

# Outcome variable
print(f"\nOutcome variable : {train_df.columns[0]}")
print(f"Features         : {len(train_df.columns) - 1}")

print("\nOutcome distribution (training):")
print(train_df['Outcome'].value_counts())
print("\nOutcome distribution (validation):")
print(val_df['Outcome'].value_counts())

# Missing value report
print("\nMissing values — training set:")
miss_train = train_df.isnull().sum()
miss_train_pct = miss_train[miss_train > 0] / len(train_df) * 100
if len(miss_train_pct):
    print(pd.DataFrame({'Count': miss_train[miss_train > 0],
                        'Pct (%)': miss_train_pct})
            .sort_values('Pct (%)', ascending=False).head(10))
else:
    print("  None")

print("\nMissing values — validation set:")
miss_val = val_df.isnull().sum()
miss_val_pct = miss_val[miss_val > 0] / len(val_df) * 100
if len(miss_val_pct):
    print(pd.DataFrame({'Count': miss_val[miss_val > 0],
                        'Pct (%)': miss_val_pct})
            .sort_values('Pct (%)', ascending=False).head(10))
else:
    print("  None")

print("\n✓ Data loading complete")


## Section 3b · Prediction One™ Data Loading *(code unchanged)*

In [ ]:
# ============================================================
# Section 3b: Prediction One™ Data Loading
# Loads external AutoML predictions for integration with
# model comparison pipeline.
# ─────────────────────────────────────────────────────────────
# IMPORTANT: This block is preserved exactly as in the original.
# ─────────────────────────────────────────────────────────────

set_global_seed(RANDOM_STATE)

print("=" * 70)
print("Prediction One™ Data Integration")
print("=" * 70)
print("\nPlease upload both files:")
print("  1. UA G1 2019 Predictive Value Prediction One.csv")
print("  2. UA G2 2018_2017 Predictive Value Prediction One.csv")

uploaded = files.upload()
print(f"\n✓ Uploaded: {list(uploaded.keys())}")

predone_train_df = None
predone_val_df   = None

for filename in uploaded.keys():
    if "G1 2019" in filename and "Predictive Value Prediction One" in filename:
        try:
            predone_train_df = pd.read_csv(io.BytesIO(uploaded[filename]),
                                           encoding='utf-8')
        except UnicodeDecodeError:
            predone_train_df = pd.read_csv(io.BytesIO(uploaded[filename]),
                                           encoding='shift-jis')
        print(f"\n✓ Training data loaded: {filename}  {predone_train_df.shape}")

    elif "G2 2018_2017" in filename and "Predictive Value Prediction One" in filename:
        try:
            predone_val_df = pd.read_csv(io.BytesIO(uploaded[filename]),
                                         encoding='utf-8')
        except UnicodeDecodeError:
            predone_val_df = pd.read_csv(io.BytesIO(uploaded[filename]),
                                         encoding='shift-jis')
        print(f"\n✓ Validation data loaded: {filename}  {predone_val_df.shape}")

if predone_train_df is None:
    raise FileNotFoundError("G1 2019 Prediction One™ file not found.")
if predone_val_df is None:
    raise FileNotFoundError("G2 2018_2017 Prediction One™ file not found.")

# Extract predicted probabilities and true labels
# Column index 1 (B): predicted probability for class 1
# Column index 7 (H): Outcome (true label)
predone_train_proba = predone_train_df.iloc[:, 1].values.astype(float)
predone_train_true  = predone_train_df.iloc[:, 7].values.astype(int)
predone_val_proba   = predone_val_df.iloc[:, 1].values.astype(float)
predone_val_true    = predone_val_df.iloc[:, 7].values.astype(int)

# Integrity checks
assert np.all((predone_train_proba >= 0) & (predone_train_proba <= 1)), \
    "Training probabilities out of [0, 1] range"
assert np.all((predone_val_proba >= 0)   & (predone_val_proba <= 1)), \
    "Validation probabilities out of [0, 1] range"
assert np.all(np.isin(predone_train_true, [0, 1])), "Training labels not binary"
assert np.all(np.isin(predone_val_true,   [0, 1])), "Validation labels not binary"

train_auc_predone = roc_auc_score(predone_train_true, predone_train_proba)
val_auc_predone   = roc_auc_score(predone_val_true,   predone_val_proba)
print(f"\nPrediction One™  Train AUC: {train_auc_predone:.4f}")
print(f"Prediction One™  Val   AUC: {val_auc_predone:.4f}")
print("✓ Prediction One™ data integrity checks passed")
# ── Verification: Prediction One™ AUC values must match original run ──


## Section 4 · Data Preprocessing & Feature Engineering

In [ ]:
# ============================================================
# Section 4: Data Preprocessing & Feature Engineering
# ============================================================

def preprocess_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    Convert object columns to numeric; leave other columns unchanged.
    This is the original preprocessing logic, unchanged.
    """
    X_proc = X.copy()
    for col in X_proc.columns:
        if X_proc[col].dtype == 'object':
            X_proc[col] = pd.to_numeric(X_proc[col], errors='coerce')
    return X_proc

# Align columns between training and validation sets
train_cols  = set(train_df.columns)
val_cols    = set(val_df.columns)
common_cols = sorted(train_cols & val_cols)

if 'Outcome' not in common_cols:
    raise ValueError("'Outcome' column not found in common columns")

train_df_aligned = train_df[common_cols]
val_df_aligned   = val_df[common_cols]

X_train = train_df_aligned.drop('Outcome', axis=1)
y_train = train_df_aligned['Outcome']
X_val   = val_df_aligned.drop('Outcome', axis=1)
y_val   = val_df_aligned['Outcome']

X_train = preprocess_features(X_train)
X_val   = preprocess_features(X_val)

# Identify categorical features (Sex column only, as in original)
categorical_features = [i for i, col in enumerate(X_train.columns) if col == 'Sex']
print(f"Categorical features: {[X_train.columns[i] for i in categorical_features]}")
print(f"Training   — X: {X_train.shape}, y: {y_train.shape}")
print(f"Validation — X: {X_val.shape},   y: {y_val.shape}")
print("✓ Feature engineering complete")


In [ ]:
# ============================================================
# Section 4: Verification — Column Alignment Check
# ============================================================

dropped_from_train = train_cols - val_cols - {'Outcome'}
dropped_from_val   = val_cols - train_cols - {'Outcome'}

print(f"Dropped from train (absent in validation): {dropped_from_train}")
print(f"Dropped from validation (absent in train): {dropped_from_val}")
print(f"Final feature count: {X_train.shape[1]}")

## Section 5 · MLRA Implementation

In [ ]:
# ============================================================
# Section 5: Multiple Logistic Regression Analysis (MLRA)
# Pre-specified prediction formula applied to 4 clinical variables.
# ============================================================

MLRA_FEATURES = [
    'Total protein (g/dL)',
    'Direct bilirubin (mg/dL)',
    'White blood cells (103/\u03bcL)',
    'C-reactive protein (mg/dL)',
]

EPSILON = 1e-10   # Prevents log(0)


def prepare_mlra_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply listwise deletion for the 4 MLRA predictor variables.
    Rows with any missing value among the 4 predictors are dropped.
    """
    df_sub = df[['Outcome'] + MLRA_FEATURES].copy()
    df_clean = df_sub.dropna(subset=MLRA_FEATURES, how='any')
    print(f"  Original N  : {len(df)}")
    print(f"  After deletion: {len(df_clean)}")
    print(f"  Rows removed: {len(df) - len(df_clean)}")
    return df_clean


def apply_mlra_formula(df: pd.DataFrame) -> np.ndarray:
    """
    Apply the pre-specified MLRA logistic formula:

        log-odds = 2.053
                   − 0.8872 × Total protein
                   + 0.7692 × log(Direct bilirubin)
                   + 0.5063 × log(WBC)
                   + 0.3229 × log(CRP)

    Returns predicted probabilities.
    """
    log_odds = (
        2.053
        - 0.8872 * df['Total protein (g/dL)']
        + 0.7692 * np.log(df['Direct bilirubin (mg/dL)'] + EPSILON)
        + 0.5063 * np.log(df['White blood cells (103/\u03bcL)']  + EPSILON)
        + 0.3229 * np.log(df['C-reactive protein (mg/dL)']       + EPSILON)
    )
    return 1.0 / (1.0 + np.exp(-log_odds))


print("=" * 50)
print("Training set — MLRA data preparation")
train_mlra = prepare_mlra_data(train_df)

print("\nValidation set — MLRA data preparation")
val_mlra = prepare_mlra_data(val_df)

# Predicted probabilities
train_mlra_proba = apply_mlra_formula(train_mlra)
val_mlra_proba   = apply_mlra_formula(val_mlra)

train_mlra_pred = (train_mlra_proba >= 0.5).astype(int)
val_mlra_pred   = (val_mlra_proba   >= 0.5).astype(int)

train_mlra_auc = roc_auc_score(train_mlra['Outcome'], train_mlra_proba)
val_mlra_auc   = roc_auc_score(val_mlra['Outcome'],   val_mlra_proba)

print(f"\nMLRA Train AUC: {train_mlra_auc:.4f}")
print(f"MLRA Val   AUC: {val_mlra_auc:.4f}")

# Store results in unified dictionary
mlra_results = {
    'train_proba': train_mlra_proba,
    'train_true' : train_mlra['Outcome'],
    'val_proba'  : val_mlra_proba,
    'val_true'   : val_mlra['Outcome'],
    'model_name' : 'MLRA',
}

print("\n✓ MLRA implementation complete")
# ── Verification: MLRA Train AUC and Val AUC must match original run ──


## Section 6 · ML Model Definitions (Hyperparameters)

In [ ]:
# ============================================================
# Section 6: Machine Learning Model Definitions
# ─────────────────────────────────────────────────────────────
# CRITICAL: All hyperparameters are preserved exactly as
#           specified in the original code.  Do NOT modify.
# ─────────────────────────────────────────────────────────────

N_ESTIMATORS           = 100   # Default n_estimators (used by several models)
EARLY_STOPPING_ROUNDS  = 20

# ── 1. XGBoost Standard ───────────────────────────────────
xgb_standard_params = {
    'n_estimators'      : 80,
    'max_depth'         : 3,
    'learning_rate'     : 0.08,
    'subsample'         : 0.7,
    'colsample_bytree'  : 0.7,
    'min_child_weight'  : 10,
    'gamma'             : 0.2,
    'reg_alpha'         : 0.5,
    'reg_lambda'        : 2.0,
    'objective'         : 'binary:logistic',
    'eval_metric'       : 'auc',
    'random_state'      : RANDOM_STATE,
    'seed'              : RANDOM_STATE,
    'nthread'           : 1,
    'tree_method'       : 'hist',
    'device'            : 'cpu',
    'enable_categorical': False,
}

# ── 2. XGBoost DART ───────────────────────────────────────
xgb_dart_params = {
    'n_estimators'      : 80,
    'max_depth'         : 3,
    'learning_rate'     : 0.08,
    'subsample'         : 0.7,
    'colsample_bytree'  : 0.7,
    'min_child_weight'  : 10,
    'gamma'             : 0.2,
    'reg_alpha'         : 0.5,
    'reg_lambda'        : 2.0,
    'booster'           : 'dart',
    'rate_drop'         : 0.2,
    'skip_drop'         : 0.5,
    'sample_type'       : 'uniform',
    'normalize_type'    : 'tree',
    'objective'         : 'binary:logistic',
    'eval_metric'       : 'auc',
    'random_state'      : RANDOM_STATE,
    'seed'              : RANDOM_STATE,
    'nthread'           : 1,
    'tree_method'       : 'hist',
    'device'            : 'cpu',
    'enable_categorical': False,
}

# ── 3. LightGBM GBDT ──────────────────────────────────────
lgb_gbdt_params = {
    'n_estimators'         : 50,
    'max_depth'            : 2,
    'learning_rate'        : 0.05,
    'subsample'            : 0.5,
    'colsample_bytree'     : 0.5,
    'min_child_samples'    : 50,
    'reg_alpha'            : 1.0,
    'reg_lambda'           : 3.0,
    'min_split_gain'       : 0.1,
    'boosting_type'        : 'gbdt',
    'objective'            : 'binary',
    'metric'               : 'auc',
    'random_state'         : RANDOM_STATE,
    'seed'                 : RANDOM_STATE,
    'feature_fraction_seed': RANDOM_STATE,
    'bagging_seed'         : RANDOM_STATE,
    'data_random_seed'     : RANDOM_STATE,
    'verbose'              : -1,
    'deterministic'        : True,
    'force_row_wise'       : True,
    'num_threads'          : 1,
    'n_jobs'               : 1,
}

# ── 4. LightGBM DART ──────────────────────────────────────
lgb_dart_params = {
    'n_estimators'         : 50,
    'max_depth'            : 3,
    'learning_rate'        : 0.05,
    'subsample'            : 0.6,
    'colsample_bytree'     : 0.6,
    'min_child_samples'    : 30,
    'reg_alpha'            : 0.5,
    'reg_lambda'           : 2.0,
    'boosting_type'        : 'dart',
    'drop_rate'            : 0.2,
    'skip_drop'            : 0.5,
    'max_drop'             : 50,
    'uniform_drop'         : True,
    'objective'            : 'binary',
    'metric'               : 'auc',
    'random_state'         : RANDOM_STATE,
    'seed'                 : RANDOM_STATE,
    'feature_fraction_seed': RANDOM_STATE,
    'bagging_seed'         : RANDOM_STATE,
    'data_random_seed'     : RANDOM_STATE,
    'drop_seed'            : RANDOM_STATE,
    'verbose'              : -1,
    'deterministic'        : True,
    'force_row_wise'       : True,
    'num_threads'          : 1,
    'n_jobs'               : 1,
}

# ── 5. LightGBM GOSS ──────────────────────────────────────
lgb_goss_params = {
    'n_estimators'         : 40,
    'max_depth'            : 2,
    'learning_rate'        : 0.03,
    'colsample_bytree'     : 0.5,
    'min_child_samples'    : 50,
    'reg_alpha'            : 1.0,
    'reg_lambda'           : 3.0,
    'min_split_gain'       : 0.1,
    'boosting_type'        : 'goss',
    'subsample'            : 0.8,
    'objective'            : 'binary',
    'metric'               : 'auc',
    'random_state'         : RANDOM_STATE,
    'seed'                 : RANDOM_STATE,
    'feature_fraction_seed': RANDOM_STATE,
    'data_random_seed'     : RANDOM_STATE,
    'verbose'              : -1,
    'deterministic'        : True,
    'force_row_wise'       : True,
    'num_threads'          : 1,
    'n_jobs'               : 1,
}

# ── 6. CatBoost Standard ──────────────────────────────────
catboost_standard_params = {
    'iterations'          : 50,
    'depth'               : 3,
    'learning_rate'       : 0.05,
    'l2_leaf_reg'         : 10.0,
    'subsample'           : 0.8,
    'random_strength'     : 2.0,
    'border_count'        : 64,
    'min_data_in_leaf'    : 50,
    'grow_policy'         : 'SymmetricTree',
    'bootstrap_type'      : 'Bernoulli',
    'sampling_frequency'  : 'PerTree',
    'loss_function'       : 'Logloss',
    'eval_metric'         : 'AUC',
    'random_state'        : RANDOM_STATE,
    'verbose'             : False,
    'thread_count'        : 1,
    'used_ram_limit'      : '3gb',
    'allow_writing_files' : False,
}

# ── 7. CatBoost Ordered ───────────────────────────────────
catboost_ordered_params = {
    'iterations'          : N_ESTIMATORS,
    'depth'               : 4,
    'learning_rate'       : 0.1,
    'l2_leaf_reg'         : 3.0,
    'subsample'           : 0.8,
    'random_strength'     : 1.0,
    'border_count'        : 128,
    'grow_policy'         : 'SymmetricTree',
    'boosting_type'       : 'Ordered',
    'bootstrap_type'      : 'Bernoulli',
    'sampling_frequency'  : 'PerTree',
    'loss_function'       : 'Logloss',
    'eval_metric'         : 'AUC',
    'random_state'        : RANDOM_STATE,
    'verbose'             : False,
    'thread_count'        : 1,
    'used_ram_limit'      : '3gb',
    'allow_writing_files' : False,
}

# ── 8. HistGradientBoosting ───────────────────────────────
hist_gb_params = {
    'max_iter'          : 50,
    'max_depth'         : 3,
    'learning_rate'     : 0.05,
    'min_samples_leaf'  : 50,
    'l2_regularization' : 3.0,
    'max_bins'          : 128,
    'max_leaf_nodes'    : 15,
    'early_stopping'    : False,   # Disabled for determinism
    'random_state'      : RANDOM_STATE,
}

print("✓ All hyperparameter dictionaries defined")


## Section 7 · Model Training

In [ ]:
# ============================================================
# Section 7: Model Training (fully deterministic)
# ============================================================

ml_results = {}

print("=" * 70)
print("Model training — fully deterministic mode")
print("=" * 70)

# ── 1. XGBoost Standard ───────────────────────────────────
print("\n1. XGBoost Standard")
set_global_seed(RANDOM_STATE)
xgb_standard = xgb.XGBClassifier(**xgb_standard_params)
xgb_standard.fit(X_train, y_train, verbose=False)
ml_results['XGBoost_Standard'] = {
    'model'      : xgb_standard,
    'train_proba': xgb_standard.predict_proba(X_train)[:, 1],
    'val_proba'  : xgb_standard.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['XGBoost_Standard']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['XGBoost_Standard']['val_proba']):.4f}")

# ── 2. XGBoost DART ───────────────────────────────────────
print("\n2. XGBoost DART")
set_global_seed(RANDOM_STATE)
xgb_dart = xgb.XGBClassifier(**xgb_dart_params)
xgb_dart.fit(X_train, y_train, verbose=False)
ml_results['XGBoost_DART'] = {
    'model'      : xgb_dart,
    'train_proba': xgb_dart.predict_proba(X_train)[:, 1],
    'val_proba'  : xgb_dart.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['XGBoost_DART']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['XGBoost_DART']['val_proba']):.4f}")

# ── 3. LightGBM GBDT ──────────────────────────────────────
print("\n3. LightGBM GBDT")
set_global_seed(RANDOM_STATE)
lgb_gbdt = lgb.LGBMClassifier(**lgb_gbdt_params)
lgb_gbdt.fit(X_train, y_train)  # eval_set・callback削除
ml_results['LightGBM_GBDT'] = {
    'model'      : lgb_gbdt,
    'train_proba': lgb_gbdt.predict_proba(X_train)[:, 1],
    'val_proba'  : lgb_gbdt.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['LightGBM_GBDT']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['LightGBM_GBDT']['val_proba']):.4f}")

# ── 4. LightGBM DART ──────────────────────────────────────
print("\n4. LightGBM DART")
set_global_seed(RANDOM_STATE)
lgb_dart = lgb.LGBMClassifier(**lgb_dart_params)
lgb_dart.fit(X_train, y_train)  # eval_set・callback削除
ml_results['LightGBM_DART'] = {
    'model'      : lgb_dart,
    'train_proba': lgb_dart.predict_proba(X_train)[:, 1],
    'val_proba'  : lgb_dart.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['LightGBM_DART']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['LightGBM_DART']['val_proba']):.4f}")

# ── 5. LightGBM GOSS ──────────────────────────────────────
print("\n5. LightGBM GOSS")
set_global_seed(RANDOM_STATE)
lgb_goss = lgb.LGBMClassifier(**lgb_goss_params)
lgb_goss.fit(X_train, y_train)  # eval_set・callback削除
ml_results['LightGBM_GOSS'] = {
    'model'      : lgb_goss,
    'train_proba': lgb_goss.predict_proba(X_train)[:, 1],
    'val_proba'  : lgb_goss.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['LightGBM_GOSS']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['LightGBM_GOSS']['val_proba']):.4f}")

# ── 6. CatBoost Standard ──────────────────────────────────
print("\n6. CatBoost Standard")
set_global_seed(RANDOM_STATE)
catboost_standard = cb.CatBoostClassifier(**catboost_standard_params)
catboost_standard.fit(X_train, y_train, verbose=False)  # eval_set削除
ml_results['CatBoost_Standard'] = {
    'model'      : catboost_standard,
    'train_proba': catboost_standard.predict_proba(X_train)[:, 1],
    'val_proba'  : catboost_standard.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['CatBoost_Standard']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['CatBoost_Standard']['val_proba']):.4f}")

# ── 7. CatBoost Ordered ───────────────────────────────────
print("\n7. CatBoost Ordered")
set_global_seed(RANDOM_STATE)
catboost_ordered = cb.CatBoostClassifier(**catboost_ordered_params)
catboost_ordered.fit(X_train, y_train, verbose=False)   # eval_set削除
ml_results['CatBoost_Ordered'] = {
    'model'      : catboost_ordered,
    'train_proba': catboost_ordered.predict_proba(X_train)[:, 1],
    'val_proba'  : catboost_ordered.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['CatBoost_Ordered']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['CatBoost_Ordered']['val_proba']):.4f}")

# ── 8. HistGradientBoosting ───────────────────────────────
print("\n8. HistGradientBoosting")
set_global_seed(RANDOM_STATE)
hist_gb = HistGradientBoostingClassifier(**hist_gb_params)
hist_gb.fit(X_train, y_train)
ml_results['HistGradientBoosting'] = {
    'model'      : hist_gb,
    'train_proba': hist_gb.predict_proba(X_train)[:, 1],
    'val_proba'  : hist_gb.predict_proba(X_val)[:, 1],
    'train_true' : y_train,
    'val_true'   : y_val,
}
print(f"   Train AUC: {roc_auc_score(y_train, ml_results['HistGradientBoosting']['train_proba']):.4f}  "
      f"Val AUC: {roc_auc_score(y_val,   ml_results['HistGradientBoosting']['val_proba']):.4f}")

# ── Combine all results (MLRA + 8 ML models + Prediction One™) ──
all_results = {
    'MLRA': mlra_results,
    **ml_results,
    'Prediction_One': {
        'train_proba': predone_train_proba,
        'train_true' : predone_train_true,
        'val_proba'  : predone_val_proba,
        'val_true'   : predone_val_true,
    },
}

val_aucs = {n: roc_auc_score(r['val_true'], r['val_proba'])
            for n, r in all_results.items()}
best_model = max(val_aucs, key=val_aucs.get)
print(f"\nBest validation AUC: {best_model} ({val_aucs[best_model]:.4f})")
print("\n✓ All models trained")
# ── Verification: AUC values for each model must match original run ──


In [ ]:
# Verification only (temporary; delete after checking): validation AUC to 6 decimals
from sklearn.metrics import roc_auc_score
for n, r in ml_results.items():
    p = r['model'].predict_proba(X_val)[:, 1]
    print(f'{n:22s} {roc_auc_score(y_val, p):.6f}')

In [ ]:
# ============================================================
# Verification: AUCs to 6 decimals (resolves 3-decimal rounding)
# Run after the final ml_results replacement (_clean models).
# ============================================================
from sklearn.metrics import roc_auc_score
import pandas as pd

rows = [
    {'Model': 'MLRA',
     'Dev AUC': roc_auc_score(train_mlra['Outcome'], train_mlra_proba),
     'Val AUC': roc_auc_score(val_mlra['Outcome'],   val_mlra_proba)},
    {'Model': 'Prediction One',
     'Dev AUC': roc_auc_score(predone_train_true, predone_train_proba),
     'Val AUC': roc_auc_score(predone_val_true,   predone_val_proba)},
]
for name, r in ml_results.items():
    rows.append({'Model': name,
                 'Dev AUC': roc_auc_score(r['train_true'], r['train_proba']),
                 'Val AUC': roc_auc_score(r['val_true'],   r['val_proba'])})

auc6 = pd.DataFrame(rows)
auc6['Dev (3 dp)'] = auc6['Dev AUC'].round(3)
auc6['Val (3 dp)'] = auc6['Val AUC'].round(3)
pd.set_option('display.float_format', '{:.6f}'.format)
print(auc6.to_string(index=False))
pd.reset_option('display.float_format')

## Section 8 · Evaluation Metrics (DeLong 95% CI)

In [ ]:
# ============================================================
# Section 8: Comprehensive Performance Metrics
# Includes DeLong-method AUC 95% confidence intervals.
# ============================================================

def calculate_auc_ci(y_true: np.ndarray,
                     y_proba: np.ndarray,
                     alpha: float = 0.05):
    """
    Compute AUC with 95% CI via the DeLong structural components method.

    Returns
    -------
    auc : float
    se  : float  — standard error
    ci_lower : float
    ci_upper : float
    """
    y_true  = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    try:
        auc_val = roc_auc_score(y_true, y_proba)
    except Exception:
        return np.nan, np.nan, np.nan, np.nan

    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    n_pos, n_neg = len(pos_idx), len(neg_idx)
    if n_pos == 0 or n_neg == 0:
        return auc_val, np.nan, np.nan, np.nan

    pos_scores = y_proba[pos_idx]
    neg_scores = y_proba[neg_idx]

    V10 = np.array([
        np.mean(pos_scores[i] > neg_scores) +
        0.5 * np.mean(pos_scores[i] == neg_scores)
        for i in range(n_pos)
    ])
    V01 = np.array([
        np.mean(pos_scores > neg_scores[i]) +
        0.5 * np.mean(pos_scores == neg_scores[i])
        for i in range(n_neg)
    ])

    S10 = np.var(V10, ddof=1) / n_pos if n_pos > 1 else 0.0
    S01 = np.var(V01, ddof=1) / n_neg if n_neg > 1 else 0.0
    se  = np.sqrt(S10 + S01)

    z   = stats.norm.ppf(1.0 - alpha / 2.0)
    ci_lower = max(0.0, auc_val - z * se)
    ci_upper = min(1.0, auc_val + z * se)
    return auc_val, se, ci_lower, ci_upper


def calculate_all_metrics(y_true, y_proba, model_name,
                           optimal_threshold=None) -> dict:
    """
    Compute a comprehensive set of binary-classification metrics.

    Parameters
    ----------
    y_true : array-like   — true binary labels
    y_proba : array-like  — predicted probabilities
    model_name : str
    optimal_threshold : float or None
        If None, the Youden-J optimal threshold is computed.

    Returns
    -------
    dict of metrics
    """
    y_true  = np.asarray(y_true,  dtype=float)
    y_proba = np.asarray(y_proba, dtype=float)

    # Handle missing values
    if np.any(np.isnan(y_true)) or np.any(np.isnan(y_proba)):
        mask    = ~(np.isnan(y_true) | np.isnan(y_proba))
        y_true  = y_true[mask]
        y_proba = y_proba[mask]

    n_samples  = len(y_true)
    n_positive = int(np.sum(y_true == 1))
    n_negative = int(np.sum(y_true == 0))

    # Optimal threshold via Youden's J
    if optimal_threshold is None:
        fpr_arr, tpr_arr, threshs = roc_curve(y_true, y_proba)
        optimal_threshold = threshs[np.argmax(tpr_arr - fpr_arr)]

    y_pred = (y_proba >= optimal_threshold).astype(int)

    auc_val, auc_se, ci_lower, ci_upper = calculate_auc_ci(y_true, y_proba)

    try:
        auc_pr     = average_precision_score(y_true, y_proba)
    except ValueError:
        auc_pr     = np.nan
    try:
        brier      = brier_score_loss(y_true, y_proba)
    except ValueError:
        brier      = np.nan
    try:
        logloss    = log_loss(y_true, np.clip(y_proba, 1e-15, 1 - 1e-15))
    except ValueError:
        logloss    = np.nan

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    except ValueError:
        tp = n_positive; fp = n_negative; tn = fn = 0

    accuracy    = (tp + tn) / n_samples if n_samples else 0
    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0
    precision   = tp / (tp + fp) if (tp + fp) else 0
    npv         = tn / (tn + fn) if (tn + fn) else 0
    f1          = (2 * precision * sensitivity / (precision + sensitivity)
                   if (precision + sensitivity) else 0)
    try:
        mcc     = matthews_corrcoef(y_true, y_pred)
    except ValueError:
        mcc     = np.nan

    fpr_rate    = fp / (fp + tn) if (fp + tn) else 0
    fnr_rate    = fn / (fn + tp) if (fn + tp) else 0
    lr_pos      = sensitivity / fpr_rate if fpr_rate > 0 else np.inf
    lr_neg      = fnr_rate   / specificity if specificity > 0 else np.inf
    dor         = (lr_pos / lr_neg
                   if (lr_neg > 0 and np.isfinite(lr_pos) and np.isfinite(lr_neg))
                   else np.nan)

    # Hosmer–Lemeshow statistic
    try:
        n_bins       = min(10, len(np.unique(y_proba)))
        bin_bounds   = np.percentile(y_proba, np.linspace(0, 100, n_bins + 1))
        bin_bounds[-1] += 1e-8
        hl_stat = 0.0
        for lo, hi in zip(bin_bounds[:-1], bin_bounds[1:]):
            mask = (y_proba >= lo) & (y_proba < hi)
            n_i  = mask.sum()
            if n_i > 5:
                o_i = y_true[mask].sum()
                e_i = y_proba[mask].sum()
                if 0 < e_i < n_i:
                    hl_stat += (o_i - e_i) ** 2 / (e_i * (1 - e_i / n_i))
        hl_p = 1.0 - stats.chi2.cdf(hl_stat, df=max(1, n_bins - 2))
    except Exception:
        hl_stat, hl_p = np.nan, np.nan

    return {
        'Model'             : model_name,
        'N_Samples'         : n_samples,
        'N_Positive'        : n_positive,
        'N_Negative'        : n_negative,
        'Positive_Rate'     : n_positive / n_samples if n_samples else 0,
        'Optimal_Threshold' : optimal_threshold,
        'AUC_ROC'           : auc_val,
        'AUC_SE'            : auc_se,
        'AUC_95CI_Lower'    : ci_lower,
        'AUC_95CI_Upper'    : ci_upper,
        # [FIXED] What was wrong: CI string was formatted as upper–lower (reversed)
        # [FIXED] What was changed: swapped to lower–upper (conventional notation)
        'AUC_95CI'          : (f"{ci_lower:.3f}\u2013{ci_upper:.3f}"
                               if not np.isnan(ci_lower) else "N/A"),
        'AUC_PR'            : auc_pr,
        'Brier_Score'       : brier,
        'Log_Loss'          : logloss,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'Accuracy'          : accuracy,
        'Sensitivity'       : sensitivity,
        'Specificity'       : specificity,
        'Precision'         : precision,
        'NPV'               : npv,
        'F1_Score'          : f1,
        'MCC'               : mcc,
        'FPR'               : fpr_rate,
        'FNR'               : fnr_rate,
        'LR_Positive'       : lr_pos,
        'LR_Negative'       : lr_neg,
        'DOR'               : dor,
        'HL_Statistic'      : hl_stat,
        'HL_P_value'        : hl_p,
    }


# ── Compute metrics for all models ────────────────────────
print("Computing training-set metrics …")
train_metrics_list = []
for model_name, results in sorted(all_results.items()):
    y_true  = (results['train_true'].values
               if hasattr(results['train_true'], 'values')
               else np.array(results['train_true']))
    display = 'Prediction One\u2122' if model_name == 'Prediction_One' else model_name
    train_metrics_list.append(calculate_all_metrics(y_true,
                                                    np.array(results['train_proba']),
                                                    display))

print("Computing validation-set metrics …")
val_metrics_list = []
for model_name, results in sorted(all_results.items()):
    y_true  = (results['val_true'].values
               if hasattr(results['val_true'], 'values')
               else np.array(results['val_true']))
    display = 'Prediction One\u2122' if model_name == 'Prediction_One' else model_name
    val_metrics_list.append(calculate_all_metrics(y_true,
                                                  np.array(results['val_proba']),
                                                  display))

train_metrics_df = pd.DataFrame(train_metrics_list)
val_metrics_df   = pd.DataFrame(val_metrics_list)

# Print summary
key_cols = ['Model', 'AUC_ROC', 'AUC_95CI', 'Sensitivity', 'Specificity', 'F1_Score']
print("\nValidation-set summary (sorted by AUC):")
print(val_metrics_df[key_cols].sort_values('AUC_ROC', ascending=False).to_string(index=False))

# Export to Excel
excel_path = 'model_performance_metrics_95CI.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    train_metrics_df.to_excel(writer, sheet_name='Training',   index=False)
    val_metrics_df.to_excel(  writer, sheet_name='Validation', index=False)
from IPython.display import display as _disp18
print("\n[Development Cohort — Key Metrics]")
_disp18(train_metrics_df[key_cols].sort_values("AUC_ROC", ascending=False))
print("\n[Validation Cohort — Key Metrics]")
_disp18(val_metrics_df[key_cols].sort_values("AUC_ROC", ascending=False))
print(f"\n✓ Saved: {excel_path}")
files.download(excel_path)
# ── Verification: AUC, Sensitivity, Specificity values must match original ──


In [ ]:
# ============================================================
# Table 4: Comprehensive Performance Metrics — Manuscript Format
# Prerequisite: Cell 18 (train_metrics_df, val_metrics_df in memory)
# Columns: Model, N, Events, AUC-ROC (95% CI), AUC-PR,
#          Threshold, Sens, Spec, PPV, NPV, Acc, F1, Brier, H-L P
# ============================================================
import pandas as pd
from google.colab import files

def _format_table4(df: pd.DataFrame) -> pd.DataFrame:
    """Reformat raw metrics DataFrame into manuscript Table 4 layout."""
    out = pd.DataFrame()
    out['Model']            = df['Model']
    out['N']                = df['N_Samples'].astype(int)
    out['Events']           = df['N_Positive'].astype(int)
    out['AUC-ROC (95% CI)'] = df.apply(
        lambda r: (
            f"{r['AUC_ROC']:.3f} "
            f"({r['AUC_95CI_Lower']:.3f}\u2013{r['AUC_95CI_Upper']:.3f})"
            if pd.notna(r['AUC_ROC']) else 'N/A'
        ), axis=1
    )
    out['AUC-PR']      = df['AUC_PR'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['Threshold']   = df['Optimal_Threshold'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['Sens']        = df['Sensitivity'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['Spec']        = df['Specificity'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['PPV']         = df['Precision'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['NPV']         = df['NPV'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['Acc']         = df['Accuracy'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['F1']          = df['F1_Score'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    out['Brier']       = df['Brier_Score'].map(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')
    # [FIXED] BUG-5: H-L P column removed — Hosmer-Lemeshow test not performed.
    return out

print("=" * 70)
print("Table 4: Comprehensive Performance Metrics for All Predictive Models")
print("=" * 70)

t4_dev = _format_table4(train_metrics_df)
t4_val = _format_table4(val_metrics_df)

print("\n[Development Cohort]")
print(t4_dev.to_string(index=False))
print("\n[Validation Cohort]")
print(t4_val.to_string(index=False))

TABLE4_XLSX = 'S11_Table_Comprehensive_Performance_Metrics_All_Models.xlsx'
from IPython.display import display as _disp19
print("\n[Table 4 — Development Cohort]")
_disp19(t4_dev)
print("\n[Table 4 — Validation Cohort]")
_disp19(t4_val)
with pd.ExcelWriter(TABLE4_XLSX, engine='openpyxl') as writer:
    t4_dev.to_excel(writer, sheet_name='Development', index=False)
    t4_val.to_excel(writer, sheet_name='Validation',  index=False)

files.download(TABLE4_XLSX)
print(f"\n✓ Downloaded: {TABLE4_XLSX}")
# Verification: AUC-ROC values and 95% CI must match Cell 18 output.

## Section 9 · ROC Curves (Publication Quality)

In [ ]:
# ============================================================
# Section 9: ROC Curves — Publication-quality figures
# Three figures: (A) full+zoom combined, (B) training, (C) validation
# Resolution: 600 DPI  Format: TIFF + PNG
# ============================================================

!pip install python-pptx -q

# Publication settings
JOURNAL_DPI = 600
FS_TITLE    = 14
FS_LABEL    = 12
FS_TICK     = 10
FS_LEGEND   = 9
LW_MAIN     = 2.5
LW_SEC      = 2.0
FIG_SIZE    = (18, 8)

plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.sans-serif'  : ['Arial', 'DejaVu Sans'],
    'font.size'        : FS_TICK,
    'axes.labelsize'   : FS_LABEL,
    'axes.titlesize'   : FS_TITLE,
    'savefig.dpi'      : JOURNAL_DPI,
    'axes.linewidth'   : 1.5,
    'grid.linewidth'   : 0.5,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
})

set_global_seed(RANDOM_STATE)

sorted_models = sorted(
    all_results.items(),
    key=lambda x: roc_auc_score(x[1]['val_true'], x[1]['val_proba']),
    reverse=True
)

roc_files = []

def _display_name(model_name: str) -> str:
    return 'Prediction One\u2122' if model_name == 'Prediction_One' else model_name

def _linewidth(model_name: str) -> float:
    return LW_MAIN if model_name in ('MLRA', 'Prediction_One') else LW_SEC

def _zorder(model_name: str) -> int:
    return 10 if model_name == 'MLRA' else (9 if model_name == 'Prediction_One' else 5)

def _plot_roc_panel(ax, models, use_train: bool, use_val: bool,
                    show_zoom: bool = False, xlim=(0, 1), ylim=(0, 1.05)):
    """Draw ROC curves onto ax."""
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Chance')
    for name, res in models:
        color = MODEL_COLORS.get(name, '#888888')
        lw    = _linewidth(name)
        zo    = _zorder(name)
        label = _display_name(name)
        if use_train:
            fpr, tpr, _ = roc_curve(res['train_true'], res['train_proba'])
            auc_tr       = roc_auc_score(res['train_true'], res['train_proba'])
            ax.plot(fpr, tpr, color=color, ls='-',
                    lw=lw, alpha=0.8, zorder=zo,
                    label=f'{label} Dev ({auc_tr:.3f})')
        if use_val:
            fpr, tpr, _ = roc_curve(res['val_true'], res['val_proba'])
            auc_vl       = roc_auc_score(res['val_true'], res['val_proba'])
            ax.plot(fpr, tpr, color=color, ls='--',
                    lw=lw, alpha=0.8, zorder=zo,
                    label=f'{label} Val ({auc_vl:.3f})')
    if show_zoom:
        ax.add_patch(Rectangle((0, 0.6), 0.4, 0.4, fill=False,
                                edgecolor='black', lw=2, ls='--'))
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('False Positive Rate', fontsize=FS_LABEL, fontweight='bold')
    ax.set_ylabel('True Positive Rate',  fontsize=FS_LABEL, fontweight='bold')
    ax.grid(True, alpha=0.3, ls=':', lw=0.5)
    ax.legend(loc='lower right', fontsize=FS_LEGEND - 1,
              frameon=True, fancybox=False, edgecolor='black', ncol=1)


# ── Figure 1: Full + Zoomed (training and validation) ─────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE)
_plot_roc_panel(ax1, sorted_models, use_train=True, use_val=True,
                show_zoom=True, xlim=(0, 1), ylim=(0, 1.0))
ax1.set_title('A. Complete ROC Curves', fontsize=FS_TITLE,
              fontweight='bold', loc='left')
ax1.set_aspect('equal', adjustable='box')
_plot_roc_panel(ax2, sorted_models, use_train=True, use_val=True,
                xlim=(0, 0.4), ylim=(0.6, 1.0))
ax2.set_title('B. High-Performance Region', fontsize=FS_TITLE,
              fontweight='bold', loc='left')
plt.tight_layout()
fname = 'Fig3_ROC_curves.jpg'
plt.savefig(fname, dpi=JOURNAL_DPI, bbox_inches='tight',
            format='jpeg', pil_kwargs={'quality': 95})
roc_files.append(fname)
print(f"✓ Saved {fname}")
plt.show()
plt.close(fig)

# ── Figure 2: Training set only ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE)
_plot_roc_panel(ax1, sorted_models, use_train=True, use_val=False,
                show_zoom=True, xlim=(0, 1), ylim=(0, 1.0))
ax1.set_title('A. Training Set ROC', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax1.set_aspect('equal', adjustable='box')
_plot_roc_panel(ax2, sorted_models, use_train=True, use_val=False,
                xlim=(0, 0.4), ylim=(0.6, 1.0))
ax2.set_title('B. High-Performance Region', fontsize=FS_TITLE, fontweight='bold', loc='left')
plt.tight_layout()
fname = 'figure2_roc_training.jpg'
plt.savefig(fname, dpi=JOURNAL_DPI, bbox_inches='tight',
            format='jpeg', pil_kwargs={'quality': 95})
roc_files.append(fname)
print(f"✓ Saved {fname}")
plt.show()
plt.close(fig)

# ── Figure 3: Validation set only ─────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE)
_plot_roc_panel(ax1, sorted_models, use_train=False, use_val=True,
                show_zoom=True, xlim=(0, 1), ylim=(0, 1.0))
ax1.set_title('A. Validation Set ROC', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax1.set_aspect('equal', adjustable='box')
_plot_roc_panel(ax2, sorted_models, use_train=False, use_val=True,
                xlim=(0, 0.4), ylim=(0.6, 1.0))
ax2.set_title('B. High-Performance Region', fontsize=FS_TITLE, fontweight='bold', loc='left')
plt.tight_layout()
fname = 'figure3_roc_validation.jpg'
plt.savefig(fname, dpi=JOURNAL_DPI, bbox_inches='tight',
            format='jpeg', pil_kwargs={'quality': 95})
roc_files.append(fname)
print(f"✓ Saved {fname}")
plt.show()
plt.close(fig)

# ── AUC summary table ──────────────────────────────────────
auc_rows = sorted([
    (_display_name(n),
     roc_auc_score(r['train_true'], r['train_proba']),
     roc_auc_score(r['val_true'],   r['val_proba']))
    for n, r in all_results.items()
], key=lambda x: x[2], reverse=True)

auc_df = pd.DataFrame(auc_rows, columns=['Model', 'Train_AUC', 'Val_AUC'])
auc_df['Difference'] = auc_df['Train_AUC'] - auc_df['Val_AUC']
print("\n", auc_df.round(4).to_string(index=False))

auc_excel = 'roc_auc_summary.xlsx'
auc_df.to_excel(auc_excel, index=False)
files.download(auc_excel)

# ── PowerPoint export ─────────────────────────────────────
prs = Presentation()
slide_data = [
    ("ROC Curves — Full View and Zoomed (Training + Validation)", roc_files[0]),
    ("ROC Curves — Training Set",   roc_files[1]),
    ("ROC Curves — Validation Set", roc_files[2]),
]
for title_text, img_path in slide_data:
    sl = prs.slides.add_slide(prs.slide_layouts[5])
    sl.shapes.title.text = title_text
    sl.shapes.add_picture(img_path, Inches(0.5), Inches(1.5), Inches(9), Inches(5))
pptx_roc = 'roc_curves.pptx'
prs.save(pptx_roc)
files.download(pptx_roc)
print(f"\n✓ Saved {pptx_roc}")
# ── Verification: ROC AUC values in figures must match train_metrics_df / val_metrics_df ──


In [ ]:
from IPython.display import Image as _Img22, display as _disp22
import os as _os22
# ============================================================
# Section 9: Download ROC Curve figures (JPG, 300+ DPI)
# ============================================================
import os
from google.colab import files

_roc_files = [
    'Fig3_ROC_curves.jpg',
    'figure2_roc_training.jpg',
    'figure3_roc_validation.jpg',
]

print("Downloading ROC curve figures...")
for _fname in _roc_files:
    if os.path.exists(_fname):
        files.download(_fname)
        print(f"  ✓ Downloaded: {_fname}")
        if _fname.endswith(".jpg") and _os22.path.exists(_fname):
            _disp22(_Img22(_fname, width=900))
    else:
        print(f"  ⚠  File not found (re-run Section 9 to regenerate): {_fname}")

## Section 10 · Calibration Analysis

In [ ]:
# ============================================================
# Section 10: Calibration Analysis (logistic recalibration; standard definition)
#   Calibration slope     : coefficient b of logit(p) in Logit(y ~ a + b*logit(p))
#   Calibration intercept : calibration-in-the-large, Logit(y ~ 1, offset = logit(p))
#   95% CIs               : model-based Wald intervals (statsmodels)
#   Plot                  : observed proportion vs mean predicted probability in
#                           ten equal-width bins (points; marker area ∝ bin size)
#                           + fitted logistic recalibration curve (line)
# [REPLACED 2026-10-24] Previous version regressed binned observed proportions
#   on mean predicted probabilities by unweighted OLS and reported the residual
#   standard error as "SE"; that is not the calibration slope as defined in the
#   Methods. Built-in check: MLRA evaluated on its development data must give
#   slope = 1 and intercept = 0 (property of maximum likelihood logistic regression).
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16,
                     'legend.fontsize': 10})

# Line styles: development = solid, validation = dashed (same in plot and legend)
LS_DEV = '-'
LS_VAL = (0, (2.0, 1.2))   # short dash; visible in legend handles at lw 3-4.5

EPS = 1e-6

def _get_arr(data):
    return data.values if hasattr(data, 'values') else np.asarray(data)

def _logit(p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)
    return np.log(p / (1 - p))

def logistic_calibration(y_true, y_proba):
    """Calibration slope (with its intercept a for plotting) and
    calibration-in-the-large intercept, each with Wald 95% CI."""
    y  = np.asarray(y_true, dtype=float)
    lp = _logit(y_proba)
    m_s = sm.Logit(y, sm.add_constant(lp)).fit(disp=False, maxiter=200)
    m_i = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(),
                 offset=lp).fit()
    assert m_s.mle_retvals['converged'], "Calibration slope model did not converge — HALT"
    s_ci = m_s.conf_int()[1]
    i_ci = m_i.conf_int()[0]
    return {'a_slope_model': m_s.params[0],
            'Slope': m_s.params[1], 'Slope_95CI_Lower': s_ci[0], 'Slope_95CI_Upper': s_ci[1],
            'Intercept': m_i.params[0], 'Intercept_95CI_Lower': i_ci[0],
            'Intercept_95CI_Upper': i_ci[1],
            'N': len(y), 'Events': int(y.sum())}

def binned_points(y_true, y_proba, n_bins=10):
    """Mean predicted probability, observed proportion and count per equal-width bin
    (descriptive points for the calibration plot only)."""
    y, p = np.asarray(y_true, dtype=float), np.asarray(y_proba, dtype=float)
    edges = np.linspace(0, 1, n_bins + 1)
    mp, obs, cnt = [], [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (p >= lo) & ((p <= hi) if i == n_bins - 1 else (p < hi))
        if m.sum() > 0:
            mp.append(p[m].mean()); obs.append(y[m].mean()); cnt.append(m.sum())
    return np.array(mp), np.array(obs), np.array(cnt)

def recal_curve(cal, y_proba, n=200):
    """Fitted logistic recalibration curve over the observed range of predictions."""
    p = np.clip(np.asarray(y_proba, dtype=float), EPS, 1 - EPS)
    x = np.linspace(p.min(), p.max(), n)
    eta = cal['a_slope_model'] + cal['Slope'] * _logit(x)
    return x, 1 / (1 + np.exp(-eta))

sorted_models_cal = sorted(
    all_results.items(),
    key=lambda x: roc_auc_score(x[1]['val_true'], x[1]['val_proba']),
    reverse=True)

def _style(name):
    lw = 4.5 if name == 'MLRA' else (3.5 if name == 'Prediction_One' else 3)
    zo = 10  if name == 'MLRA' else (8   if name == 'Prediction_One' else 5)
    mk = 's' if name == 'MLRA' else ('d' if name == 'Prediction_One' else 'o')
    return lw, zo, mk

def _ordered(models):
    for priority in ('other', 'Prediction_One', 'MLRA'):
        for name, res in models:
            if priority == 'other' and name in ('MLRA', 'Prediction_One'): continue
            if priority in ('Prediction_One', 'MLRA') and name != priority: continue
            yield name, res

def draw_panel(ax, cohort, stats_list, linestyle=LS_DEV, ms_scale=12):
    yk, pk = ('train_true', 'train_proba') if cohort == 'Development' else ('val_true', 'val_proba')
    ax.plot([0, 1], [0, 1], 'k:', lw=2.0, label='Perfect calibration', alpha=0.7)
    for name, res in _ordered(sorted_models_cal):
        if yk not in res or pk not in res:
            print(f"  ⚠ {name}: no {cohort.lower()} predictions — skipped"); continue
        yt, yp = _get_arr(res[yk]), np.asarray(res[pk])
        cal = logistic_calibration(yt, yp)
        mp, obs, cnt = binned_points(yt, yp)
        xc, yc = recal_curve(cal, yp)
        color = MODEL_COLORS.get(name, '#888888')
        lw, zo, mk = _style(name)
        label = _display_name(name).replace('_', ' ')
        ax.scatter(mp, obs, color=color, s=np.sqrt(cnt) * ms_scale, alpha=0.85,
                   edgecolors='white', lw=1.5, zorder=zo, marker=mk)
        ax.plot(xc, yc, color=color, ls=linestyle, lw=lw, alpha=0.85, zorder=zo - 1,
                label=f"{label} (slope {cal['Slope']:.2f})")
        stats_list.append({'Model': label, 'Cohort': cohort,
                           **{k: v for k, v in cal.items() if k != 'a_slope_model'}})
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    ax.set_xlabel('Predicted probability', fontsize=14, fontweight='bold')
    ax.set_ylabel('Observed proportion', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, ls='--'); ax.set_aspect('equal', adjustable='box')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)

cal_files = []

# ── Figure 4 (validation, single panel) ──────────────────────
cal_stats_val = []
fig, ax = plt.subplots(figsize=(12, 10))
draw_panel(ax, 'Validation', cal_stats_val, LS_VAL)
ax.legend(loc='lower right', fontsize=9, framealpha=0.95, fancybox=True, shadow=True,
          handlelength=4)
plt.tight_layout()
f4 = 'figure4_calibration_validation.jpg'
plt.savefig(f4, dpi=300, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
cal_files.append(f4); plt.show(); plt.close(fig); print(f"✓ Saved {f4}")

# ── Figure 5 (development vs validation) ─────────────────────
cal_stats_train, _tmp = [], []
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
draw_panel(ax1, 'Development', cal_stats_train, LS_DEV, 10)
draw_panel(ax2, 'Validation', _tmp, LS_VAL, 10)
ax1.set_title('Development cohort'); ax2.set_title('Validation cohort')
for a in (ax1, ax2):
    a.legend(loc='lower right', fontsize=8, framealpha=0.95, handlelength=4)
plt.tight_layout()
f5 = 'Fig4_Calibration_plots.jpg'
plt.savefig(f5, dpi=300, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
cal_files.append(f5); plt.show(); plt.close(fig); print(f"✓ Saved {f5}")

cal_train_df = pd.DataFrame(cal_stats_train)
cal_val_df   = pd.DataFrame(cal_stats_val)

# ── Built-in definition check (HALT on failure) ──────────────
_m = cal_train_df[cal_train_df['Model'] == _display_name('MLRA')]
if len(_m) == 1:
    assert abs(_m['Slope'].iloc[0] - 1) < 0.01 and abs(_m['Intercept'].iloc[0]) < 0.01, \
        "MLRA apparent calibration is not slope 1 / intercept 0 — implementation error — HALT"
    print("✓ Definition check passed: MLRA apparent slope = 1, intercept = 0")

from IPython.display import display as _disp24
pd.set_option('display.width', 200)
print('\n[Calibration — Development]'); _disp24(cal_train_df.round(3))
print('\n[Calibration — Validation]');  _disp24(cal_val_df.round(3))

cal_excel = 'calibration_statistics.xlsx'
with pd.ExcelWriter(cal_excel, engine='openpyxl') as writer:
    cal_train_df.to_excel(writer, sheet_name='Training',   index=False)
    cal_val_df.to_excel(  writer, sheet_name='Validation', index=False)
files.download(cal_excel)
for f in cal_files:
    files.download(f)
print("\n✓ Calibration analysis complete")

In [ ]:
# ============================================================
# S13 Table (former Table 5): Calibration slope and intercept
# Prerequisite: Section 10 (cal_train_df, cal_val_df in memory)
# Columns: Model | N | Events | Calibration slope (95% CI) | Calibration intercept (95% CI)
# ============================================================
import pandas as pd
from IPython.display import display as _disp25
from google.colab import files

def _ci(v, lo, hi):
    return f"{v:.3f} ({lo:.3f} to {hi:.3f})" if pd.notna(v) else 'N/A'

def _format_s14(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'Model'                             : df['Model'],
        'N'                                 : df['N'],
        'Events'                            : df['Events'],
        'Calibration slope (95% CI)'        : [_ci(*r) for r in df[['Slope', 'Slope_95CI_Lower', 'Slope_95CI_Upper']].values],
        'Calibration intercept (95% CI)'    : [_ci(*r) for r in df[['Intercept', 'Intercept_95CI_Lower', 'Intercept_95CI_Upper']].values],
    })

s14_dev = _format_s14(cal_train_df)
s14_val = _format_s14(cal_val_df)

print("=" * 70); print("S13 Table: Calibration slope and intercept"); print("=" * 70)
print("\n[Development cohort]"); _disp25(s14_dev)
print("\n[Validation cohort]");  _disp25(s14_val)

S14_XLSX = 'S13_Table_Calibration_Slope_Intercept.xlsx'
with pd.ExcelWriter(S14_XLSX, engine='openpyxl') as writer:
    s14_dev.to_excel(writer, sheet_name='Development', index=False)
    s14_val.to_excel(writer, sheet_name='Validation',  index=False)
files.download(S14_XLSX)
print(f"\n✓ Downloaded: {S14_XLSX}")
# CIs use "to" because intercepts can be negative (avoids an ambiguous en dash).

In [ ]:
# ============================================================
# Verification: logistic calibration slope / intercept (standard definition)
#   slope     : coefficient of logit(p) in Logit(y ~ logit(p))       [Cox]
#   intercept : calibration-in-the-large, Logit(y ~ 1, offset=logit(p))
#   95% CI    : model-based (Wald), from statsmodels
# Independent route (GLM offset model) compared with Section 10 (cal_train_df / cal_val_df).
# Does not overwrite any existing result.
# ============================================================
import numpy as np
import pandas as pd
import statsmodels.api as sm

EPS = 1e-6

def logistic_calibration(y, p):
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)
    lp = np.log(p / (1 - p))
    m_s = sm.Logit(y, sm.add_constant(lp)).fit(disp=False, maxiter=200)
    m_i = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(),
                 offset=lp).fit()
    s_lo, s_hi = m_s.conf_int()[1]
    i_lo, i_hi = m_i.conf_int()[0]
    return dict(Slope=m_s.params[1], Slope_L=s_lo, Slope_U=s_hi,
                Intercept_CITL=m_i.params[0], Int_L=i_lo, Int_U=i_hi,
                n=len(y), events=int(y.sum()))

rows = []
for name, res in all_results.items():
    for coh, yk, pk in [('Development', 'train_true', 'train_proba'),
                        ('Validation',  'val_true',   'val_proba')]:
        if yk in res and pk in res:
            r = logistic_calibration(res[yk], res[pk])
            rows.append({'Model': name, 'Cohort': coh, **r})

logcal = pd.DataFrame(rows)

# current binned OLS values for side-by-side comparison
cur = pd.concat([cal_train_df.assign(Cohort='Development'),
                 cal_val_df.assign(Cohort='Validation')])[['Model', 'Cohort', 'Slope', 'Intercept']]
cur = cur.rename(columns={'Slope': 'Section10_slope', 'Intercept': 'Section10_intercept'})
cur['Model'] = cur['Model'].str.replace('™', '', regex=False).str.strip()
logcal['Model_key'] = logcal['Model'].str.replace('™', '', regex=False).str.replace(' ', '_').str.strip()
cur['Model_key'] = cur['Model'].str.replace(' ', '_')
cmp = logcal.merge(cur.drop(columns='Model'), on=['Model_key', 'Cohort'], how='left')

pd.set_option('display.width', 220)
cols = ['Model', 'Cohort', 'n', 'events', 'Slope', 'Slope_L', 'Slope_U',
        'Intercept_CITL', 'Int_L', 'Int_U', 'Section10_slope', 'Section10_intercept']
print(cmp[cols].sort_values(['Cohort', 'Model']).round(3).to_string(index=False))

cmp[cols].to_csv('verification_calibration_two_routes.csv', index=False, encoding='utf-8-sig')
from google.colab import files
files.download('verification_calibration_two_routes.csv')

## S14 Table · Calibration and Classification Performance — Selected Gradient Boosting Models (Validation Cohort)

In [ ]:
# ============================================================
# S14 Table: Calibration and Classification Performance of
#            Selected Gradient Boosting Models — Validation Cohort
# Prerequisite: Cell 18 (train_metrics_df, val_metrics_df)
#               Cell 24 (cal_val_df) must be in memory.
# Columns: Model | AUC-ROC (95% CI) | ΔAUC (Dev–Val) | Sensitivity |
#          Specificity | F1-score | Calibration slope | Brier score
# Models: All ML gradient-boosting models (excludes MLRA & Prediction One)
# Sort order: Validation AUC descending
# ============================================================
from IPython.display import display as _disp
import pandas as pd, numpy as np
from google.colab import files as _files

# Display name mapping (internal key → publication name)
_GB_KEYS = [
    'XGBoost_Standard', 'XGBoost_DART',
    'LightGBM_GBDT', 'LightGBM_DART', 'LightGBM_GOSS',
    'CatBoost_Standard', 'CatBoost_Ordered',
    'HistGradientBoosting',
]
_DNAME = {
    'XGBoost_Standard'    : 'XGBoost Standard',
    'XGBoost_DART'        : 'XGBoost DART',
    'LightGBM_GBDT'       : 'LightGBM GBDT',
    'LightGBM_DART'       : 'LightGBM DART',
    'LightGBM_GOSS'       : 'LightGBM GOSS',
    'CatBoost_Standard'   : 'CatBoost Standard',
    'CatBoost_Ordered'    : 'CatBoost Ordered',
    'HistGradientBoosting': 'HistGradientBoosting',
}

def _lookup_metrics(df, key):
    """Return row from a metrics DataFrame matching model key or display name."""
    pub = _DNAME.get(key, key)
    for _, row in df.iterrows():
        rname = row['Model'].replace(' ', '_')
        if rname == key or row['Model'] == pub:
            return row
    return None

def _hl_fmt(v):
    if pd.isna(v): return 'N/A'
    return '<0.001' if v < 0.001 else f'{v:.4f}'

print('=' * 70)
print('S14 Table: Calibration and Classification Performance')
print('           Selected Gradient Boosting Models — Validation Cohort')
print('=' * 70)

rows_s12 = []
for key in _GB_KEYS:
    pub  = _DNAME.get(key, key)
    rdev = _lookup_metrics(train_metrics_df, key)
    rval = _lookup_metrics(val_metrics_df,   key)
    rcal = _lookup_metrics(cal_val_df,       key)

    if rval is None:
        print(f'  ⚠  {pub}: not found in val_metrics_df — skipped')
        continue

    val_auc  = rval['AUC_ROC']
    val_cilo = rval['AUC_95CI_Lower']
    val_cihi = rval['AUC_95CI_Upper']
    dev_auc  = rdev['AUC_ROC'] if rdev is not None else np.nan
    delta    = dev_auc - val_auc if (pd.notna(dev_auc) and pd.notna(val_auc)) else np.nan
    slope    = rcal['Slope'] if rcal is not None else np.nan

    rows_s12.append({
        'Model'            : pub,
        'AUC-ROC (95% CI)' : (f"{val_auc:.3f} ({val_cilo:.3f}\u2013{val_cihi:.3f})"
                               if pd.notna(val_auc) else 'N/A'),
        'ΔAUC (Dev–Val)'   : (f'{delta:+.3f}' if pd.notna(delta) else 'N/A'),
        'Sensitivity'      : (f"{rval['Sensitivity']:.3f}" if pd.notna(rval.get('Sensitivity')) else 'N/A'),
        'Specificity'      : (f"{rval['Specificity']:.3f}" if pd.notna(rval.get('Specificity')) else 'N/A'),
        'F1-score'         : (f"{rval['F1_Score']:.3f}" if pd.notna(rval.get('F1_Score')) else 'N/A'),
        'Calibration slope': (f'{slope:.3f}' if pd.notna(slope) else 'N/A'),
        'Brier score'      : (f"{rval['Brier_Score']:.3f}" if pd.notna(rval.get('Brier_Score')) else 'N/A'),
    })

df_s12 = pd.DataFrame(rows_s12)
# Sort by validation AUC descending
df_s12['_sort'] = df_s12['AUC-ROC (95% CI)'].apply(
    lambda x: float(x.split(' ')[0]) if x != 'N/A' else 0.0)
df_s12 = df_s12.sort_values('_sort', ascending=False).drop(columns=['_sort']).reset_index(drop=True)

print('\n')
_disp(df_s12)

S12_XLSX = 'S14_Table_Calibration_Classification_Gradient_Boosting_Validation.xlsx'
with pd.ExcelWriter(S12_XLSX, engine='openpyxl') as writer:
    df_s12.to_excel(writer, sheet_name='S14_Table', index=False)
print(f'\n\u2713 Saved: {S12_XLSX}')
_files.download(S12_XLSX)

# [Reproducibility] AUC-ROC values must match Section 8 (val_metrics_df).
# ΔAUC = Dev AUC − Val AUC; positive = external performance drop.
# Calibration slope ~ 1.0 = ideal; values from cal_val_df (Section 10).
# Calibration slope: logistic recalibration (Section 10).


## Section 11 · Decision Curve Analysis (DCA)

In [ ]:
# ============================================================
# Section 11: Decision Curve Analysis (DCA)
# ============================================================

def calculate_net_benefit(y_true, y_proba, threshold_prob: float) -> float:
    """
    Compute net benefit at a single threshold probability.

    NB = TP/n − FP/n × (pt / (1 − pt))
    """
    y_true  = np.asarray(y_true,  dtype=float)
    y_proba = np.asarray(y_proba, dtype=float)
    min_len = min(len(y_true), len(y_proba))
    y_true  = y_true[:min_len];  y_proba = y_proba[:min_len]
    mask    = ~(np.isnan(y_true) | np.isnan(y_proba))
    y_true  = y_true[mask];      y_proba = y_proba[mask]
    if len(y_true) == 0 or threshold_prob >= 1.0:
        return 0.0
    n   = len(y_true)
    pp  = y_proba >= threshold_prob
    tp  = np.sum(pp & (y_true == 1))
    fp  = np.sum(pp & (y_true == 0))
    return (tp / n) - (fp / n) * (threshold_prob / (1.0 - threshold_prob))


THRESHOLDS = np.arange(0.01, 0.99, 0.01)

def _compute_dca(results_dict, dataset: str, y_full: np.ndarray = None) -> dict:
    """Compute net benefit curves for all models.

    [FIXED] BUG-1: Treat All previously used val_true of the first model
    alphabetically (CatBoost_Ordered) as the prevalence denominator,
    creating inconsistency when different models have different val_true N.
    Treat All is now computed from y_full (full cohort outcome array) passed
    explicitly, so the reference strategy is always population-level.
    """
    dca = {}
    if y_full is None:
        ref_name = 'XGBoost_Standard' if 'XGBoost_Standard' in results_dict                    else sorted(results_dict.keys())[0]
        y_full = _get_arr(results_dict[ref_name][f'{dataset}_true'])
    prev = float(np.mean(y_full))

    dca['Treat All'] = [
        prev - (1.0 - prev) * (t / (1.0 - t)) if t < 1.0 else 0.0
        for t in THRESHOLDS
    ]
    dca['Treat None'] = [0.0] * len(THRESHOLDS)

    for name, res in results_dict.items():
        yt = _get_arr(res[f'{dataset}_true'])
        yp = np.asarray(res[f'{dataset}_proba'])
        dca[name] = [calculate_net_benefit(yt, yp, t) for t in THRESHOLDS]
    return dca


# [FIXED] BUG-1: Pass full cohort outcomes explicitly for Treat All computation
dca_train = _compute_dca(all_results, 'train',
                          y_full=_get_arr(all_results['XGBoost_Standard']['train_true']))
dca_val   = _compute_dca(all_results, 'val',
                          y_full=_get_arr(all_results['XGBoost_Standard']['val_true']))

def _plot_dca_panel(ax, dca: dict, linestyle: str, title: str):
    ax.plot(THRESHOLDS, dca['Treat All'], 'k--', lw=2.5, alpha=0.8, label='Treat All')
    ax.plot(THRESHOLDS, dca['Treat None'], 'k-', lw=2.0, alpha=0.8, label='Treat None')
    for priority in ('other', 'Prediction_One', 'MLRA'):
        for name in dca:
            if name in ('Treat All', 'Treat None'):
                continue
            if priority == 'other' and name in ('MLRA', 'Prediction_One'):
                continue
            if priority == 'Prediction_One' and name != 'Prediction_One':
                continue
            if priority == 'MLRA' and name != 'MLRA':
                continue
            color = MODEL_COLORS.get(name, '#888888')
            lw    = 4 if name == 'MLRA' else (3.5 if name == 'Prediction_One' else 2.5)
            zo    = 10 if name == 'MLRA' else (8 if name == 'Prediction_One' else 5)
            ax.plot(THRESHOLDS, dca[name], color=color,
                    lw=lw, ls=linestyle, alpha=0.85, zorder=zo,
                    label=_display_name(name))
    ax.set_xlim([0, 0.5]); ax.set_ylim([-0.05, 0.075])
    ax.set_xlabel('Threshold Probability', fontsize=14, fontweight='bold')
    ax.set_ylabel('Net Benefit',           fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left')
    ax.grid(True, alpha=0.3, ls='--')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    ax.legend(loc='lower right', fontsize=8, framealpha=0.95, ncol=2)


dca_files = []

# Figure 6: Training + Validation side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
_plot_dca_panel(ax1, dca_train, '-',  'A. Training Set DCA (solid lines)')
_plot_dca_panel(ax2, dca_val,   '--', 'B. Validation Set DCA (dashed lines)')
plt.tight_layout()
f6 = 'figure6_dca_train_vs_val.jpg'
plt.savefig(f6, dpi=300, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
dca_files.append(f6)
plt.show()
plt.close(fig)
print(f"✓ Saved {f6}")

# Figure 7: Validation set only
fig, ax = plt.subplots(figsize=(12, 10))
_plot_dca_panel(ax, dca_val, '-', 'Decision Curve Analysis — Validation Set')
plt.tight_layout()
f7 = 'figure7_dca_validation.jpg'
plt.savefig(f7, dpi=300, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
dca_files.append(f7)
plt.show()
plt.close(fig)
print(f"✓ Saved {f7}")

# Net benefit at key thresholds
key_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30]
dca_rows = []
for t in key_thresholds:
    idx = np.argmin(np.abs(THRESHOLDS - t))
    row = {'Threshold': t,
           'Treat_All' : dca_val['Treat All'][idx],
           'Treat_None': dca_val['Treat None'][idx]}
    for name in all_results:
        if name in dca_val:
            row[_display_name(name)] = dca_val[name][idx]
    dca_rows.append(row)
dca_stats_df = pd.DataFrame(dca_rows)
print("\nNet benefit at key thresholds (validation):")
print(dca_stats_df.round(4).to_string(index=False))
from IPython.display import display as _disp29
print('\n[Decision Curve Statistics]')
_disp29(dca_stats_df)

dca_excel = 'decision_curve_statistics.xlsx'
dca_stats_df.to_excel(dca_excel, index=False)
files.download(dca_excel)
for f in dca_files:
    files.download(f)
print("\n✓ DCA complete")
# ── Verification: Net benefit values at threshold 0.20 must match original ──


In [ ]:
# ============================================================
# S16 Table (former Table 6): Net Benefit Values — Decision Curve Analysis
#           External Validation Cohort
# [FIXED v2] BUG-2: Previous fix incorrectly used MLRA complete-case
#   prevalence (N=1,066) for Treat All, causing the S16 Table footnote
#   (N=1,243) to be inconsistent with the Treat All values.
#   Treat All is now computed from the full validation cohort (N=1,243)
#   via y_val (the full outcome array from Section 3/7).
#   This is the population-level reference strategy per Vickers (2006).
# ============================================================
import pandas as pd
import numpy as np
from google.colab import files

# ── Treat All using full validation cohort (N=1,243) ──────────────────
# [FIXED v2] Use y_val (full cohort) not MLRA complete-case subset.
_y_val_full_t6 = (
    y_val.values if hasattr(y_val, 'values') else np.array(y_val)
).astype(int)
_prev_full  = float(np.mean(_y_val_full_t6))
_n_full     = len(_y_val_full_t6)
_ev_full    = int(np.sum(_y_val_full_t6))

print(f"Treat All — full validation cohort:")
print(f"  N={_n_full}, Events={_ev_full}, prevalence={_prev_full:.5f}")

# Recompute Treat All at each threshold using full cohort prevalence
_treat_all_corrected = np.array([
    _prev_full - (1.0 - _prev_full) * (t / (1.0 - t))
    if t < 1.0 else 0.0
    for t in THRESHOLDS
])

# ── Display-name mapping ────────────────────────────────────────────────
_MODEL_DISPLAY = {
    'MLRA'                 : 'MLRA',
    'XGBoost_Standard'     : 'XGBoost Standard',
    'XGBoost_DART'         : 'XGBoost DART',
    'LightGBM_GBDT'        : 'LightGBM GBDT',
    'LightGBM_DART'        : 'LightGBM DART',
    'LightGBM_GOSS'        : 'LightGBM GOSS',
    'CatBoost_Standard'    : 'CatBoost Standard',
    'CatBoost_Ordered'     : 'CatBoost Ordered',
    'HistGradientBoosting' : 'HistGradientBoosting',
    'Prediction_One'       : 'Prediction One\u2122',
}

# ── Build S16 Table (thresholds 0.10–0.50) ───────────
_key_thresholds = np.round(np.arange(0.10, 0.51, 0.05), 2)   # S16 Table: 0.10–0.50

rows = []
for t in _key_thresholds:
    idx = int(np.argmin(np.abs(THRESHOLDS - t)))
    row = {'Threshold Probability': f"{t:.2f}"}

    for internal_key, display_name in _MODEL_DISPLAY.items():
        if internal_key in dca_val:
            val = dca_val[internal_key][idx]
            row[display_name] = f"{val:.4f}" if not np.isnan(val) else 'N/A'
        else:
            row[display_name] = 'N/A'

    # Treat All: full validation cohort (N = 1,243)
    row['Treat All']  = f"{_treat_all_corrected[idx]:.4f}"
    row['Treat None'] = '0.0000'
    rows.append(row)

t6_df = pd.DataFrame(rows)

print("\n" + "=" * 70)
print("S16 Table: Net Benefit Values — DCA (Validation cohort)")
print("=" * 70)
print(t6_df.to_string(index=False))
from IPython.display import display as _disp30
_disp30(t6_df)


TABLE6_XLSX = 'S16_Table_Net_Benefit_DCA_External_Validation.xlsx'
t6_df.to_excel(TABLE6_XLSX, index=False, sheet_name='Validation_DCA')
files.download(TABLE6_XLSX)
print(f"\n✓ Downloaded: {TABLE6_XLSX}")
# Verification: Treat All at t=0.10 should be ≈ −0.0628 (N=1,243, events=54)
# BUG-2 corrected: previous value of −0.0590 used MLRA N=1,066 (wrong)

In [ ]:
# ============================================================
# Verification only — not a manuscript table (checks S16 Table and Results/Discussion statements).
# Verification: DCA statements in Results/Discussion (validation cohort)
#   (1) Net benefit at 0.10–0.50 recomputed from stored predictions (6 decimals)
#   (2) Model with the highest net benefit at each threshold (full precision)
#   (3) MLRA net benefit < 0 at thresholds >= 0.35
#   (4) XGBoost Standard > MLRA at 0.15–0.50
#   (5) Prediction One net benefit < 0 at all thresholds
#   (6) Prediction One minimum predicted probability >= 0.20 and
#       Prediction One = Treat All wherever threshold <= that minimum
# Outputs: inline display + DCA_verification_validation.xlsx (downloaded)
# ============================================================
import numpy as np
import pandas as pd
from google.colab import files

TH = np.round(np.arange(0.10, 0.501, 0.05), 2)          # 0.10, 0.15, ..., 0.50 (9)

def nb(y, p, t):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    n = len(y); pos = p >= t
    tp = np.sum(pos & (y == 1)); fp = np.sum(pos & (y == 0))
    return tp / n - fp / n * (t / (1 - t))

y_full = np.asarray(all_results['XGBoost_Standard']['val_true']).astype(int)   # N = 1,243
prev   = y_full.mean()

rows, n_info = [], {}
for name, res in all_results.items():
    y, p = res['val_true'], res['val_proba']
    n_info[name] = (len(y), int(np.sum(y)))
    rows.append(pd.Series({t: nb(y, p, t) for t in TH}, name=name))
nbt = pd.DataFrame(rows).T
nbt['Treat All'] = [prev - (1 - prev) * t / (1 - t) for t in TH]
nbt.index.name = 'Threshold'

pd.set_option('display.width', 250)
print("N / events used per model:", n_info)
print("\n(1) Net benefit, validation cohort (6 dp)")
print(nbt.round(6).to_string())

models = [c for c in nbt.columns if c != 'Treat All']
best = nbt[models].idxmax(axis=1)
print("\n(2) Highest net benefit per threshold (full precision)")
for t in TH:
    top2 = nbt.loc[t, models].sort_values(ascending=False).head(2)
    print(f"  {t:.2f}: {top2.index[0]} {top2.iloc[0]:.6f}  | 2nd {top2.index[1]} {top2.iloc[1]:.6f}")
n_xgb = int((best == 'XGBoost_Standard').sum())
print(f"  XGBoost Standard highest at {n_xgb} of {len(TH)} thresholds: "
      f"{[f'{t:.2f}' for t in TH if best[t] == 'XGBoost_Standard']}")

chk3 = bool((nbt.loc[TH >= 0.35, 'MLRA'] < 0).all())
first_neg = next((f'{t:.2f}' for t in TH if nbt.loc[t, 'MLRA'] < 0), None)
print("\n(3) MLRA < 0 at thresholds >= 0.35:", chk3, "| first negative threshold:", first_neg)

chk4 = bool((nbt.loc[TH >= 0.15, 'XGBoost_Standard'] > nbt.loc[TH >= 0.15, 'MLRA']).all())
print("(4) XGBoost Standard > MLRA at 0.15–0.50:", chk4)

po = 'Prediction_One'
chk5 = bool((nbt[po] < 0).all())
print("(5) Prediction One < 0 at all thresholds:", chk5)

p_po = np.asarray(all_results[po]['val_proba'], dtype=float)
pmin = p_po.min()
print(f"(6) Prediction One validation predictions: min = {pmin:.6f}, min >= 0.20: {pmin >= 0.20}")
eq = [t for t in TH if t <= pmin]
eq_res = {f'{t:.2f}': bool(np.isclose(nbt.loc[t, po], nbt.loc[t, 'Treat All'])) for t in eq}
print("    Prediction One = Treat All at thresholds <= min:", eq_res)

# ── Export and download ──────────────────────────────────────
summary = pd.DataFrame({
    'Threshold'          : [f'{t:.2f}' for t in TH],
    'Highest_NB_model'   : [best[t] for t in TH],
    'Highest_NB'         : [nbt.loc[t, best[t]] for t in TH],
    'XGB_Std_minus_MLRA' : [nbt.loc[t, 'XGBoost_Standard'] - nbt.loc[t, 'MLRA'] for t in TH],
})
checks = pd.DataFrame({
    'Check': ['XGBoost Standard highest (thresholds)',
              'MLRA < 0 at >= 0.35',
              'XGBoost Standard > MLRA at 0.15-0.50',
              'Prediction One < 0 at all thresholds',
              'Prediction One minimum predicted probability',
              'Prediction One = Treat All at thresholds <= minimum'],
    'Result': [f"{n_xgb} of {len(TH)}", chk3, chk4, chk5,
               round(float(pmin), 6), str(eq_res)],
})
n_df = pd.DataFrame([{'Model': k, 'N': v[0], 'Events': v[1]} for k, v in n_info.items()])

DCA_VERIF_XLSX = 'verification_DCA_for_S16_Table_and_Results.xlsx'
with pd.ExcelWriter(DCA_VERIF_XLSX, engine='openpyxl') as w:
    nbt.round(6).reset_index().to_excel(w, sheet_name='NetBenefit_6dp', index=False)
    summary.to_excel(w, sheet_name='Highest_by_threshold', index=False)
    checks.to_excel(w, sheet_name='Checks', index=False)
    n_df.to_excel(w, sheet_name='N_events', index=False)
print(f"\n✓ Saved: {DCA_VERIF_XLSX}")
files.download(DCA_VERIF_XLSX)

In [ ]:
# =============================================================================
# Section DCA-Extended: Figure — DCA Comparison
#   XGBoost Standard (Blue) vs MLRA (Black)
#   Development cohort: solid line  |  Validation cohort: dashed line
#   Resolution: 600 DPI  |  Format: PNG + TIFF
# =============================================================================
# [FIXED] NameError: train_xgb_proba / train_mlra_proba not defined
#   → All predictions are reconstructed from mlra_results and ml_results
#     which persist throughout the notebook session.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files as colab_files

# ─── Prerequisites check ──────────────────────────────────────────────────────
print("=" * 70)
print("Section DCA-Extended: XGBoost Standard vs MLRA")
print("=" * 70)

try:
    _ = mlra_results
    _ = ml_results
    _ = train_df   # raw training dataframe (defined in Data Loading section)
    _ = val_df     # raw validation dataframe
    print("✓ mlra_results, ml_results, train_df, val_df are available")
except NameError as e:
    raise RuntimeError(
        f"❌ Required variable not found: {e}\n"
        "Run the Data Loading and Model Training sections first."
    )

# ─── Reconstruct predictions from stored results ──────────────────────────────
# MLRA
_mlra_train_proba = mlra_results['train_proba']
_mlra_val_proba   = mlra_results['val_proba']
_mlra_train_true  = mlra_results['train_true']
_mlra_val_true    = mlra_results['val_true']

# XGBoost Standard
_xgb_train_proba = ml_results['XGBoost_Standard']['train_proba']
_xgb_val_proba   = ml_results['XGBoost_Standard']['val_proba']
_xgb_train_true  = ml_results['XGBoost_Standard']['train_true']
_xgb_val_true    = ml_results['XGBoost_Standard']['val_true']

# Resolve index for MLRA complete-case alignment
# MLRA uses listwise deletion → its index is a subset of the raw index
if hasattr(_mlra_train_true, 'index'):
    _mlra_dev_idx = _mlra_train_true.index
else:
    _mlra_dev_idx = pd.RangeIndex(len(_mlra_train_true))

if hasattr(_mlra_val_true, 'index'):
    _mlra_val_idx = _mlra_val_true.index
else:
    _mlra_val_idx = pd.RangeIndex(len(_mlra_val_true))

# ─── Align XGBoost predictions to MLRA complete-case indices ──────────────────
# Development cohort
if hasattr(_xgb_train_true, 'index'):
    _xgb_dev_series = pd.Series(
        _xgb_train_proba
        if not hasattr(_xgb_train_proba, 'values')
        else _xgb_train_proba.values,
        index=_xgb_train_true.index
    )
else:
    _xgb_dev_series = pd.Series(
        _xgb_train_proba
        if not hasattr(_xgb_train_proba, 'values')
        else _xgb_train_proba.values
    )

# Validation cohort
if hasattr(_xgb_val_true, 'index'):
    _xgb_val_series = pd.Series(
        _xgb_val_proba
        if not hasattr(_xgb_val_proba, 'values')
        else _xgb_val_proba.values,
        index=_xgb_val_true.index
    )
else:
    _xgb_val_series = pd.Series(
        _xgb_val_proba
        if not hasattr(_xgb_val_proba, 'values')
        else _xgb_val_proba.values
    )

# Restrict XGBoost to MLRA indices (valid paired comparison)
try:
    _xgb_dev_aligned = _xgb_dev_series.loc[_mlra_dev_idx].values
    _xgb_val_aligned = _xgb_val_series.loc[_mlra_val_idx].values
except KeyError:
    # Fallback: indices don't match → use as-is (same length assumed)
    print("⚠  Index alignment failed; using predictions as-is.")
    _xgb_dev_aligned = np.array(_xgb_train_proba
                                 if not hasattr(_xgb_train_proba, 'values')
                                 else _xgb_train_proba.values)
    _xgb_val_aligned = np.array(_xgb_val_proba
                                 if not hasattr(_xgb_val_proba, 'values')
                                 else _xgb_val_proba.values)

# Ground-truth outcomes (MLRA cohort)
_y_dev = (np.array(_mlra_train_true.values
                    if hasattr(_mlra_train_true, 'values')
                    else _mlra_train_true)).astype(int)
_y_val = (np.array(_mlra_val_true.values
                    if hasattr(_mlra_val_true, 'values')
                    else _mlra_val_true)).astype(int)

# MLRA probabilities aligned to same index
_mlra_dev_proba_arr = (np.array(_mlra_train_proba.values
                                 if hasattr(_mlra_train_proba, 'values')
                                 else _mlra_train_proba))
_mlra_val_proba_arr = (np.array(_mlra_val_proba.values
                                 if hasattr(_mlra_val_proba, 'values')
                                 else _mlra_val_proba))

print(f"\n✓ Sample sizes confirmed")
print(f"  Development — MLRA: {len(_y_dev)}  "
      f"| XGBoost (aligned): {len(_xgb_dev_aligned)}")
print(f"  Validation  — MLRA: {len(_y_val)}  "
      f"| XGBoost (aligned): {len(_xgb_val_aligned)}")

# ─── Net Benefit functions ────────────────────────────────────────────────────
def compute_net_benefit(y_true, y_prob, thresholds):
    """
    Net Benefit = (TP/N) - (FP/N) × pt/(1−pt)
    Reference: Vickers & Elkin, Med Decis Making 2006; 26(6):565-574
    """
    y_true = np.array(y_true, dtype=int)
    y_prob = np.array(y_prob, dtype=float)
    n      = len(y_true)
    nb     = np.full(len(thresholds), np.nan)
    for i, pt in enumerate(thresholds):
        if pt <= 0 or pt >= 1:
            continue
        y_pred = (y_prob >= pt).astype(int)
        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        nb[i] = (tp / n) - (fp / n) * (pt / (1.0 - pt))
    return nb


def compute_treat_all(y_true, thresholds):
    """Net benefit of the 'treat all' strategy."""
    y_true = np.array(y_true, dtype=int)
    n      = len(y_true)
    prev   = y_true.mean()
    nb     = np.full(len(thresholds), np.nan)
    for i, pt in enumerate(thresholds):
        if pt <= 0 or pt >= 1:
            continue
        nb[i] = prev - (1.0 - prev) * (pt / (1.0 - pt))
    return nb

# ─── Threshold grid ───────────────────────────────────────────────────────────
_thresholds = np.linspace(0.01, 0.50, 200)

# ─── Compute net benefit ──────────────────────────────────────────────────────
# [FIXED] BUG-3: XGBoost NB now computed on full cohort (N=1,243 val / N=756 dev)
# using native missing-data handling, not on MLRA complete-case subset.
# MLRA NB remains on its complete-case population (correct per its design).
_y_dev_full = train_df['Outcome'].values.astype(int)   # N=756
_y_val_full = val_df['Outcome'].values.astype(int)     # N=1,243

_xgb_val_proba_full = np.array(
    ml_results['XGBoost_Standard']['val_proba'].values
    if hasattr(ml_results['XGBoost_Standard']['val_proba'], 'values')
    else ml_results['XGBoost_Standard']['val_proba']
)
_xgb_dev_proba_full = np.array(
    ml_results['XGBoost_Standard']['train_proba'].values
    if hasattr(ml_results['XGBoost_Standard']['train_proba'], 'values')
    else ml_results['XGBoost_Standard']['train_proba']
)

_nb_xgb_dev  = compute_net_benefit(_y_dev_full, _xgb_dev_proba_full, _thresholds)
_nb_xgb_val  = compute_net_benefit(_y_val_full, _xgb_val_proba_full, _thresholds)
_nb_mlra_dev = compute_net_benefit(_y_dev, _mlra_dev_proba_arr, _thresholds)
_nb_mlra_val = compute_net_benefit(_y_val, _mlra_val_proba_arr, _thresholds)

# [FIXED] BUG-3: Treat All uses full cohort outcomes (N=756 dev / N=1,243 val)
_nb_treat_all_dev = compute_treat_all(_y_dev_full, _thresholds)  # N=756
_nb_treat_all_val = compute_treat_all(_y_val_full, _thresholds)  # N=1,243
_nb_treat_none    = np.zeros(len(_thresholds))

# ─── Plot ─────────────────────────────────────────────────────────────────────
# [FIXED] TIFF removed; replaced with JPG (300+ DPI)
_DCA_FIG_PNG  = 'Fig6_DCA_XGBoost_Standard_vs_MLRA.jpg'

fig_dca, ax_dca = plt.subplots(figsize=(10, 6.5))

# XGBoost Standard (Blue)
ax_dca.plot(_thresholds, _nb_xgb_dev,
            color='royalblue', linewidth=2.5, linestyle='-',
            label='XGBoost Standard — Development')
ax_dca.plot(_thresholds, _nb_xgb_val,
            color='royalblue', linewidth=2.5, linestyle='--',
            label='XGBoost Standard — Validation')

# MLRA (Black)
ax_dca.plot(_thresholds, _nb_mlra_dev,
            color='black', linewidth=2.5, linestyle='-',
            label='MLRA — Development')
ax_dca.plot(_thresholds, _nb_mlra_val,
            color='black', linewidth=2.5, linestyle='--',
            label='MLRA — Validation')

# Reference strategies
ax_dca.plot(_thresholds, _nb_treat_all_dev,
            color='gray', linewidth=1.5, linestyle=':',
            label='Treat All — Development')
ax_dca.plot(_thresholds, _nb_treat_all_val,
            color='darkgray', linewidth=1.5, linestyle=':',
            label='Treat All — Validation')
ax_dca.plot(_thresholds, _nb_treat_none,
            color='silver', linewidth=1.5, linestyle='-.',
            label='Treat None')

# Axis limits
_all_nb = np.concatenate([
    _nb_xgb_dev[~np.isnan(_nb_xgb_dev)],
    _nb_xgb_val[~np.isnan(_nb_xgb_val)],
    _nb_mlra_dev[~np.isnan(_nb_mlra_dev)],
    _nb_mlra_val[~np.isnan(_nb_mlra_val)],
])
ax_dca.set_xlim(0.0, 0.50)
ax_dca.set_ylim(
    min(-0.01, _all_nb.min() - 0.005),
    _all_nb.max() + 0.015,
)

ax_dca.set_xlabel('Threshold Probability', fontsize=13, fontweight='bold')
ax_dca.set_ylabel('Net Benefit',           fontsize=13, fontweight='bold')
# Figure title removed (PLOS ONE: the legend carries the title)
ax_dca.axhline(y=0, color='black', linewidth=0.8, alpha=0.4)
ax_dca.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)
ax_dca.legend(loc='upper right', fontsize=9.5,
              framealpha=0.92, edgecolor='black', frameon=True)

plt.tight_layout()

# ─── Save ─────────────────────────────────────────────────────────────────────
fig_dca.savefig(_DCA_FIG_PNG, dpi=600, bbox_inches='tight', facecolor='white',
                format='jpeg', pil_kwargs={'quality': 95})
print(f"\n✓ Saved: {_DCA_FIG_PNG}")
plt.show()

# ─── Download ─────────────────────────────────────────────────────────────────
print("\nDownloading DCA figures...")
for _fpath in [_DCA_FIG_PNG]:
    if os.path.exists(_fpath):
        try:
            colab_files.download(_fpath)
            print(f"  ✓ Downloaded: {_fpath}")
        except Exception as _e:
            print(f"  ⚠  Failed: {_fpath} — {_e}")
    else:
        print(f"  ✗ Not found: {_fpath}")

# ─── Reproducibility checkpoint ───────────────────────────────────────────────
_idx_010 = np.argmin(np.abs(_thresholds - 0.10))
print(f"\nNet Benefit at threshold = 0.10:")
print(f"  XGBoost Dev  : {_nb_xgb_dev[_idx_010]:.4f}")
print(f"  XGBoost Val  : {_nb_xgb_val[_idx_010]:.4f}")
print(f"  MLRA Dev     : {_nb_mlra_dev[_idx_010]:.4f}")
print(f"  MLRA Val     : {_nb_mlra_val[_idx_010]:.4f}")
# ── Values should be identical across re-runs (deterministic calculation) ──

## Section 12 · SHAP Analysis (Top-3 Models by Validation AUC)

In [ ]:
# ============================================================
# Section 12: SHAP Analysis
# Tree-based SHAP for XGBoost / LightGBM / CatBoost;
# Permutation-based SHAP (shap.Explainer) for HistGradientBoosting.
# ============================================================

import shap

set_global_seed(RANDOM_STATE)

# Select top-3 ML models by validation AUC (deterministic sort)
ml_aucs = {
    n: roc_auc_score(_get_arr(r['val_true']), np.asarray(r['val_proba']))
    for n, r in all_results.items()
    if n not in ('MLRA', 'Prediction_One')
}
top3 = sorted(ml_aucs, key=lambda x: (-ml_aucs[x], x))[:3]
print("Top-3 models for SHAP:", top3)

shap_vals = {}
for name in top3:
    mdl = ml_results[name]['model']
    print(f"  Computing SHAP for {name}...")
    try:
        if any(k in name for k in ('XGBoost', 'LightGBM', 'CatBoost')):
            exp   = shap.TreeExplainer(mdl)
            sv    = exp.shap_values(X_val)
            sv    = sv[1] if isinstance(sv, list) else sv
        else:
            exp_g = shap.Explainer(mdl, X_train)
            sv    = exp_g(X_val).values
        shap_vals[name] = {'shap': sv, 'auc': ml_aucs[name]}
        print(f"    ✓ Shape: {sv.shape}")
    except Exception as e:
        print(f"    ✗ Error: {e}")

shap_files = []

# ── SHAP bar plot, top-3 models (supplementary check; not a manuscript figure) ───
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for idx, name in enumerate(top3):
    if name not in shap_vals:
        continue
    sv  = shap_vals[name]['shap']
    imp = pd.DataFrame({'feature': X_train.columns,
                         'importance': np.mean(np.abs(sv), axis=0)
                        }).sort_values('importance', ascending=False).head(20)
    ax  = axes[idx]
    yp  = np.arange(len(imp))
    ax.barh(yp, imp['importance'], color='#1f77b4', alpha=0.8)
    ax.set_yticks(yp); ax.set_yticklabels(imp['feature'], fontsize=9)
    ax.invert_yaxis()
    for i, (_, row) in enumerate(imp.iterrows()):
        ax.text(row['importance'] * 1.02, i, f"{row['importance']:.3f}",
                va='center', fontsize=8)
    ax.set_xlabel('Mean |SHAP value|', fontsize=11, fontweight='bold')
    ax.set_title(f'#{idx+1}: {name}\n(AUC={ml_aucs[name]:.4f})',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3, ls='--')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
plt.tight_layout()
f8 = 'figure8_shap_bar_top3.jpg'
plt.savefig(f8, dpi=300, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
shap_files.append(f8)
plt.show()
plt.close(fig)
print(f"✓ Saved {f8}")

# ── Figure 9: SHAP Beeswarm for each top-3 model ──────────
for idx, name in enumerate(top3):
    if name not in shap_vals:
        continue
    sv = shap_vals[name]['shap']
    fig = plt.figure(figsize=(8, 8))
    shap.summary_plot(sv, X_val, max_display=20, show=False,
                      plot_type='dot', color_bar=True)
    plt.tight_layout()
    fname = f'figure9{chr(65+idx)}_shap_beeswarm_{name}.jpg'
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white',
                format='jpeg', pil_kwargs={'quality': 95})
    shap_files.append(fname)
    plt.show()
    plt.close(fig)
    print(f"✓ Saved {fname}")

# ── S9 Fig: SHAP dependence plots, XGBoost Standard (validation cohort, n = 1,243) ──
# Features with non-zero SHAP values (max 20), ordered by the same rule as Fig 8 / shap.summary_plot.
# PLOS ONE: drawn at final size (7.5 in wide, <= 8.75 in high), Arial 8 pt, 600 dpi; no title in image.
# Missing feature values are not plotted.
# SHAP_DISPLAY_LABELS is also used by the Fig 8 cell (identical labels in Fig 8 and S9 Fig).
S9_MODEL = 'XGBoost_Standard'
SHAP_DISPLAY_LABELS = {   # display labels only; data and column names unchanged
    'White blood cells (103/μL)'  : r'White blood cells ($\mathregular{10^3}$/μL)',
    'Plt (104/μL)'                : r'Platelets ($\mathregular{10^4}$/μL)',
    'Aspartate transaminase (U/L)': 'Aspartate aminotransferase (U/L)',
    'RPG (mg/dL)'                 : 'Random plasma glucose (mg/dL)',
    'Hb (g/dL)'                   : 'Hemoglobin (g/dL)',
    'APTT (sec)'                  : 'APTT (s)',
    'Serum Fe'                    : 'Serum iron (μg/dL)',   # not displayed (all SHAP = 0); unit unconfirmed
}
S9_LOG_X = [    # option A (log x axis); set S9_LOG_X = [] for option B (linear)
    'Lactate dehydrogenase (U/L)', 'Aspartate transaminase (U/L)',
    'White blood cells (103/μL)', 'Creatine kinase (U/L)', 'γ-Glutamyl transferase (U/L)',
]

if S9_MODEL in shap_vals:
    sv_s9 = np.asarray(shap_vals[S9_MODEL]['shap'], dtype=float)
    cols_s9 = list(X_val.columns)
    assert sv_s9.shape == X_val.shape and len(X_val) == 1243, 'S9 Fig: SHAP/X_val shape mismatch — HALT'
    assert cols_s9 == list(X_train.columns), 'S9 Fig: column order differs from training — HALT'
    assert all(c in cols_s9 for c in S9_LOG_X), 'S9 Fig: S9_LOG_X name not found — HALT'

    abs_sum     = np.abs(sv_s9).sum(axis=0)
    mean_abs_s9 = np.abs(sv_s9).mean(axis=0)
    k_nz        = int((abs_sum > 0).sum())
    order_s9    = np.argsort(abs_sum)[-min(k_nz, 20):][::-1]       # same tie rule as Fig 8
    assert all(abs_sum[j] > 0 for j in order_s9)
    print(f'S9 Fig: features with non-zero SHAP values: {k_nz} of {len(cols_s9)}')
    _ua = cols_s9.index('Uric acid (mg/dL)')
    print(f'  Uric acid mean |SHAP| = {mean_abs_s9[_ua]:.6f}')

    ncols_s9 = 4
    nrows_s9 = int(np.ceil(len(order_s9) / ncols_s9))
    with plt.rc_context({'font.family': 'sans-serif',
                         'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
                         'font.size': 8, 'mathtext.default': 'regular'}):
        fig, axes = plt.subplots(nrows_s9, ncols_s9, figsize=(7.5, 1.65 * nrows_s9), squeeze=False)
        for i, j in enumerate(order_s9):
            ax = axes[i // ncols_s9][i % ncols_s9]
            col = cols_s9[j]
            x = X_val[col].to_numpy(dtype=float)
            s = sv_s9[:, j]
            m = ~np.isnan(x)
            if col in S9_LOG_X:
                assert (x[m] > 0).all(), f'S9 Fig: {col} has non-positive values; log axis invalid — HALT'
                ax.set_xscale('log')
            ax.scatter(x[m], s[m], s=3, alpha=0.5, linewidths=0, color='#1f77b4')
            ax.axhline(0, color='0.5', lw=0.5)
            ax.set_title(f'{mean_abs_s9[j]:.3f}', fontsize=8, loc='right', pad=2)
            ax.set_xlabel(SHAP_DISPLAY_LABELS.get(col, col), fontsize=8)
            ax.margins(y=0.08)
            ax.tick_params(labelsize=8, width=0.6, length=2)
            for sp in ax.spines.values():
                sp.set_linewidth(0.6)
            print(f'  {i+1:2d}. {col:32s} plotted n = {m.sum():4d}, mean|SHAP| = {mean_abs_s9[j]:.6f}, '
                  f'x range = {np.nanmin(x):.4g}–{np.nanmax(x):.4g}')
        for i in range(len(order_s9), nrows_s9 * ncols_s9):
            axes[i // ncols_s9][i % ncols_s9].axis('off')
        fig.supylabel('SHAP value (log-odds)', fontsize=8)
        fig.tight_layout()
        w_in, h_in = fig.get_size_inches()
        assert w_in <= 7.5 + 1e-6 and h_in <= 8.75 + 1e-6, 'S9 Fig: exceeds PLOS size — HALT'
        f10 = 'S9_Fig_SHAP_dependence_XGBoost_Standard.jpg'   # file name read by the final TIFF export cell
        plt.savefig(f10, dpi=600, facecolor='white', format='jpeg', pil_kwargs={'quality': 95})
        shap_files.append(f10)
        plt.show()
        plt.close(fig)
    print(f"✓ Saved {f10}: {w_in:.2f} x {h_in:.2f} in, 600 dpi")

for f in shap_files:
    files.download(f)
print("\n✓ SHAP analysis complete")
# ── Verification: Top-3 model order and mean |SHAP| for #1 feature must match original ──


In [ ]:
# ============================================================
# Verification: SHAP sign-transition thresholds
# (LDH, albumin, potassium, WBC, AST — as described in the Results)
# XGBoost Standard, validation cohort (n = 1,243)
# Run after Section 12 (requires shap_vals, ml_results, X_val).
# Verification only — not a manuscript table.
# Outputs: inline display + verification_SHAP_thresholds.xlsx (downloaded)
# ============================================================
import numpy as np
import pandas as pd
from google.colab import files

MODEL_NAME = 'XGBoost_Standard'
sv_xgb  = shap_vals[MODEL_NAME]['shap']                 # expected (1243, 43)
booster = ml_results[MODEL_NAME]['model'].get_booster()
tree_df = booster.trees_to_dataframe()

print(f"X_val: {X_val.shape}, SHAP: {sv_xgb.shape}")
assert sv_xgb.shape == X_val.shape and len(X_val) == 1243, "Shape mismatch — HALT"

# key: substring matched to exactly one column; bins: predictor-specific (right-open)
# reported: threshold(s) stated in the Results text
TARGETS = {
    'LDH':       dict(key='Lactate dehydrogenase', unit='U/L',
                      bins=[0, 200, 300, 350, 400, 450, 500, 750, 1000, np.inf],
                      reported='positive >~400; negative below'),
    'Albumin':   dict(key='Albumin', unit='g/dL',
                      bins=[0, 2.0, 2.5, 2.8, 3.0, 3.5, 4.0, np.inf],
                      reported='positive <~2.8; negative >=~3.0'),
    'Potassium': dict(key='Potassium', unit='mEq/L',
                      bins=[0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, np.inf],
                      reported='positive >~5; negative below'),
    'WBC':       dict(key='White blood cells', unit='x10^3/uL',
                      bins=[0, 4, 8, 10, 12, 15, 20, np.inf],
                      reported='positive >~12; negative <~10'),
    'AST':       dict(key='Aspartate', unit='U/L',
                      bins=[0, 20, 30, 35, 40, 45, 50, 100, np.inf],
                      reported='positive >~40; negative below'),
}

summary_rows, band_tables, split_tables = [], {}, {}
for label, cfg in TARGETS.items():
    cols = [c for c in X_val.columns if cfg['key'].lower() in c.lower()]
    assert len(cols) == 1, f"{label}: matched {cols} — HALT"
    col  = cols[0]
    fidx = list(X_val.columns).index(col)
    x    = X_val[col].values.astype(float)
    s    = sv_xgb[:, fidx]
    ok   = ~np.isnan(x)
    x, s = x[ok], s[ok]

    # (1) Split points learned by XGBoost for this feature
    splits = (tree_df.loc[tree_df['Feature'] == col, 'Split']
              .round(2).value_counts().sort_index())

    # (2) Empirical sign boundaries in the validation data
    max_neg = x[s < 0].max() if (s < 0).any() else np.nan
    min_pos = x[s > 0].min() if (s > 0).any() else np.nan
    min_neg = x[s < 0].min() if (s < 0).any() else np.nan
    max_pos = x[s > 0].max() if (s > 0).any() else np.nan

    # (3) Proportion of positive SHAP and median SHAP by concentration band
    band = pd.cut(x, bins=cfg['bins'], right=False)
    tab  = (pd.DataFrame({'band': band, 'pos': s > 0, 'shap': s})
            .groupby('band', observed=True)
            .agg(n=('pos', 'size'),
                 prop_positive=('pos', 'mean'),
                 median_shap=('shap', 'median')))

    print(f"\n{'=' * 64}\n{label}: {col}  (n with value = {ok.sum()} / {len(ok)})")
    print(f"  Reported in Results : {cfg['reported']} {cfg['unit']}")
    print(f"  Split points (value: count):\n{splits.to_string()}")
    print(f"  SHAP<0 range        : {min_neg:.2f} – {max_neg:.2f}")
    print(f"  SHAP>0 range        : {min_pos:.2f} – {max_pos:.2f}")
    print(tab.round(3).to_string())

    band_tables[label]  = tab.reset_index().assign(band=lambda d: d['band'].astype(str))
    split_tables[label] = splits.rename_axis('Split').reset_index(name='Count')
    summary_rows.append({'Predictor': label, 'Column': col, 'n': int(ok.sum()),
                         'SHAP<0 min': min_neg, 'SHAP<0 max': max_neg,
                         'SHAP>0 min': min_pos, 'SHAP>0 max': max_pos,
                         'Reported': cfg['reported']})

summary_df = pd.DataFrame(summary_rows)
print(f"\n{'=' * 64}\nSummary")
print(summary_df.round(2).to_string(index=False))

# ── Export and download ──────────────────────────────────────
SHAP_VERIF_XLSX = 'verification_SHAP_thresholds.xlsx'
with pd.ExcelWriter(SHAP_VERIF_XLSX, engine='openpyxl') as w:
    summary_df.to_excel(w, sheet_name='Summary', index=False)
    for label in TARGETS:
        band_tables[label].to_excel(w, sheet_name=f'{label}_bands', index=False)
        split_tables[label].to_excel(w, sheet_name=f'{label}_splits', index=False)
print(f"\n✓ Saved: {SHAP_VERIF_XLSX}")
files.download(SHAP_VERIF_XLSX)

In [ ]:
# ============================================================
# Verification: SHAP additivity (XGBoost Standard, validation cohort)
#   base value + sum of SHAP values = model margin (log-odds)
#   expected_value is read AFTER shap_values() is called, because the
#   base value is finalized at that call in the SHAP version used.
# Verification only — does not overwrite shap_vals.
# ============================================================
import numpy as np, shap

mdl    = ml_results['XGBoost_Standard']['model']
exp    = shap.TreeExplainer(mdl)
ev_before = float(np.ravel(exp.expected_value)[0])
sv_chk = exp.shap_values(X_val)
sv_chk = sv_chk[1] if isinstance(sv_chk, list) else sv_chk
ev_after  = float(np.ravel(exp.expected_value)[0])

margin = mdl.predict(X_val, output_margin=True)
recon  = ev_after + shap_vals['XGBoost_Standard']['shap'].sum(axis=1)

print(f"expected_value before shap_values(): {ev_before:.6f}")
print(f"expected_value after  shap_values(): {ev_after:.6f}  (difference {ev_after - ev_before:+.6f})")
print(f"Section 12 SHAP identical to recomputed: {np.allclose(sv_chk, shap_vals['XGBoost_Standard']['shap'])}")
print(f"max |margin - (base + sum SHAP)| = {np.max(np.abs(margin - recon)):.2e}")
assert np.allclose(margin, recon, atol=1e-4), "SHAP additivity violated — HALT"
print("✓ SHAP additivity holds")

## Fig 8 · XGBoost Standard SHAP summary (A: mean |SHAP| bar plot; B: beeswarm) — PLOS ONE TIFF

In [ ]:
# ============================================================
# Fig 8: XGBoost Standard SHAP summary — (A) mean |SHAP| bar plot, (B) beeswarm
# Validation cohort (n = 1,243); features with non-zero SHAP values (max 20),
# ranked by mean absolute SHAP value; same features and order as S9 Fig.
# Run after Section 12 (requires shap_vals, X_val, SHAP_DISPLAY_LABELS, order_s9).
#
# PLOS ONE figure specifications applied:
#   TIFF with LZW compression, RGB (no alpha), 600 dpi
#   width 7.5 in (maximum), height <= 8.75 in
#   Arial (or the metric-compatible Liberation Sans), text 8-12 pt
#   no figure title in the image; panels labelled A and B
# ============================================================
import io
import numpy as np
import matplotlib.pyplot as plt
import shap
from PIL import Image
from IPython.display import Image as _Img_f8, display as _disp_f8
from google.colab import files

FIG8_TIF  = 'Fig8.tif'
DPI_PLOS  = 600
MAX_W_IN, MAX_H_IN = 7.5, 8.75
MODEL_F8  = 'XGBoost_Standard'
assert 'SHAP_DISPLAY_LABELS' in globals() and 'order_s9' in globals(), 'Run Section 12 first — HALT'
FIG8_LABELS = SHAP_DISPLAY_LABELS   # defined in Section 12 (S9 Fig): identical labels in Fig 8 and S9 Fig


def save_plos_tiff(fig, path, dpi=DPI_PLOS):
    """Save a matplotlib figure as an RGB, LZW-compressed TIFF meeting PLOS ONE size limits."""
    buf = io.BytesIO()
    with plt.rc_context({'savefig.bbox': 'standard'}):          # keep the set figure size (no tight crop)
        fig.savefig(buf, format='png', dpi=dpi, facecolor='white')
    buf.seek(0)
    img = Image.open(buf).convert('RGB')
    w_in, h_in = img.size[0] / dpi, img.size[1] / dpi
    assert w_in <= MAX_W_IN + 1e-6 and h_in <= MAX_H_IN + 1e-6, \
        f'Figure size {w_in:.2f} x {h_in:.2f} in exceeds PLOS limits — HALT'
    img.save(path, format='TIFF', compression='tiff_lzw', dpi=(dpi, dpi))
    print(f'✓ Saved {path}: {img.size[0]} x {img.size[1]} px, {w_in:.2f} x {h_in:.2f} in, '
          f'{dpi} dpi, RGB, LZW')


# ── Data: the same SHAP values used in Section 12 and S9 Fig ──────────────
assert MODEL_F8 in shap_vals, 'Run Section 12 first — HALT'
sv = np.asarray(shap_vals[MODEL_F8]['shap'], dtype=float)
assert sv.shape == X_val.shape, f'SHAP matrix {sv.shape} does not match X_val {X_val.shape} — HALT'
assert sv.shape[0] == 1243, 'SHAP values must be computed for the full validation cohort — HALT'

mean_abs = np.abs(sv).mean(axis=0)

# Features shown: non-zero SHAP values only (max 20); same set and order as S9 Fig
abs_sum_f8 = np.abs(sv).sum(axis=0)
N_TOP = min(int((abs_sum_f8 > 0).sum()), 20)
order = np.argsort(abs_sum_f8)[-N_TOP:][::-1]      # same ranking rule as shap.summary_plot
assert all(abs_sum_f8[j] > 0 for j in order), 'Zero-SHAP feature in Fig 8 — HALT'
assert list(order) == list(order_s9), 'Fig 8 and S9 Fig features differ — HALT'
print(f'Fig 8: {N_TOP} features with non-zero SHAP values (of {sv.shape[1]})')

feat_names = [str(c) for c in X_val.columns]
print('Top features (mean |SHAP|):')
for r, j in enumerate(order[:10], 1):
    print(f'  {r:2d}. {feat_names[j]:<40} {mean_abs[j]:.3f}')

# ── Figure ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 8,
    'mathtext.default': 'regular',
})
fig = plt.figure(figsize=(MAX_W_IN, 5.6))
gs  = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.15], wspace=0.06,
                       left=0.30, right=0.93, top=0.95, bottom=0.10)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)

# (B) beeswarm drawn first; SHAP places the most important feature at y = N_TOP - 1
plt.sca(ax_b)
shap.summary_plot(sv, X_val, max_display=N_TOP, show=False, plot_size=None,
                  plot_type='dot', color_bar=True)
# Feature order used by SHAP (top to bottom) must equal the bar-plot order
_ylabels_b = [t.get_text() for t in ax_b.get_yticklabels()][::-1]
_names_a = [feat_names[j] for j in order]
if _ylabels_b != _names_a:
    for _r, (_u, _v) in enumerate(zip(_names_a, _ylabels_b), 1):
        if _u != _v:
            print(f'  rank {_r}: panel A "{_u}" vs panel B "{_v}"')
assert _ylabels_b == _names_a, 'Feature order differs between panels — HALT'

# (A) bar plot on the same y positions
y_pos = np.arange(N_TOP)[::-1]                             # most important at top
ax_a.barh(y_pos, mean_abs[order], color='#1E88E5', height=0.7)
for y, v in zip(y_pos, mean_abs[order]):
    ax_a.text(v + mean_abs[order].max() * 0.015, y, f'{v:.3f}' if v >= 0.0005 else '<0.001',
              va='center', fontsize=8)
ax_a.set_yticks(y_pos)
ax_a.set_yticklabels([FIG8_LABELS.get(feat_names[j], feat_names[j]) for j in order], fontsize=8)
ax_a.set_xlabel('Mean |SHAP value|', fontsize=8.5)
ax_a.set_xlim(0, mean_abs[order].max() * 1.25)
for sp in ('top', 'right'):
    ax_a.spines[sp].set_visible(False)

# Panel B: labels shared with panel A; harmonize font sizes set by SHAP
ax_b.tick_params(axis='y', labelleft=False)
ax_b.set_xlabel('SHAP value (impact on model output, log-odds)', fontsize=8.5)
ax_b.tick_params(axis='x', labelsize=8)
for ax in fig.axes:
    if ax not in (ax_a, ax_b):                             # SHAP colour bar
        ax.tick_params(labelsize=8)
        ax.set_ylabel(ax.get_ylabel(), fontsize=8)
ax_a.tick_params(axis='x', labelsize=8)

# Panel labels (no figure title, per PLOS ONE)
ax_a.text(-0.02, 1.02, 'A', transform=ax_a.transAxes, fontsize=10, fontweight='bold',
          ha='right', va='bottom')
ax_b.text(0.0, 1.02, 'B', transform=ax_b.transAxes, fontsize=10, fontweight='bold',
          ha='left', va='bottom')

save_plos_tiff(fig, FIG8_TIF)
plt.show()
plt.close(fig)

# Inline preview of the saved file (PNG copy for display only)
Image.open(FIG8_TIF).save('Fig8_preview.png')
_disp_f8(_Img_f8('Fig8_preview.png', width=900))
files.download(FIG8_TIF)

## Section 17 · Subgroup Analysis + Forest Plot (MLRA vs XGBoost Standard)
**Prerequisite**: Sections 3, 5, 7 (train_df, val_df, all_results in memory). The development and validation CSV files are uploaded again in this cell; row alignment with the main dataset is checked by assert.

- Subgroups: sex, age (<65 / 65–74 / ≥75 years), albumin (g/dL), CRP (mg/dL), CCI (development cohort only)
- MLRA: complete cases on the 4 predictors; XGBoost Standard: final-model predictions (native missing-value handling)
- Cutoffs: development-derived medians (MLRA complete-case sample; full cohort for XGBoost Standard)
- Outputs: `S6_Fig_Subgroup_Forest_Plot.jpg` (S6 Fig), `S17_Table_Subgroup_Analysis.xlsx` (S17 Table, publication format with numeric cutoffs), `S17_Table_Subgroup_MLRA.csv`, `S17_Table_Subgroup_XGBoost_Standard.csv` (unrounded values)

In [ ]:
# Cell 59============================================================
# Section 17 · Subgroup Analysis + Forest Plot (MLRA vs XGBoost Standard)
# Development vs Validation
# Age strata: <65, 65–74, ≥75 (display order fixed)
# Development subgroups: Sex, Age, Albumin (g/L), CRP (mg/L), CCI
# Validation subgroups:  Sex, Age, Albumin (g/L), CRP (mg/L) (+ keep CCI space)
# MLRA: complete-case on 4 variables | XGB: native missing; hyperparameters fixed; DO NOT change
# Cutoffs: analytic-sample medians (MLRA complete-case; XGB full cohort), development-derived
#
# [FIX-1] Remove duplicated "category==subgroup name" rows
# [FIX-2] Tune vertical spacing (header-to-first-category + between blocks)
# [FIX-3] Keep CCI empty block in Validation to match Development layout
# [FIX-4] Labels unified to "Low (≤ median)/High (> median)" for Albumin/CRP/CCI; SI units in headers.
#         CRP membership stays model-specific via CRP_group; the shared label collapses
#         MLRA and XGBoost onto common Low/High rows (removes the 0.99/0.92 four-row clutter).
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier  # (no longer used for fitting; final model predictions are used)
from google.colab import files

# --- Version control (Coding protocol Tier 2 §9) ---
import xgboost, sklearn, scipy, matplotlib
for _lib in [np, pd, scipy, sklearn, xgboost, matplotlib]:
    print(f"{_lib.__name__}: {_lib.__version__}")

plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

COL_MLRA = "#2B2B2B"   # near-black
COL_XGB  = "#2F6FAE"   # muted blue
ALPHA_POINT = 0.95
ALPHA_CI = 0.55

# -------------------------
# DeLong AUC CI
# -------------------------
def delong_roc_variance(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(float)

    order = np.argsort(y_pred)
    y_true = y_true[order]
    y_pred = y_pred[order]

    n1 = np.sum(y_true == 1)
    n0 = np.sum(y_true == 0)
    if n1 == 0 or n0 == 0:
        return None

    pos = np.where(y_true == 1)[0]
    neg = np.where(y_true == 0)[0]

    V10 = np.zeros(n1, dtype=float)
    for i, pr in enumerate(pos):
        V10[i] = np.sum(y_pred[neg] < y_pred[pr]) + 0.5 * np.sum(y_pred[neg] == y_pred[pr])

    V01 = np.zeros(n0, dtype=float)
    for i, nr in enumerate(neg):
        V01[i] = np.sum(y_pred[pos] > y_pred[nr]) + 0.5 * np.sum(y_pred[pos] == y_pred[nr])

    S10 = np.var(V10 / n0, ddof=1)
    S01 = np.var(V01 / n1, ddof=1)
    return (S10 / n1) + (S01 / n0)

def auc_ci_delong(y_true, y_pred, alpha=0.05):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(float)

    auc = roc_auc_score(y_true, y_pred)
    var = delong_roc_variance(y_true, y_pred)
    if var is None or var <= 0:
        return auc, np.nan, np.nan

    se = np.sqrt(var)
    z = stats.norm.ppf(1 - alpha / 2)
    lo = max(0.0, auc - z * se)
    hi = min(1.0, auc + z * se)
    return auc, lo, hi

# -------------------------
# MLRA fixed formula
# -------------------------
MLRA_FEATURES = [
    "Total protein (g/dL)",
    "Direct bilirubin (mg/dL)",
    "White blood cells (103/μL)",
    "C-reactive protein (mg/dL)",
]

def apply_mlra_formula(df):
    eps = 1e-10
    lp = (
        2.053
        - 0.8872 * df["Total protein (g/dL)"]
        + 0.7692 * np.log(df["Direct bilirubin (mg/dL)"] + eps)
        + 0.5063 * np.log(df["White blood cells (103/μL)"] + eps)
        + 0.3229 * np.log(df["C-reactive protein (mg/dL)"] + eps)
    )
    return 1.0 / (1.0 + np.exp(-lp))

def prepare_mlra_complete_case(df, dataset_name):
    needed = ["Outcome", "Age", "Sex"] + MLRA_FEATURES
    keep_cols = [c for c in needed if c in df.columns]
    out = df[keep_cols].copy()
    n0 = len(out)
    out = out.dropna(subset=[c for c in MLRA_FEATURES if c in out.columns], how="any")
    print(f"[MLRA {dataset_name}] complete-case: {n0} -> {len(out)} (removed {n0-len(out)})")
    return out

# -------------------------
# Numeric coercion
# -------------------------
def coerce_numeric_base(df):
    out = df.copy()
    if "Sex" in out.columns and out["Sex"].dtype == "object":
        out["Sex"] = out["Sex"].map({"M": 1, "Male": 1, "F": 0, "Female": 0}).fillna(out["Sex"])
    for c in out.columns:
        if c == "Outcome":
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

# -------------------------
# Age grouping (FIXED)
# -------------------------
AGE_BINS = [18, 65, 75, np.inf]
AGE_LABELS = ["<65", "65–74", "≥75"]

def add_age_group(df):
    out = df.copy()
    out["Age_group"] = pd.cut(out["Age"], bins=AGE_BINS, labels=AGE_LABELS, right=False)
    return out

# -------------------------
# Threshold helpers (analytic-sample medians)
# -------------------------
def median_on_index(df, idx, col):
    if col not in df.columns:
        return None
    x = pd.to_numeric(df.loc[idx, col], errors="coerce")
    if x.dropna().shape[0] == 0:
        return None
    return float(np.nanmedian(x.values))

def median_on_all(df, col):
    if col not in df.columns:
        return None
    x = pd.to_numeric(df[col], errors="coerce")
    if x.dropna().shape[0] == 0:
        return None
    return float(np.nanmedian(x.values))

def add_binary_groups_strict(df, thresholds):
    # High: > cutoff ; Low: <= cutoff
    out = df.copy()
    if "Albumin (g/dL)" in out.columns and thresholds.get("Albumin") is not None:
        out["Albumin_group"] = np.where(out["Albumin (g/dL)"] > thresholds["Albumin"], "High", "Low")
    if "C-reactive protein (mg/dL)" in out.columns and thresholds.get("CRP") is not None:
        out["CRP_group"] = np.where(out["C-reactive protein (mg/dL)"] > thresholds["CRP"], "High", "Low")
    if "Total CCI score" in out.columns and thresholds.get("CCI") is not None:
        out["CCI_group"] = np.where(out["Total CCI score"] > thresholds["CCI"], "High", "Low")
    return out

# -------------------------
# Subgroup rows
#   [FIX-4] Unified median labels for Albumin/CRP/CCI (SI headers).
# -------------------------
def subgroup_rows_model(dataset_kind, thresholds, model_name):
    rows = []
    # Sex
    rows.append(("Sex", "Female", lambda d: d["Sex"] == 0))
    rows.append(("Sex", "Male",   lambda d: d["Sex"] == 1))

    # Age (years)
    rows.append(("Age (years)", "<65",   lambda d: d["Age_group"].astype(str) == "<65"))
    rows.append(("Age (years)", "65–74", lambda d: d["Age_group"].astype(str) == "65–74"))
    rows.append(("Age (years)", "≥75",   lambda d: d["Age_group"].astype(str) == "≥75"))

    # Albumin — unified median label (SI header)
    rows.append(("Albumin (g/dL)", "Low (≤ median)",  lambda d: d.get("Albumin_group", pd.Series([np.nan]*len(d))) == "Low"))
    rows.append(("Albumin (g/dL)", "High (> median)", lambda d: d.get("Albumin_group", pd.Series([np.nan]*len(d))) == "High"))

    # CRP — unified median label (SI header); membership stays model-specific via CRP_group
    rows.append(("CRP (mg/dL)", "Low (≤ median)",  lambda d: d.get("CRP_group", pd.Series([np.nan]*len(d))) == "Low"))
    rows.append(("CRP (mg/dL)", "High (> median)", lambda d: d.get("CRP_group", pd.Series([np.nan]*len(d))) == "High"))

    # CCI — development analysis only (Validation reserves layout space in the figure)
    if dataset_kind == "Development":
        rows.append(("CCI", "Low (≤ median)",  lambda d: d.get("CCI_group", pd.Series([np.nan]*len(d))) == "Low"))
        rows.append(("CCI", "High (> median)", lambda d: d.get("CCI_group", pd.Series([np.nan]*len(d))) == "High"))

    return rows

def compute_subgroup_table(df_base, proba, dataset_label, model_label, dataset_kind, thresholds):
    proba = pd.Series(np.asarray(proba, dtype=float), index=df_base.index)
    out_rows = []
    rows = subgroup_rows_model(dataset_kind, thresholds, model_label)
    # Numeric cutoff of each dichotomized subgroup (development-derived median of this model)
    _CUT_KEY = {"Albumin (g/dL)": ("Albumin", 2), "CRP (mg/dL)": ("CRP", 2), "CCI": ("CCI", 1)}
    def _cutoff(subgroup, category):
        if subgroup not in _CUT_KEY or thresholds.get(_CUT_KEY[subgroup][0]) is None:
            return ""
        key, nd = _CUT_KEY[subgroup]
        thr = thresholds[key]
        return f"≤{thr:.{nd}f}" if category.startswith("Low") else f">{thr:.{nd}f}"

    for subgroup, category, mask_fn in rows:
        if subgroup == "CCI" and "CCI_group" not in df_base.columns:
            continue

        mask = mask_fn(df_base).fillna(False)
        n = int(mask.sum())
        events = int(df_base.loc[mask, "Outcome"].sum()) if n > 0 else 0

        if n < 10 or df_base.loc[mask, "Outcome"].nunique() < 2:
            out_rows.append({"Model": model_label, "Dataset": dataset_label, "Subgroup": subgroup,
                             "Category": category, "Cutoff": _cutoff(subgroup, category), "N": n, "Events": events,
                             "AUC": np.nan, "CI_Lower": np.nan, "CI_Upper": np.nan})
            continue

        auc, lo, hi = auc_ci_delong(df_base.loc[mask, "Outcome"], proba.loc[mask])
        out_rows.append({"Model": model_label, "Dataset": dataset_label, "Subgroup": subgroup,
                         "Category": category, "Cutoff": _cutoff(subgroup, category), "N": n, "Events": events,
                         "AUC": auc, "CI_Lower": lo, "CI_Upper": hi})

    return pd.DataFrame(out_rows)

# -------------------------
# Forest plot
# -------------------------
def forest_plot_dev_val(df_dev, df_val, out_png):
    models = ["MLRA", "XGBoost Standard"]
    color = {"MLRA": COL_MLRA, "XGBoost Standard": COL_XGB}
    marker = {"MLRA": "s", "XGBoost Standard": "D"}
    offsets = {"MLRA": -0.14, "XGBoost Standard": +0.14}

    # [FIX-2] spacing knobs
    HEADER_H = 0.80
    ROW_H    = 0.52
    BLOCK_GAP = 0.60

    def canonical_subgroup_order():
        # [FIX-3] Always include CCI in the layout to keep panels aligned
        return ["Sex", "Age (years)", "Albumin (g/dL)", "CRP (mg/dL)", "CCI"]

    def canonical_categories(sg):
        if sg == "Sex":
            return ["Female", "Male"]
        if sg == "Age (years)":
            return ["<65", "65–74", "≥75"]
        if sg in ["Albumin (g/dL)", "CRP (mg/dL)", "CCI"]:
            return ["Low", "High"]
        return []

    def extract_ordered_pairs(df_panel, dataset_kind):
        subgroup_order = canonical_subgroup_order()
        ordered_pairs = []
        for sg in subgroup_order:
            df_sg = df_panel[df_panel["Subgroup"] == sg].copy()
            if df_sg.empty:
                continue
            cats = list(df_sg["Category"].dropna().unique())
            # [FIX-1] drop rows where category duplicates the subgroup header
            cats = [c for c in cats if str(c).strip() != sg]
            if sg == "Age (years)":
                def pick(lbl):
                    return [c for c in cats if str(c).strip().startswith(lbl)]
                cats = pick("<65") + pick("65–74") + pick("≥75") + \
                       [c for c in cats if c not in (pick("<65") + pick("65–74") + pick("≥75"))]
            else:
                low = [c for c in cats if str(c).strip().startswith("Low")]
                high = [c for c in cats if str(c).strip().startswith("High")]
                other = [c for c in cats if c not in low + high]
                cats = low + high + sorted(other)
            for c in cats:
                ordered_pairs.append((sg, c))
        return ordered_pairs, subgroup_order

    def panel(ax, df_panel, dataset_kind, title, panel_letter):
        ordered_pairs, subgroup_order = extract_ordered_pairs(df_panel, dataset_kind)

        y_positions, yticklabels = [], []
        header_y_set, cat_y_map = set(), {}

        cur = 0.0
        for sg in subgroup_order:
            pairs = [p for p in ordered_pairs if p[0] == sg]
            if not pairs:
                # Subgroup not analysed in this cohort (CCI in validation): keep the rows blank
                # so that both panels share the same y positions
                cur += HEADER_H + ROW_H * len(canonical_categories(sg)) + BLOCK_GAP
                continue
            y_positions.append(cur); yticklabels.append(sg)
            header_y_set.add(cur); cur += HEADER_H
            for (_, c) in pairs:
                y_positions.append(cur); yticklabels.append(f"  {c}")
                cat_y_map[(sg, c)] = cur; cur += ROW_H
            cur += BLOCK_GAP

        for m in models:
            df_m = df_panel[df_panel["Model"] == m].copy()
            for (sg, c) in ordered_pairs:
                r = df_m[(df_m["Subgroup"] == sg) & (df_m["Category"] == c)]
                if r.empty:
                    continue
                r = r.iloc[0]
                base_y = cat_y_map.get((sg, c))
                if base_y is None:
                    continue
                yy = base_y + offsets[m]
                if pd.notna(r["CI_Lower"]) and pd.notna(r["CI_Upper"]):
                    ax.plot([r["CI_Lower"], r["CI_Upper"]], [yy, yy],
                            color=color[m], alpha=ALPHA_CI, linewidth=1.4, solid_capstyle="round")
                if pd.notna(r["AUC"]):
                    ax.plot(r["AUC"], yy, marker=marker[m], markersize=5,
                            color=color[m], alpha=ALPHA_POINT, linestyle="None", label=m)

        ax.set_yticks(y_positions)
        ax.set_yticklabels(yticklabels, fontsize=8)
        for tick, yy in zip(ax.get_yticklabels(), y_positions):
            if yy in header_y_set:
                tick.set_fontweight("bold")

        # Identical y range in both panels (top row at the top); replaces invert_yaxis()
        ax.set_ylim(cur - BLOCK_GAP + 0.45, -0.45)
        ax.set_xlim(0.50, 1.02)
        ax.grid(axis="x", linestyle=":", alpha=0.35)
        ax.tick_params(axis="x", labelsize=8)
        ax.set_xlabel("AUC (95% CI)", fontsize=9)
        ax.set_title(title, fontsize=9, pad=6)                     # cohort label, not a figure title
        ax.text(-0.02, 1.01, panel_letter, transform=ax.transAxes, fontsize=10,
                fontweight="bold", ha="right", va="bottom")

        handles, labs = ax.get_legend_handles_labels()
        by_lab = dict(zip(labs, handles))
        ax.legend(by_lab.values(), by_lab.keys(), loc="lower left",
                  frameon=True, framealpha=0.95, fontsize=8, borderpad=0.6)
        return cur

    # PLOS ONE: width 7.5 in, height <= 8.75 in, text 8-10 pt, no figure title in the image
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.5, 8.0), sharex=True)
    end1 = panel(ax1, df_dev, "Development", "Development cohort\n(2019)", "A")
    end2 = panel(ax2, df_val, "Validation",  "Validation cohort\n(2017–2018)", "B")
    assert np.isclose(end1, end2), "Panels A and B have different y layouts — HALT"
    plt.tight_layout(w_pad=1.5)
    plt.savefig(out_png, dpi=300, format="jpeg", pil_kwargs={"quality": 95})   # no bbox='tight': keeps 7.5 in
    plt.show()
    plt.close()
    print(f"Saved: {out_png}")

# ============================================================
# MAIN
# ============================================================
print("Upload DEVELOPMENT (Training) CSV ...")
up1 = files.upload()
train_raw = pd.read_csv(list(up1.keys())[0])

print("\nUpload VALIDATION CSV ...")
up2 = files.upload()
val_raw = pd.read_csv(list(up2.keys())[0])

train_raw = coerce_numeric_base(train_raw)
val_raw   = coerce_numeric_base(val_raw)
train_raw["Outcome"] = pd.to_numeric(train_raw["Outcome"], errors="coerce").astype(int)
val_raw["Outcome"]   = pd.to_numeric(val_raw["Outcome"], errors="coerce").astype(int)

# Data shape + events confirmation (Coding protocol Tier 1 §6)
print(f"\nX_train: {train_raw.shape}, events: {int(train_raw['Outcome'].sum())}")
print(f"X_val:   {val_raw.shape}, events: {int(val_raw['Outcome'].sum())}")

train_raw = add_age_group(train_raw)
val_raw   = add_age_group(val_raw)

# MLRA complete-case frames
train_mlra = prepare_mlra_complete_case(train_raw, "Development")
val_mlra   = prepare_mlra_complete_case(val_raw,   "Validation")

# Cutoffs (analytic-sample medians): MLRA complete-case index vs XGB full cohort
thresholds_mlra = {
    "Albumin": median_on_index(train_raw, train_mlra.index, "Albumin (g/dL)"),
    "CRP":     median_on_index(train_raw, train_mlra.index, "C-reactive protein (mg/dL)"),
    "CCI":     median_on_index(train_raw, train_mlra.index, "Total CCI score"),
}
thresholds_xgb = {
    "Albumin": median_on_all(train_raw, "Albumin (g/dL)"),
    "CRP":     median_on_all(train_raw, "C-reactive protein (mg/dL)"),
    "CCI":     median_on_all(train_raw, "Total CCI score"),
}
print("\n[Cutoffs (development-derived)]")
print(f"  Albumin: MLRA={thresholds_mlra['Albumin']} | XGB={thresholds_xgb['Albumin']}")
print(f"  CRP:     MLRA={thresholds_mlra['CRP']} | XGB={thresholds_xgb['CRP']}")
print(f"  CCI:     MLRA={thresholds_mlra['CCI']} | XGB={thresholds_xgb['CCI']}")

# Apply group columns
train_raw_mlra_groups = add_binary_groups_strict(train_raw, thresholds_mlra)
val_raw_mlra_groups   = add_binary_groups_strict(val_raw,   thresholds_mlra)
train_raw_xgb_groups  = add_binary_groups_strict(train_raw, thresholds_xgb)
val_raw_xgb_groups    = add_binary_groups_strict(val_raw,   thresholds_xgb)

# Attach MLRA groups to MLRA population frames
for col in ["Age_group", "Albumin_group", "CRP_group", "CCI_group"]:
    if col in train_raw_mlra_groups.columns and col not in train_mlra.columns:
        train_mlra[col] = train_raw_mlra_groups.loc[train_mlra.index, col]
    if col in val_raw_mlra_groups.columns and col not in val_mlra.columns:
        val_mlra[col] = val_raw_mlra_groups.loc[val_mlra.index, col]

# MLRA probabilities
train_mlra_proba = apply_mlra_formula(train_mlra)
val_mlra_proba   = apply_mlra_formula(val_mlra)


# ── XGBoost Standard: use the FINAL model's predictions (no refitting) ──────
# [REPLACED 2026-10-24] The previous code refitted XGBoost on the subgroup CSV,
#   whose numeric columns include 'Total CCI score' (44 features); the final
#   model uses 43 candidate predictors without CCI. The subgroup CSV is now used
#   only to define subgroup membership (sex, age, albumin, CRP, CCI).
# Row alignment between the subgroup CSV and the main dataset is verified first.
for _name, _sub, _main in [('Development', train_raw, train_df),
                           ('Validation',  val_raw,   val_df)]:
    assert len(_sub) == len(_main), f"{_name}: row count differs — HALT"
    for _c in ['Outcome', 'Age', 'Uric acid (mg/dL)']:
        _a = pd.to_numeric(_sub[_c], errors='coerce').to_numpy(dtype=float)
        _b = pd.to_numeric(_main[_c], errors='coerce').to_numpy(dtype=float)
        assert np.allclose(_a, _b, equal_nan=True), f"{_name}: '{_c}' not aligned — HALT"
print("✓ Subgroup CSVs are row-aligned with the main dataset")

train_xgb_proba = np.asarray(ml_results['XGBoost_Standard']['train_proba'], dtype=float)
val_xgb_proba   = np.asarray(ml_results['XGBoost_Standard']['val_proba'],   dtype=float)
assert len(train_xgb_proba) == len(train_raw) and len(val_xgb_proba) == len(val_raw)
print(f"XGBoost Standard (final model) AUC — development: "
      f"{roc_auc_score(train_raw['Outcome'], train_xgb_proba):.4f}, "
      f"validation: {roc_auc_score(val_raw['Outcome'], val_xgb_proba):.4f}  "
      )

# XGB subgroup frames (full population; xgb-based cutoffs)
train_xgb_df = train_raw_xgb_groups.copy()
val_xgb_df   = val_raw_xgb_groups.copy()

# Subgroup tables
df_mlra_dev = compute_subgroup_table(train_mlra, train_mlra_proba, "Development", "MLRA", "Development", thresholds_mlra)
df_mlra_val = compute_subgroup_table(val_mlra,   val_mlra_proba,   "Validation",  "MLRA", "Validation",  thresholds_mlra)
df_xgb_dev  = compute_subgroup_table(train_xgb_df, train_xgb_proba, "Development", "XGBoost Standard", "Development", thresholds_xgb)
df_xgb_val  = compute_subgroup_table(val_xgb_df,   val_xgb_proba,   "Validation",  "XGBoost Standard", "Validation",  thresholds_xgb)

df_dev = pd.concat([df_mlra_dev, df_xgb_dev], ignore_index=True)
df_val = pd.concat([df_mlra_val, df_xgb_val], ignore_index=True)

df_mlra_all = pd.concat([df_mlra_dev, df_mlra_val], ignore_index=True)
df_xgb_all  = pd.concat([df_xgb_dev,  df_xgb_val],  ignore_index=True)

from IPython.display import display as _disp
print("\n[Subgroup Analysis Results — MLRA]"); _disp(df_mlra_all)
print("\n[Subgroup Analysis Results — XGBoost Standard]"); _disp(df_xgb_all)

# ── S17 Table (publication format): numeric cutoffs shown in the category labels ──
def _fmt_auc(r):
    return 'N/A' if pd.isna(r['AUC']) else f"{r['AUC']:.3f} ({r['CI_Lower']:.3f}–{r['CI_Upper']:.3f})"
def _label(r):
    return f"{r['Category'].split(' (')[0]} ({r['Cutoff']})" if r['Cutoff'] else r['Category']
s17_sheets = {}
for _ds, _dfm, _dfx in [('Development', df_mlra_dev, df_xgb_dev), ('Validation', df_mlra_val, df_xgb_val)]:
    assert list(_dfm['Subgroup']) == list(_dfx['Subgroup']) and list(_dfm['Category']) == list(_dfx['Category']), \
        f"{_ds}: subgroup rows differ between models — HALT"
    s17_sheets[_ds] = pd.DataFrame({
        'Subgroup'                     : _dfm['Subgroup'],
        'MLRA category'                : _dfm.apply(_label, axis=1),
        'MLRA N'                       : _dfm['N'],
        'MLRA events'                  : _dfm['Events'],
        'MLRA AUC (95% CI)'            : _dfm.apply(_fmt_auc, axis=1),
        'XGBoost Standard category'    : _dfx.apply(_label, axis=1),
        'XGBoost Standard N'           : _dfx['N'],
        'XGBoost Standard events'      : _dfx['Events'],
        'XGBoost Standard AUC (95% CI)': _dfx.apply(_fmt_auc, axis=1),
    })
    print(f"\n[S17 Table — {_ds} cohort]"); _disp(s17_sheets[_ds])
print("\nCutoffs (development-derived medians): "
      f"MLRA {thresholds_mlra} | XGBoost Standard {thresholds_xgb}")
S17_XLSX = 'S17_Table_Subgroup_Analysis.xlsx'
with pd.ExcelWriter(S17_XLSX, engine='openpyxl') as _w:
    for _ds, _t in s17_sheets.items():
        _t.to_excel(_w, sheet_name=_ds, index=False)

df_mlra_all.to_csv("S17_Table_Subgroup_MLRA.csv", index=False, encoding="utf-8-sig")
df_xgb_all.to_csv("S17_Table_Subgroup_XGBoost_Standard.csv", index=False, encoding="utf-8-sig")

# Forest plot -> S6 Fig
forest_plot_dev_val(df_dev, df_val, out_png="S6_Fig_Subgroup_Forest_Plot.jpg")

files.download("S17_Table_Subgroup_MLRA.csv")
files.download("S17_Table_Subgroup_XGBoost_Standard.csv")
files.download("S6_Fig_Subgroup_Forest_Plot.jpg")
files.download(S17_XLSX)

print("\nDone.")

## Section 18 · Missing Data Analysis
**Prerequisite**: Section 3 (`train_df`, `val_df` in memory).

| Output | Cell | Content |
|---|---|---|
| S3 Fig + S3 Table | 18-A, 18-B | Variable-level missingness (variables with ≥50% missing values in the development cohort) and the distribution of missing variables per patient; S3 Table Part A (top 20 variables) and Part B (patient level) |
| S7 Fig | Part 2 | Apparent AUCs of a logistic regression including all 42 numeric candidate predictors after mean, median, or single chained-equation imputation (IterativeImputer, one imputed data set), development cohort |

Little's MCAR test could not be performed because no patient had complete data across all candidate predictors.

Outputs: `S3_Fig_Missing_Data_Patterns.jpg`, `S3_Table_Missing_Data_Patterns.xlsx`, `S7_Fig_Imputation_Comparison.jpg/.pdf`, and supporting CSV/TXT files.

In [ ]:
# ============================================================
# Section 18-A: Missing Data Pattern Analysis — Publication Outputs
# Source: OK_41_Section18_Missing_Data_Pattern_Analysis_by_Claude_v2 Cell 4
# Updated: 2026-03-07
# Uses: train_df / val_df from Section 3 (no re-upload required)
# Outputs: S3_Fig_Missing_Data_Patterns.jpg (600 DPI; S3 Fig)
#          S3_Table_Missing_Data_Patterns.xlsx (S3 Table, Part A + Part B)
# ============================================================
# ============================================================
# Cell 4: Missing Data Pattern Analysis
# Source: 41_Missing_Analysis Cell 1 (Phase 1)
# Uses: train_df, val_df (Cell 3 above, or Section 3 of OK_11_Main)
# ============================================================
# [FIXED] What was wrong : pd, np, plt, openpyxl were not imported in the
#                          standalone version of this section.
# [FIXED] What was changed: Added dedicated install/import/data-load cells
#                            (Cells 1–3) mirroring the OK_51 notebook pattern.

print("=" * 70)
print("Section 18: Missing Data Pattern Analysis")
print("=" * 70)

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ============================================================================
# Abbreviation mapping: CSV column name → publication-ready display label
# Keys are lowercase substrings; first match wins.
# ============================================================================
_ABBREV_RULES = [
    ("glycated albumin",         "GA (%)"),
    ("antithrombin",             "AT-3 (%)"),
    ("ferritin",                 "Ferritin (ng/mL)"),
    ("fibrin/fibrinogen",        "FDP (μg/mL)"),
    ("fdp",                      "FDP (μg/mL)"),
    ("serum fe",                 "Serum Fe"),
    ("serum iron",               "Serum Fe"),
    (" iron",                    "Serum Fe"),
    ("amylase",                  "Amylase (U/L)"),
    ("fib-d",                    "FIB-D (mg/dL)"),
    ("fibrinogen",               "FIB-D (mg/dL)"),
    ("ldl",                      "LDL-C (mg/dL)"),
    ("d-dimer",                  "D-dimer (μg/mL)"),
    ("hba1c",                    "HbA1c (%)"),
    ("glycohemoglobin",          "HbA1c (%)"),
    ("hdl",                      "HDL-C (mg/dL)"),
    ("total cholesterol",        "T-CHO (mg/dL)"),
    ("t-cho",                    "T-CHO (mg/dL)"),
    ("triglyceride",             "TG (mg/dL)"),
    (" tg ",                     "TG (mg/dL)"),
    ("aptt",                     "APTT (sec)"),
    ("activated partial thromb", "APTT (sec)"),
    ("pt-inr",                   "PT-INR"),
    ("prothrombin time",         "PT-INR"),
    ("magnesium",                "Magnesium (mg/dL)"),
    ("random plasma glucose",    "RPG (mg/dL)"),
    ("rpg",                      "RPG (mg/dL)"),
    ("calcium",                  "Calcium (mg/dL)"),
    ("phosphorus",               "Phosphorus (mg/dL)"),
    ("creatine kinase",          "Creatine kinase (U/L)"),
]

def _abbrev(col_name: str) -> str:
    """Return publication-ready abbreviation for a column name."""
    col_lower = col_name.lower()
    for pattern, label in _ABBREV_RULES:
        if pattern.lower() in col_lower:
            return label
    return col_name if len(col_name) <= 28 else col_name[:26] + "…"


# ============================================================================
# 1. Per-variable missingness
# ============================================================================
def _var_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Return per-variable missingness DataFrame, sorted descending."""
    miss_n   = df.isnull().sum()
    miss_pct = (miss_n / len(df) * 100).round(2)
    return (pd.DataFrame({
                "Variable":     df.columns,
                "Missing_N":    miss_n.values,
                "Missing_Pct":  miss_pct.values,
                "Complete_N":   df.notnull().sum().values,
                "Complete_Pct": (df.notnull().sum() / len(df) * 100).round(2).values,
            })
            .query("Missing_N > 0")
            .sort_values("Missing_Pct", ascending=False)
            .reset_index(drop=True))

train_var_miss = _var_missing(train_df)
val_var_miss   = _var_missing(val_df)

print(f"\n[1/3] Variable-level missingness")
print(f"  Development : {len(train_var_miss)} variables with missing data")
print(f"  Validation  : {len(val_var_miss)} variables with missing data")


# ============================================================================
# 2. Patient-level missingness
# ============================================================================
train_pt_miss = train_df.isnull().sum(axis=1)
val_pt_miss   = val_df.isnull().sum(axis=1)

def _patient_stats(df: pd.DataFrame, pt_miss: pd.Series) -> dict:
    total         = len(df)
    complete      = int((pt_miss == 0).sum())
    incomplete    = total - complete
    n_vars        = df.shape[1]
    total_cells   = total * n_vars
    missing_cells = int(df.isnull().sum().sum())
    unique_patt   = int(df.isnull().astype(int).drop_duplicates().shape[0])
    return {
        "Total patients":             total,
        "Complete cases (n, %)":      f"{complete} ({complete/total*100:.1f}%)",
        "Incomplete cases (n, %)":    f"{incomplete} ({incomplete/total*100:.1f}%)",
        "Mean missing per patient":   f"{pt_miss.mean():.2f}",
        "Median missing per patient": f"{int(pt_miss.median())}",
        "Max missing per patient":    int(pt_miss.max()),
        "Unique missing patterns":    unique_patt,
        "Overall missing rate (%)":   f"{missing_cells/total_cells*100:.2f}%",
    }

train_pt_stats = _patient_stats(train_df, train_pt_miss)
val_pt_stats   = _patient_stats(val_df,   val_pt_miss)

print(f"\n[2/3] Patient-level statistics")
print(f"  Development — complete cases : {train_pt_stats['Complete cases (n, %)']}")
print(f"  Development — mean missing   : {train_pt_stats['Mean missing per patient']}")
print(f"  Validation  — complete cases : {val_pt_stats['Complete cases (n, %)']}")
print(f"  Validation  — mean missing   : {val_pt_stats['Mean missing per patient']}")



# ============================================================================
# 3. S3 Fig — 4-panel publication figure (600 DPI)
# ============================================================================
print("\n[3/3] Generating S3 Fig ...")

# Variables shown in Panels A/B: ≥50% missing in development cohort
MISS_THRESHOLD_PCT = 50.0
panel_ab_vars = (train_var_miss[train_var_miss["Missing_Pct"] >= MISS_THRESHOLD_PCT]
                 .copy()
                 .reset_index(drop=True))
panel_ab_vars["Label"]   = panel_ab_vars["Variable"].apply(_abbrev)
val_miss_dict            = dict(zip(val_var_miss["Variable"],
                                    val_var_miss["Missing_Pct"]))
panel_ab_vars["Val_Pct"] = panel_ab_vars["Variable"].map(val_miss_dict).fillna(0.0)

# ── Color palette (matches reference figure) ────────────────────────────────
COLOR_DEV_BAR  = "#E07070"   # salmon-red  (Panel A)
COLOR_VAL_BAR  = "#5599CC"   # steel-blue  (Panel B)
COLOR_DEV_HIST = "#5CBB8A"   # green       (Panel C)
COLOR_VAL_HIST = "#F5A742"   # orange      (Panel D)
COLOR_MEAN     = "#CC2222"   # red dashed  (mean)
COLOR_MEDIAN   = "#2244CC"   # blue dashed (median)

fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
(ax_a, ax_b), (ax_c, ax_d) = axes

def _barh_panel(ax, labels, values, color, title):
    """Horizontal bar chart for variable-level missingness."""
    n     = len(labels)
    y_pos = np.arange(n)
    bars  = ax.barh(y_pos, values, color=color, edgecolor="none", height=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 105)
    ax.set_xlabel("Missing Data (%)", fontsize=9.5, fontweight="bold")
    ax.set_title(title, fontsize=10.5, fontweight="bold", pad=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    for bar, v in zip(bars, values):
        ax.text(v + 0.8, bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}%", va="center", ha="left",
                fontsize=7.5, color="#333333")

_barh_panel(ax_a,
            panel_ab_vars["Label"].tolist(),
            panel_ab_vars["Missing_Pct"].tolist(),
            COLOR_DEV_BAR,
            "Panel A: Variable-Level Missingness (Development Cohort)")

_barh_panel(ax_b,
            panel_ab_vars["Label"].tolist(),
            panel_ab_vars["Val_Pct"].tolist(),
            COLOR_VAL_BAR,
            "Panel B: Variable-Level Missingness (Validation Cohort)")

def _hist_panel(ax, pt_miss, color, title):
    """Histogram of per-patient missing counts with mean/median lines."""
    ax.hist(pt_miss, bins=range(0, int(pt_miss.max()) + 2),
            color=color, edgecolor="white", linewidth=0.4, alpha=0.92)
    mean_v   = pt_miss.mean()
    median_v = pt_miss.median()
    ax.axvline(mean_v,   color=COLOR_MEAN,   linestyle="--", linewidth=1.8,
               label=f"Mean: {mean_v:.1f}")
    ax.axvline(median_v, color=COLOR_MEDIAN, linestyle="--", linewidth=1.8,
               label=f"Median: {int(median_v)}")
    ax.set_xlabel("Number of Missing Variables per Patient",
                  fontsize=9.5, fontweight="bold")
    ax.set_ylabel("Frequency", fontsize=9.5, fontweight="bold")
    ax.set_title(title, fontsize=10.5, fontweight="bold", pad=8)
    ax.legend(fontsize=8.5, frameon=True, framealpha=0.8,
              edgecolor="none", loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25, linewidth=0.5)

_hist_panel(ax_c, train_pt_miss, COLOR_DEV_HIST,
            "Panel C: Patient-Level Missing Patterns (Development)")
_hist_panel(ax_d, val_pt_miss,   COLOR_VAL_HIST,
            "Panel D: Patient-Level Missing Patterns (Validation)")

# Figure title removed (PLOS ONE: the legend carries the title)
plt.tight_layout(rect=[0, 0, 1, 0.988])

FIG_S4_PATH = "S3_Fig_Missing_Data_Patterns.jpg"
plt.savefig(FIG_S4_PATH, dpi=600, bbox_inches="tight", facecolor="white",
            format='jpeg', pil_kwargs={'quality': 95})
plt.show()
print(f"  Saved: {FIG_S4_PATH}  (600 DPI, JPEG quality=95)")

# ── Verification ────────────────────────────────────────────────────────────
# Panel A top bar: GA (%) ~97.1%, second: AT-3 ~97.1%
# Panel A total bars: 15 (variables ≥50% missing in dev cohort)
# Panel C mean: ~13.2, median: 13 | Panel D mean: ~13.2, median: 13
print(f"  [Verify] Panel A top variable : {panel_ab_vars.iloc[0]['Label']!r}"
      f" = {panel_ab_vars.iloc[0]['Missing_Pct']:.1f}%  | expected GA ~97.1%")
print(f"  [Verify] Panel A n_bars       : {len(panel_ab_vars)}"
      f"  | expected 15")
print(f"  [Verify] Panel C mean         : {train_pt_miss.mean():.2f}"
      f"  | expected ~13.21")
print(f"  [Verify] Panel D mean         : {val_pt_miss.mean():.2f}"
      f"  | expected ~13.18")


# ============================================================================
# 4. S3 Table — Excel workbook (Part A + Part B)
# ============================================================================
TOP_N = 20
top20 = train_var_miss.head(TOP_N).copy()
top20["Label"]   = top20["Variable"].apply(_abbrev)
val_n_dict       = dict(zip(val_var_miss["Variable"], val_var_miss["Missing_N"]))
top20["Val_N"]   = top20["Variable"].map(val_n_dict).fillna(0).astype(int)
top20["Val_Pct"] = top20["Variable"].map(val_miss_dict).fillna(0.0)

part_a_df = pd.DataFrame({
    "Variable":              top20["Label"],
    "Development Missing N": top20["Missing_N"],
    "Development Missing %": top20["Missing_Pct"],
    "Validation Missing N":  top20["Val_N"],
    "Validation Missing %":  top20["Val_Pct"],
})

metrics_order = [
    "Total patients",
    "Complete cases (n, %)",
    "Incomplete cases (n, %)",
    "Mean missing per patient",
    "Median missing per patient",
    "Max missing per patient",
    "Unique missing patterns",
    "Overall missing rate (%)",
]
part_b_df = pd.DataFrame({
    "Metric":             metrics_order,
    "Development Cohort": [train_pt_stats[m] for m in metrics_order],
    "Validation Cohort":  [val_pt_stats[m]   for m in metrics_order],
})

# ── Excel styling helpers ─────────────────────────────────────────────────
HEADER_FILL  = PatternFill("solid", fgColor="2C5F8A")
HEADER_FONT  = Font(bold=True, color="FFFFFF", size=10)
SUBHDR_FILL  = PatternFill("solid", fgColor="D0E4F7")
SUBHDR_FONT  = Font(bold=True, size=10)
BODY_FONT    = Font(size=9)
ALT_FILL     = PatternFill("solid", fgColor="F2F8FF")
WHITE_FILL   = PatternFill("solid", fgColor="FFFFFF")
THIN_BORDER  = Border(
    left=Side(style="thin", color="BBBBBB"),
    right=Side(style="thin", color="BBBBBB"),
    top=Side(style="thin", color="BBBBBB"),
    bottom=Side(style="thin", color="BBBBBB"),
)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center", wrap_text=True)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "S3 Table"

row = 1

# Title
ws.merge_cells(f"A{row}:E{row}")
c = ws[f"A{row}"]
c.value = ("S3 Table. Missing Data Patterns and Handling "
           "Strategy in the Development and Validation Cohorts")
c.font      = Font(bold=True, size=12)
c.alignment = LEFT
ws.row_dimensions[row].height = 30
row += 2

# Part A subheader
ws.merge_cells(f"A{row}:E{row}")
c = ws[f"A{row}"]
c.value = "Part A: Variable-Level Missingness Rates (Top 20)"
c.font  = Font(bold=True, size=11)
c.fill  = SUBHDR_FILL
ws.row_dimensions[row].height = 18
row += 1

headers_a    = ["Variable",
                "Development\nMissing N",
                "Development\nMissing %",
                "Validation\nMissing N",
                "Validation\nMissing %"]
col_widths_a = [24, 14, 14, 14, 14]
for c_idx, (hdr, wid) in enumerate(zip(headers_a, col_widths_a), start=1):
    cell = ws.cell(row=row, column=c_idx, value=hdr)
    cell.font = HEADER_FONT; cell.fill = HEADER_FILL
    cell.alignment = CENTER; cell.border = THIN_BORDER
    ws.column_dimensions[get_column_letter(c_idx)].width = wid
ws.row_dimensions[row].height = 30
row += 1

for r_idx, (_, dr) in enumerate(part_a_df.iterrows()):
    fill = ALT_FILL if r_idx % 2 == 1 else WHITE_FILL
    vals = [dr["Variable"],
            int(dr["Development Missing N"]),
            dr["Development Missing %"],
            int(dr["Validation Missing N"]),
            dr["Validation Missing %"]]
    for c_idx, val in enumerate(vals, start=1):
        cell = ws.cell(row=row, column=c_idx, value=val)
        cell.font      = BODY_FONT
        cell.fill      = fill
        cell.alignment = LEFT if c_idx == 1 else CENTER
        cell.border    = THIN_BORDER
        if c_idx in (3, 5) and isinstance(val, float):
            cell.number_format = "0.00"
    row += 1

row += 1  # blank row

# Part B subheader
ws.merge_cells(f"A{row}:C{row}")
c = ws[f"A{row}"]
c.value = "Part B: Patient-Level Patterns"
c.font  = Font(bold=True, size=11)
c.fill  = SUBHDR_FILL
ws.row_dimensions[row].height = 18
row += 1

headers_b  = ["Metric", "Development Cohort", "Validation Cohort"]
col_widths_b = [34, 22, 22]
for c_idx, (hdr, wid) in enumerate(zip(headers_b, col_widths_b), start=1):
    cell = ws.cell(row=row, column=c_idx, value=hdr)
    cell.font = HEADER_FONT; cell.fill = HEADER_FILL
    cell.alignment = CENTER; cell.border = THIN_BORDER
    ws.column_dimensions[get_column_letter(c_idx)].width = wid
ws.row_dimensions[row].height = 28
row += 1

for r_idx, (_, dr) in enumerate(part_b_df.iterrows()):
    fill = ALT_FILL if r_idx % 2 == 1 else WHITE_FILL
    for c_idx, val in enumerate(
            [dr["Metric"], dr["Development Cohort"], dr["Validation Cohort"]],
            start=1):
        cell = ws.cell(row=row, column=c_idx, value=val)
        cell.font      = BODY_FONT
        cell.fill      = fill
        cell.alignment = LEFT if c_idx == 1 else CENTER
        cell.border    = THIN_BORDER
    row += 1

row += 1  # blank row

# Footnotes
footnotes = [
    ("In Part A, missingness is reported as the number and percentage of patients "
     "with missing values for each variable. Variables are ordered by missingness "
     "proportion in the development cohort."),
    ("In Part B, patient-level missingness metrics include the mean, median, and "
     "maximum number of missing variables per patient, as well as the number of "
     "unique missing data patterns. Overall missing rate represents the proportion "
     "of missing values across all predictors and patients."),
    ("All analyses were conducted using 43 candidate predictor variables, including "
     "age, sex, and 41 laboratory measurements. No imputation was performed for the "
     "primary multivariable logistic regression analysis, which was based on complete "
     "case analysis; tree-based machine learning models used native handling of missing "
     "values, as specified in the Methods."),
    ("Abbreviations: APTT, activated partial thromboplastin time; FDP, "
     "fibrin/fibrinogen degradation products; GA, glycated albumin; HDL-C, "
     "high-density lipoprotein cholesterol; LDL-C, low-density lipoprotein "
     "cholesterol; PT-INR, prothrombin time–international normalized ratio; "
     "RPG, random plasma glucose; TG, triglycerides."),
]
for fn in footnotes:
    ws.merge_cells(f"A{row}:E{row}")
    fn_cell = ws[f"A{row}"]
    fn_cell.value     = fn
    fn_cell.font      = Font(italic=True, size=8)
    fn_cell.alignment = Alignment(horizontal="left", vertical="top",
                                  wrap_text=True)
    ws.row_dimensions[row].height = 40
    row += 1

TBL_S8_PATH = "S3_Table_Missing_Data_Patterns.xlsx"
wb.save(TBL_S8_PATH)
print(f"\n  Saved: {TBL_S8_PATH}")

# ── Inline display of S3 Table content ──────────────────────────────────
# [FIXED] S3 Table was saved to xlsx but not displayed inline; added display()
from IPython.display import display as _disp48
print("\n[S3 Table — Part A: Variable-Level Missingness (Top 20 by % Missing)]")
_disp48(part_a_df.reset_index(drop=True))
print("\n[S3 Table — Part B: Patient-Level Missingness Summary]")
_disp48(part_b_df.reset_index(drop=True))

# ── Verification ────────────────────────────────────────────────────────────
# S3 Table Part A row 1: GA ~97.09% dev, ~98.07% val
# S3 Table Part B: dev total=756, val total=1243, both complete=0
print(f"  [Verify] S3 Table Part A row 1 : {part_a_df.iloc[0]['Variable']!r}"
      f"  dev {part_a_df.iloc[0]['Development Missing %']:.2f}%"
      f"  val {part_a_df.iloc[0]['Validation Missing %']:.2f}%"
      f"  | expected GA 97.09% / 98.07%")

print("\n✓ Section 18 complete — S3 Fig and S3 Table generated")


In [ ]:
# ============================================================
# Section 18-B: Download — S3 Fig + S3 Table
# Run AFTER Cell 18-A has completed.
# ============================================================
# ============================================================
# Cell 5: Download outputs
# Run AFTER Cell 4 has completed successfully.
# ============================================================
from google.colab import files as _colab_files

_colab_files.download("S3_Fig_Missing_Data_Patterns.jpg")

# ── Inline preview of S3 Fig (visual confirmation) ───────────────────
from IPython.display import Image as _Img49, display as _disp49
import os as _os49
if _os49.path.exists("S3_Fig_Missing_Data_Patterns.jpg"):
    print("\n[S3 Fig — Missing Data Patterns (inline preview)]")
    _disp49(_Img49("S3_Fig_Missing_Data_Patterns.jpg", width=900))
_colab_files.download("S3_Table_Missing_Data_Patterns.xlsx")
print("✓ Downloads initiated:")
print("  • S3_Fig_Missing_Data_Patterns.jpg  (600 DPI, JPEG quality=95)")
print("  • S3_Table_Missing_Data_Patterns.xlsx")


In [ ]:
# ============================================================
# Section 18 — Part 2: MCAR Test, Imputation Comparison & Model Performance
# Source: 41_Missing_Analysis Cell 1 (Phase 1) — restored from v6
# Prerequisite: Section 3 (train_df, val_df in memory)
#               Section 1 (imports: pandas, numpy, matplotlib, scipy, sklearn)
# Analyses:
#   [4/8] Little's MCAR test (chi-square approximation)
#   [5/8] Complete vs incomplete case comparison
#   [6/8] Imputation method comparison (CCA / mean / median / single chained-equation imputation)
#   [7/8] Model performance comparison across imputation methods
#   [8/8] Visualizations (Figure1_Missing_Data_Patterns + Figure2_Imputation)
# Outputs:
#   - Missing_Data_Summary_Training/Validation.csv
#   - Complete_vs_Incomplete_Training/Validation.csv
#   - Imputation_Method_Performance.csv
#   - Missing_Data_Analysis_Report.txt
#   - Figure1_Missing_Data_Patterns.png/.pdf
#   - Figure2_Imputation_Method_Comparison.png/.pdf
# [FIXED] What was wrong : old code was removed in v7 in error.
# [FIXED] What was changed: restored from v6 (Cell 48) as a separate cell
#                            following the S3 Fig / S3 Table cell.
# ============================================================
# [FIXED] What was wrong : chi2 (scipy.stats.chi2 distribution object) was used
#                          in littles_mcar_test() without being imported.
#                          stats.chi2_contingency (the test function) is already
#                          available via `from scipy import stats` in Section 2,
#                          but the distribution object chi2 requires explicit import.
# [FIXED] What was changed: Added `from scipy.stats import chi2` at cell top.
#
# [FIXED] What was wrong : IterativeImputer (used in compare_imputation_methods)
#                          is not included in Section 2 imports.
# [FIXED] What was changed: Added the experimental enable import and
#                           `from sklearn.impute import IterativeImputer`.
from scipy.stats import chi2
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer

print("=" * 70)
print("Section 18: Missing Data Strategy Analysis  [Part 2/2]")
print("=" * 70)

# Install missingno if not present
try:
    import missingno as msno
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "missingno"])
    import missingno as msno

# NOTE: train_df and val_df already loaded from Section 3
# ============================================================================
# SECTION 3: MISSING DATA PATTERN ANALYSIS
# ============================================================================
print("\n[3/8] Analyzing missing data patterns...")
print("=" * 80)

def analyze_missing_patterns(df, cohort_name):
    """Comprehensive missing data pattern analysis"""

    print(f"\n--- {cohort_name} Cohort ---")

    # Overall missing statistics
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    missing_pct = (missing_cells / total_cells) * 100

    print(f"\nOverall Missing Data:")
    print(f"  Total cells: {total_cells:,}")
    print(f"  Missing cells: {missing_cells:,} ({missing_pct:.2f}%)")

    # Per-variable missing counts
    missing_summary = pd.DataFrame({
        'Variable': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2),
        'Complete_Count': df.notnull().sum(),
        'Complete_Percent': (df.notnull().sum() / len(df) * 100).round(2)
    })

    missing_summary = missing_summary[missing_summary['Missing_Count'] > 0].sort_values(
        'Missing_Percent', ascending=False
    )

    if len(missing_summary) > 0:
        print(f"\nVariables with Missing Data (Top 10):")
        print(missing_summary.head(10).to_string(index=False))
    else:
        print("\n✓ No missing data found!")

    # Per-patient missing counts
    patient_missing = df.isnull().sum(axis=1)
    complete_cases = (patient_missing == 0).sum()
    incomplete_cases = len(df) - complete_cases

    print(f"\nPatient-Level Missing Data:")
    print(f"  Complete cases: {complete_cases} ({complete_cases/len(df)*100:.1f}%)")
    print(f"  Incomplete cases: {incomplete_cases} ({incomplete_cases/len(df)*100:.1f}%)")
    print(f"  Mean missing per patient: {patient_missing.mean():.2f}")
    print(f"  Median missing per patient: {patient_missing.median():.0f}")
    print(f"  Max missing per patient: {patient_missing.max()}")

    return missing_summary, patient_missing

# Analyze both cohorts
train_missing, train_patient_missing = analyze_missing_patterns(train_df, "Training")
val_missing, val_patient_missing = analyze_missing_patterns(val_df, "Validation")

# ============================================================================
# SECTION 4: LITTLE'S MCAR TEST
# ============================================================================
print("\n[4/8] Performing Little's MCAR Test...")
print("=" * 80)

def littles_mcar_test(data, alpha=0.05):
    """
    Perform Little's MCAR test
    H0: Data are Missing Completely At Random (MCAR)
    H1: Data are not MCAR (MAR or MNAR)
    """

    # Select only numeric columns
    numeric_data = data.select_dtypes(include=[np.number])

    # Remove columns with no missing data
    cols_with_missing = numeric_data.columns[numeric_data.isnull().any()].tolist()

    if len(cols_with_missing) == 0:
        return {
            'chi_square': 0,
            'df': 0,
            'p_value': 1.0,
            'conclusion': 'No missing data - MCAR test not applicable',
            'missing_vars': 0
        }

    data_subset = numeric_data[cols_with_missing].copy()

    # Get missing patterns
    patterns = data_subset.isnull().astype(int)
    unique_patterns = patterns.drop_duplicates()

    n_patterns = len(unique_patterns)
    n_vars = len(cols_with_missing)

    print(f"\nMCAR Test Statistics:")
    print(f"  Variables with missing: {n_vars}")
    print(f"  Unique missing patterns: {n_patterns}")

    # Simplified chi-square approximation
    chi_square = 0
    df = 0

    for idx, pattern_row in unique_patterns.iterrows():
        pattern_mask = (patterns == pattern_row).all(axis=1)
        n_obs = pattern_mask.sum()

        if n_obs > 1:
            for col in cols_with_missing:
                if not pattern_row[col]:  # If this variable is observed in this pattern
                    observed_mean = data_subset.loc[pattern_mask, col].mean()
                    overall_mean = data_subset[col].mean()

                    if not np.isnan(observed_mean) and not np.isnan(overall_mean):
                        chi_square += n_obs * ((observed_mean - overall_mean) ** 2) / (data_subset[col].std() ** 2 + 1e-10)
                        df += 1

    if df > 0:
        p_value = 1 - chi2.cdf(chi_square, df)   # chi2 now correctly imported
    else:
        p_value = 1.0

    conclusion = "MCAR" if p_value >= alpha else "Not MCAR (MAR or MNAR)"

    result = {
        'chi_square': chi_square,
        'df': df,
        'p_value': p_value,
        'conclusion': conclusion,
        'missing_vars': n_vars,
        'alpha': alpha
    }

    print(f"\nLittle's MCAR Test Results:")
    print(f"  Chi-square statistic: {chi_square:.2f}")
    print(f"  Degrees of freedom: {df}")
    print(f"  P-value: {p_value:.4f}")
    print(f"  Conclusion (α={alpha}): {conclusion}")

    return result

# Perform MCAR test on both cohorts
train_mcar = littles_mcar_test(train_df)
val_mcar = littles_mcar_test(val_df)

# ============================================================================
# SECTION 5: COMPLETE VS INCOMPLETE CASE COMPARISON
# ============================================================================
print("\n[5/8] Comparing complete vs incomplete cases...")
print("=" * 80)

def compare_complete_incomplete(df, cohort_name):
    """Compare characteristics between complete and incomplete cases"""

    # Identify complete cases
    complete_mask = ~df.isnull().any(axis=1)
    complete_cases = df[complete_mask].copy()
    incomplete_cases = df[~complete_mask].copy()

    print(f"\n--- {cohort_name} Cohort Comparison ---")
    print(f"Complete cases: {len(complete_cases)} ({len(complete_cases)/len(df)*100:.1f}%)")
    print(f"Incomplete cases: {len(incomplete_cases)} ({len(incomplete_cases)/len(df)*100:.1f}%)")

    # Compare outcome
    if 'Outcome' in df.columns:
        complete_outcome = complete_cases['Outcome'].mean()
        incomplete_outcome = incomplete_cases['Outcome'].mean()

        # Chi-square test
        contingency = pd.crosstab(complete_mask, df['Outcome'])
        chi2_stat, p_value = stats.chi2_contingency(contingency)[:2]

        print(f"\nOutcome Comparison:")
        print(f"  Complete cases mortality: {complete_outcome:.3f} ({complete_outcome*100:.1f}%)")
        print(f"  Incomplete cases mortality: {incomplete_outcome:.3f} ({incomplete_outcome*100:.1f}%)")
        print(f"  Chi-square test: χ²={chi2_stat:.2f}, p={p_value:.4f}")

    # Compare key numeric variables
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'Outcome' in numeric_cols:
        numeric_cols.remove('Outcome')

    comparison_results = []

    for col in numeric_cols[:10]:  # Top 10 variables
        complete_mean = complete_cases[col].mean()
        incomplete_mean = incomplete_cases[col].mean()

        if not np.isnan(complete_mean) and not np.isnan(incomplete_mean):
            # T-test
            t_stat, p_val = stats.ttest_ind(
                complete_cases[col].dropna(),
                incomplete_cases[col].dropna(),
                equal_var=False
            )

            comparison_results.append({
                'Variable': col,
                'Complete_Mean': complete_mean,
                'Incomplete_Mean': incomplete_mean,
                'Difference': incomplete_mean - complete_mean,
                'P_value': p_val,
                'Significant': 'Yes' if p_val < 0.05 else 'No'
            })

    if comparison_results:
        comparison_df = pd.DataFrame(comparison_results)
        print(f"\nKey Variable Comparisons (α=0.05):")
        print(comparison_df.to_string(index=False))

        sig_count = (comparison_df['Significant'] == 'Yes').sum()
        print(f"\nSignificant differences: {sig_count}/{len(comparison_df)} variables")

    return comparison_df if comparison_results else None

# Compare both cohorts
train_comparison = compare_complete_incomplete(train_df, "Training")
val_comparison = compare_complete_incomplete(val_df, "Validation")

# ============================================================================
# SECTION 6: METHOD-SPECIFIC HANDLING COMPARISON
# ============================================================================
print("\n[6/8] Comparing imputation methods...")
print("=" * 80)

def compare_imputation_methods(df):
    """Compare different imputation strategies"""

    # Prepare data
    X = df.drop('Outcome', axis=1) if 'Outcome' in df.columns else df
    y = df['Outcome'].values if 'Outcome' in df.columns else None

    # Convert to numeric
    X_numeric = X.select_dtypes(include=[np.number])

    results = {}

    # 1. Complete Case Analysis (listwise deletion)
    complete_mask = ~X_numeric.isnull().any(axis=1)
    X_complete = X_numeric[complete_mask].values
    y_complete = y[complete_mask] if y is not None else None

    results['Complete_Case'] = {
        'n_samples': len(X_complete),
        'n_features': X_complete.shape[1],
        'X': X_complete,
        'y': y_complete
    }

    print(f"\n1. Complete Case Analysis:")
    print(f"   Retained: {len(X_complete)}/{len(X_numeric)} samples ({len(X_complete)/len(X_numeric)*100:.1f}%)")

    # 2. Mean Imputation
    mean_imputer = SimpleImputer(strategy='mean')
    X_mean = mean_imputer.fit_transform(X_numeric)

    results['Mean_Imputation'] = {
        'n_samples': len(X_mean),
        'n_features': X_mean.shape[1],
        'X': X_mean,
        'y': y,
        'imputer': mean_imputer
    }

    print(f"\n2. Mean Imputation:")
    print(f"   Retained: {len(X_mean)}/{len(X_numeric)} samples (100.0%)")

    # 3. Median Imputation
    median_imputer = SimpleImputer(strategy='median')
    X_median = median_imputer.fit_transform(X_numeric)

    results['Median_Imputation'] = {
        'n_samples': len(X_median),
        'n_features': X_median.shape[1],
        'X': X_median,
        'y': y,
        'imputer': median_imputer
    }

    print(f"\n3. Median Imputation:")
    print(f"   Retained: {len(X_median)}/{len(X_numeric)} samples (100.0%)")

    # 4. Single chained-equation imputation (IterativeImputer, one imputed data set; key 'MICE' kept internally)
    print(f"\n4. Single chained-equation imputation (IterativeImputer):")
    print(f"   Training imputer...")

    mice_imputer = IterativeImputer(
        max_iter=10,
        random_state=42,
        verbose=0
    )
    X_mice = mice_imputer.fit_transform(X_numeric)

    results['MICE'] = {
        'n_samples': len(X_mice),
        'n_features': X_mice.shape[1],
        'X': X_mice,
        'y': y,
        'imputer': mice_imputer
    }

    print(f"   Retained: {len(X_mice)}/{len(X_numeric)} samples (100.0%)")
    print(f"   ✓ Chained-equation imputation completed")

    return results, X_numeric.columns

# Compare methods on training data
print("\nTraining Cohort Imputation Comparison:")
train_imputation_results, feature_names = compare_imputation_methods(train_df)

# ============================================================================
# SECTION 7: MODEL PERFORMANCE COMPARISON
# ============================================================================
print("\n[7/8] Comparing model performance across imputation methods...")
print("=" * 80)

def evaluate_imputation_methods(imputation_results, cohort_name):
    """Train models using different imputation methods and compare"""

    print(f"\n--- {cohort_name} Model Performance ---")

    performance = []

    for method_name, data in imputation_results.items():
        X = data['X']
        y = data['y']

        if y is None or len(y) == 0:
            continue

        # Train logistic regression
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X, y)

        # Predictions
        y_pred = model.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, y_pred)

        performance.append({
            'Method': method_name,
            'N_Samples': len(X),
            'AUC': auc,
            'N_Features': X.shape[1]
        })

        print(f"\n{method_name}:")
        print(f"  N={len(X)}, AUC={auc:.4f}")

    performance_df = pd.DataFrame(performance)

    return performance_df

# Evaluate all methods
train_performance = evaluate_imputation_methods(train_imputation_results, "Training")

# ============================================================================
# SECTION 8: VISUALIZATIONS
# ============================================================================
print("\n[8/8] Creating visualizations...")
print("=" * 80)

# Figure 1: Missing Data Patterns
print("\n  Creating Figure 1: Missing data patterns...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Training - Matrix
ax1 = axes[0, 0]
if len(train_missing) > 0:
    top_missing = train_missing.head(15)
    ax1.barh(range(len(top_missing)), top_missing['Missing_Percent'], color='#E74C3C', edgecolor='black')
    ax1.set_yticks(range(len(top_missing)))
    ax1.set_yticklabels(top_missing['Variable'], fontsize=9)
    ax1.set_xlabel('Missing Percentage (%)', fontweight='bold')
    ax1.set_title('Training Cohort: Variables with Missing Data', fontweight='bold', pad=10)
    ax1.grid(axis='x', alpha=0.3)
    ax1.set_xlim(0, 100)
else:
    ax1.text(0.5, 0.5, 'No Missing Data', ha='center', va='center', fontsize=14)
    ax1.set_title('Training Cohort: Variables with Missing Data', fontweight='bold', pad=10)

# Validation - Matrix
ax2 = axes[0, 1]
if len(val_missing) > 0:
    top_missing = val_missing.head(15)
    ax2.barh(range(len(top_missing)), top_missing['Missing_Percent'], color='#3498DB', edgecolor='black')
    ax2.set_yticks(range(len(top_missing)))
    ax2.set_yticklabels(top_missing['Variable'], fontsize=9)
    ax2.set_xlabel('Missing Percentage (%)', fontweight='bold')
    ax2.set_title('Validation Cohort: Variables with Missing Data', fontweight='bold', pad=10)
    ax2.grid(axis='x', alpha=0.3)
    ax2.set_xlim(0, 100)
else:
    ax2.text(0.5, 0.5, 'No Missing Data', ha='center', va='center', fontsize=14)
    ax2.set_title('Validation Cohort: Variables with Missing Data', fontweight='bold', pad=10)

# Patient-level missing distribution
ax3 = axes[1, 0]
ax3.hist(train_patient_missing, bins=30, color='#E74C3C', alpha=0.7, edgecolor='black', label='Development')
ax3.axvline(train_patient_missing.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {train_patient_missing.mean():.1f}')
ax3.set_xlabel('Number of Missing Variables per Patient', fontweight='bold')
ax3.set_ylabel('Frequency', fontweight='bold')
ax3.set_title('Training: Distribution of Missing Data per Patient', fontweight='bold', pad=10)
ax3.legend()
ax3.grid(alpha=0.3)

ax4 = axes[1, 1]
ax4.hist(val_patient_missing, bins=30, color='#3498DB', alpha=0.7, edgecolor='black', label='Validation')
ax4.axvline(val_patient_missing.mean(), color='blue', linestyle='--', linewidth=2, label=f'Mean: {val_patient_missing.mean():.1f}')
ax4.set_xlabel('Number of Missing Variables per Patient', fontweight='bold')
ax4.set_ylabel('Frequency', fontweight='bold')
ax4.set_title('Validation: Distribution of Missing Data per Patient', fontweight='bold', pad=10)
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('Figure1_Missing_Data_Patterns.jpg', dpi=300, bbox_inches='tight',
            format='jpeg', pil_kwargs={'quality': 95})
plt.savefig('Figure1_Missing_Data_Patterns.pdf', bbox_inches='tight')
plt.show()

print("    ✓ Figure 1 saved (JPG + PDF)")

# Figure 2: Imputation Method Comparison
print("  Creating Figure 2: Imputation method comparison...")

fig, ax = plt.subplots(figsize=(10, 6))

methods = train_performance['Method'].tolist()
aucs = train_performance['AUC'].tolist()
samples = train_performance['N_Samples'].tolist()

colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']
bars = ax.bar(range(len(methods)), aucs, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)

# Add value labels
for i, (bar, auc, n) in enumerate(zip(bars, aucs, samples)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
           f'{auc:.4f}\n(N={n})', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Apparent AUC (development cohort)', fontsize=12, fontweight='bold')
ax.set_xlabel('Imputation Method', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([{'Mean_Imputation': 'Mean imputation', 'Median_Imputation': 'Median imputation', 'MICE': 'Single chained-equation\nimputation'}.get(m, m) for m in methods], fontsize=10)
ax.set_ylim(0.85, 0.95)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('S7_Fig_Imputation_Comparison.jpg', dpi=300, bbox_inches='tight',
            format='jpeg', pil_kwargs={'quality': 95})
plt.savefig('S7_Fig_Imputation_Comparison.pdf', bbox_inches='tight')
plt.show()

print("    ✓ Figure 2 saved (JPG + PDF)")

# ============================================================================
# SAVE RESULTS
# ============================================================================
print("\n✓ Saving results...")

# Save summary tables
train_missing.to_csv('Missing_Data_Summary_Training.csv', index=False)
val_missing.to_csv('Missing_Data_Summary_Validation.csv', index=False)

if train_comparison is not None:
    train_comparison.to_csv('Complete_vs_Incomplete_Training.csv', index=False)
if val_comparison is not None:
    val_comparison.to_csv('Complete_vs_Incomplete_Validation.csv', index=False)

train_performance.to_csv('Imputation_Method_Performance.csv', index=False)

# Create comprehensive report
report = f"""
================================================================================
PHASE 1 - MISSING DATA STRATEGY ANALYSIS REPORT
================================================================================

1. MISSING DATA OVERVIEW
--------------------------------------------------------------------------------
Training Cohort (N={len(train_df)}):
  - Complete cases: {(~train_df.isnull().any(axis=1)).sum()} ({(~train_df.isnull().any(axis=1)).sum()/len(train_df)*100:.1f}%)
  - Variables with missing: {len(train_missing)}

Validation Cohort (N={len(val_df)}):
  - Complete cases: {(~val_df.isnull().any(axis=1)).sum()} ({(~val_df.isnull().any(axis=1)).sum()/len(val_df)*100:.1f}%)
  - Variables with missing: {len(val_missing)}

2. LITTLE'S MCAR TEST RESULTS
--------------------------------------------------------------------------------
Training Cohort:
  - Chi-square: {train_mcar['chi_square']:.2f}
  - P-value: {train_mcar['p_value']:.4f}
  - Conclusion: {train_mcar['conclusion']}

Validation Cohort:
  - Chi-square: {val_mcar['chi_square']:.2f}
  - P-value: {val_mcar['p_value']:.4f}
  - Conclusion: {val_mcar['conclusion']}

3. IMPUTATION METHOD PERFORMANCE
--------------------------------------------------------------------------------
{train_performance.to_string(index=False)}

4. RECOMMENDATIONS
--------------------------------------------------------------------------------
Based on the analyses:

✓ MCAR Assessment: {train_mcar['conclusion']}
✓ Complete Case Analysis: {'Justified' if (train_mcar['p_value'] >= 0.05 and (~train_df.isnull().any(axis=1)).sum()/len(train_df) > 0.5) else 'Consider imputation'}
✓ Best Imputation Method: {train_performance.loc[train_performance['AUC'].idxmax(), 'Method']}

CLINICAL JUSTIFICATION:
- If MCAR holds and complete cases >50%, complete case analysis is justified
- Median imputation recommended for robustness to outliers
- Single chained-equation imputation (one imputed data set); multiple imputation was not performed

================================================================================
"""

with open('Missing_Data_Analysis_Report.txt', 'w') as f:
    f.write(report)

print(report)

# ============================================================================
# DOWNLOAD FILES
# ============================================================================
print("\n" + "=" * 80)
print("DOWNLOADING RESULTS")
print("=" * 80)

files.download('Missing_Data_Summary_Training.csv')
files.download('Missing_Data_Summary_Validation.csv')
if train_comparison is not None:
    files.download('Complete_vs_Incomplete_Training.csv')
if val_comparison is not None:
    files.download('Complete_vs_Incomplete_Validation.csv')
files.download('Imputation_Method_Performance.csv')
files.download('Missing_Data_Analysis_Report.txt')
files.download('Figure1_Missing_Data_Patterns.jpg')
files.download('Figure1_Missing_Data_Patterns.pdf')
files.download('S7_Fig_Imputation_Comparison.jpg')
files.download('S7_Fig_Imputation_Comparison.pdf')

print("\n✓ All files downloaded successfully!")
print("\n" + "=" * 80)
print("PHASE 1 MISSING DATA ANALYSIS COMPLETED!")
print("=" * 80)


print("Section 18 complete.")
# [Reproducibility] Verify: Development complete-case N ~ 1066 on 4 MLRA predictors
# [TODO] Full Little MCAR test requires statsmodels or R (pypi: rpy2). Current result is approximate.

## Section 20 · Fairness Analysis — MLRA vs XGBoost Standard
**Prerequisite**: Sections 3, 5, 7 (all_results in memory).

- Unified cohort: MLRA complete cases in the validation cohort (n = 1,066); XGBoost Standard predictions aligned to the same patients
- Youden-optimal thresholds computed from the predictions in the validation cohort (MLRA, n = 1,066; XGBoost Standard, n = 1,243)
- Fairness metrics: absolute sex differences in TPR (equal opportunity), positive prediction rate (demographic parity), PPV, NPV, and AUC; no composite score or acceptance threshold
- Outputs: `S8_Fig_Integrated_Fairness.jpg` (S8 Fig), `S20_Table_Fairness_Analysis.xlsx` (S20 Table), single-metric figures and supporting files

In [ ]:
# ============================================================
# Section 20: Extended Fairness Analysis — MLRA vs XGBoost Standard
# Source: 61_Fairness Cell 4 (final version — 15 outputs)
# Predictions loaded from all_results (Sections 5 and 7)
# [FIXED] Classification threshold: fixed 0.5 → Youden-optimal per model
#         computed from the predictions (roc_curve), not entered as fixed values
# [FIXED] Composite Fairness Score removed (no statistical basis)
# [FIXED] Acceptable (≤0.10) / Warning (0.20) threshold lines removed
#         (no statistical basis; Methods §8.13 revised)
# ============================================================
print("=" * 70)
print("Section 20: Extended Fairness Analysis (MLRA vs XGBoost Standard)")
print("=" * 70)

import matplotlib.patches as _mpatch21
from google.colab import files

plt.rcParams.update({
    "figure.dpi":600, "savefig.dpi":600,
    "font.family":"sans-serif", "font.sans-serif":["Arial","DejaVu Sans"],
    "font.size":11, "axes.labelsize":12, "axes.titlesize":13,
    "axes.linewidth":1.2, "xtick.labelsize":10, "ytick.labelsize":10,
    "legend.fontsize":10, "lines.linewidth":2.0,
    "axes.grid":True, "grid.alpha":0.3, "grid.linestyle":"--",
    "savefig.bbox":"tight", "savefig.pad_inches":0.1,
})
COLOR_MLRA_21   = "#696969"
COLOR_XGB_21    = "#6495ED"
COLOR_MLRA      = COLOR_MLRA_21
COLOR_XGBOOST   = COLOR_XGB_21

# ── Retrieve predictions from all_results ─────────────────────────────────
mlra_res21 = all_results.get("MLRA", {})
xgb_res21  = all_results.get("XGBoost_Standard", {})

if not mlra_res21 or not xgb_res21:
    print("[WARNING] MLRA or XGBoost_Standard not found in all_results.")
    print("  Please ensure Sections 5 and 7 have been executed first.")
else:
    mlra_val_proba21 = np.asarray(mlra_res21.get("val_proba", []))
    mlra_val_true21  = np.asarray(mlra_res21.get("val_true",  []))
    mlra_val_idx21   = mlra_res21.get("val_idx", None)
    xgb_val_proba21  = np.asarray(xgb_res21.get("val_proba", []))

    # ── Alignment: XGBoost → MLRA complete-case indices ──────────────────
    # val_idx must be the positional indices of MLRA complete cases in val_df.
    # CRITICAL: the simple truncation xgb_val_proba21[:N] is INCORRECT because
    # MLRA complete cases are not necessarily the first N rows of val_df.
    # Truncation assigns XGBoost predictions from different patients than
    # mlra_val_true21, yielding near-random AUC (~0.5) for XGBoost.
    #
    # Key names attempted in priority order:
    _mlra_val_idx_raw = (mlra_res21.get("val_idx") or
                         mlra_res21.get("val_index") or
                         mlra_res21.get("complete_idx") or
                         mlra_res21.get("complete_case_idx") or
                         mlra_res21.get("idx"))

    if _mlra_val_idx_raw is not None:
        mlra_val_idx21 = np.asarray(_mlra_val_idx_raw)
        if len(mlra_val_idx21) == len(mlra_val_proba21) and \
                len(mlra_val_idx21) < len(xgb_val_proba21):
            xgb_aligned21 = xgb_val_proba21[mlra_val_idx21]
            val_aligned21 = val_df.iloc[mlra_val_idx21].reset_index(drop=True)
            print(f"  ✓ val_idx found: XGBoost aligned to MLRA complete cases "
                  f"(N={len(mlra_val_idx21)})")
        else:
            raise RuntimeError(
                f"val_idx shape mismatch: val_idx N={len(mlra_val_idx21)}, "
                f"mlra_val_proba N={len(mlra_val_proba21)}, "
                f"xgb_val_proba N={len(xgb_val_proba21)}. "
                "Re-run Section 5 to ensure consistent val_idx storage."
            )
    else:
        # val_idx unavailable — attempt reconstruction from val_df completeness.
        # Check which rows in val_df have no missing values in the columns
        # that correspond to MLRA predictor candidates.
        _candidate_cols = [c for c in val_df.columns
                           if any(kw in c.lower() for kw in
                                  ['protein', 'bilirubin', 'bili', 'wbc',
                                   'crp', 'c-react', 'white blood'])]
        if not _candidate_cols:
            # Broadest fallback: completeness across all numeric columns
            _candidate_cols = val_df.select_dtypes(include='number').columns.tolist()
            _candidate_cols = [c for c in _candidate_cols if c != 'Outcome']

        _complete_mask = val_df[_candidate_cols].notna().all(axis=1)
        _recon_idx     = np.where(_complete_mask)[0]

        if len(_recon_idx) == len(mlra_val_proba21):
            xgb_aligned21  = xgb_val_proba21[_recon_idx]
            val_aligned21  = val_df.iloc[_recon_idx].reset_index(drop=True)
            mlra_val_idx21 = _recon_idx
            _auc_check = roc_auc_score(mlra_val_true21, xgb_aligned21)
            if _auc_check < 0.70:
                raise RuntimeError(
                    f"Reconstructed alignment validation failed: "
                    f"XGBoost AUC on aligned cohort = {_auc_check:.3f} "
                    f"(expected > 0.70). The completeness-based reconstruction "
                    f"does not match MLRA complete cases. "
                    f"Fix: store mlra_results['val_idx'] in Section 5 code."
                )
            print(f"  ⚠  val_idx reconstructed from completeness check "
                  f"(N={len(_recon_idx)}, AUC_check={_auc_check:.3f})")
        else:
            raise RuntimeError(
                f"'val_idx' not found in mlra_results under any known key, "
                f"and completeness reconstruction gave N={len(_recon_idx)} "
                f"(expected N={len(mlra_val_proba21)}). "
                f"Fix: add 'val_idx' storage to Section 5 MLRA code, "
                f"e.g. mlra_results['val_idx'] = complete_case_indices, "
                f"OR run Section 15 before this section."
            )

    y_aligned21 = mlra_val_true21
    print(f"  Aligned validation N = {len(y_aligned21)}")
    print(f"  MLRA AUC    = {roc_auc_score(y_aligned21, mlra_val_proba21):.3f}")
    print(f"  XGBoost AUC = {roc_auc_score(y_aligned21, xgb_aligned21):.3f}")

    # =========================================================================
    # Step 3: Fairness Metrics Calculation
    # =========================================================================
    print("\n" + "=" * 80)
    print("STEP 3: Fairness Metrics Calculation")
    print("=" * 80)

    def calculate_fairness_metrics(y_true, y_proba, group_labels, threshold=0.5):
        """
        Compute classification metrics by subgroup at the given threshold.
        threshold: Youden-optimal per model (not fixed 0.5).
        """
        results = {}
        unique_groups = sorted(pd.Series(group_labels).dropna().unique())

        for group in unique_groups:
            mask          = pd.Series(group_labels) == group
            y_true_group  = np.array(y_true)[mask]
            y_proba_group = np.array(y_proba)[mask]
            y_pred_group  = (y_proba_group >= threshold).astype(int)

            if len(np.unique(y_true_group)) == 2:
                cm = confusion_matrix(y_true_group, y_pred_group)
                if cm.shape == (2, 2):
                    tn, fp, fn, tp = cm.ravel()
                else:
                    tn = fp = fn = tp = 0
            else:
                tn = fp = fn = tp = 0

            n_total         = len(y_true_group)
            n_positive_pred = y_pred_group.sum()
            n_positive_true = y_true_group.sum()

            tpr           = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            fpr           = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            tnr           = tn / (tn + fp) if (tn + fp) > 0 else np.nan
            ppv           = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            npv           = tn / (tn + fn) if (tn + fn) > 0 else np.nan
            positive_rate = n_positive_pred / n_total if n_total > 0 else np.nan
            prevalence    = n_positive_true / n_total if n_total > 0 else np.nan

            try:
                auc_group = (roc_auc_score(y_true_group, y_proba_group)
                             if len(np.unique(y_true_group)) == 2 else np.nan)
            except:
                auc_group = np.nan

            results[group] = {
                'N': n_total, 'Events': int(n_positive_true),
                'TPR': tpr, 'FPR': fpr, 'TNR': tnr,
                'PPV': ppv, 'NPV': npv,
                'Positive_Rate': positive_rate,
                'Prevalence': prevalence,
                'AUC': auc_group,
            }

        return pd.DataFrame(results).T

    def calculate_fairness_disparities(metrics_df):
        """Absolute pairwise fairness disparities between two groups."""
        disparities = {}
        groups = metrics_df.index.tolist()
        if len(groups) == 2:
            g0, g1 = groups[0], groups[1]
            disparities = {
                'Equal_Opportunity' : abs(metrics_df.loc[g0, 'TPR']           - metrics_df.loc[g1, 'TPR']),
                'Equalized_Odds_TPR': abs(metrics_df.loc[g0, 'TPR']           - metrics_df.loc[g1, 'TPR']),
                'Equalized_Odds_FPR': abs(metrics_df.loc[g0, 'FPR']           - metrics_df.loc[g1, 'FPR']),
                'Demographic_Parity': abs(metrics_df.loc[g0, 'Positive_Rate'] - metrics_df.loc[g1, 'Positive_Rate']),
                'PPV_Difference'    : abs(metrics_df.loc[g0, 'PPV']           - metrics_df.loc[g1, 'PPV']),
                'NPV_Difference'    : abs(metrics_df.loc[g0, 'NPV']           - metrics_df.loc[g1, 'NPV']),
                'AUC_Difference'    : abs(metrics_df.loc[g0, 'AUC']           - metrics_df.loc[g1, 'AUC']),
            }
        return disparities

    all_fairness_results = []
    all_sex_metrics      = {}

    # ── Aligned cohort setup ──────────────────────────────────────────────
    # Both models must be evaluated on the IDENTICAL patient cohort.
    # Reference cohort: MLRA complete cases (N=1,066, listwise deletion).
    # XGBoost uses xgb_aligned21 (probabilities indexed to MLRA indices).
    # Shared y_true and sex_labels derived from val_aligned21.
    #
    # Rationale: Fairness disparities computed on different denominators
    # (MLRA N=1,066 vs XGBoost N=1,243) are not directly comparable.
    # Restricting both models to the identical cohort ensures that
    # observed disparities reflect model behaviour, not cohort differences.
    # This is consistent with Section 15 and the Step 5 Download cell.
    # Youden-optimal thresholds computed from predictions (same rule as calculate_all_metrics, Section 8)
    from sklearn.metrics import roc_curve

    def _youden_threshold(y_true, y_proba):
        fpr, tpr, thr = roc_curve(np.asarray(y_true, float), np.asarray(y_proba, float))
        return float(thr[np.argmax(tpr - fpr)])

    _xgb_full = all_results['XGBoost_Standard']
    _YOUDEN_THRESHOLDS_20 = {
        'MLRA'            : _youden_threshold(mlra_val_true21, mlra_val_proba21),                 # n = 1,066
        'XGBoost Standard': _youden_threshold(_xgb_full['val_true'], _xgb_full['val_proba']),     # n = 1,243
    }
    assert len(mlra_val_true21) == 1066 and len(_xgb_full['val_true']) == 1243

    # Impact check against the previously used rounded thresholds
    for _m, _p in [('MLRA', mlra_val_proba21), ('XGBoost Standard', xgb_aligned21)]:
        _t = _YOUDEN_THRESHOLDS_20[_m]
        _lo, _hi = sorted([_t, round(_t, 3)])
        _n_between = int(((np.asarray(_p) >= _lo) & (np.asarray(_p) < _hi)).sum())
        print(f"{_m}: exact threshold = {_t:.6f}; patients between exact and 3-decimal value "
              f"(unified cohort, n = 1,066): {_n_between}")
    _PROBA_MAP = {
        'MLRA'            : mlra_val_proba21,  # N=1,066 complete cases
        'XGBoost Standard': xgb_aligned21,     # N=1,066 aligned to MLRA idx
    }
    _y_true_shared = mlra_val_true21                   # N=1,066 shared outcome
    _sex_shared    = val_aligned21['Sex'].values       # N=1,066 shared sex labels

    print(f"\n  Fairness cohort: N={len(_y_true_shared)} "
          f"(MLRA complete cases; XGBoost aligned to same indices)")

    for display_name in ['MLRA', 'XGBoost Standard']:
        print(f"\nAnalyzing {display_name}...")
        y_proba    = _PROBA_MAP[display_name]
        y_true     = _y_true_shared
        sex_labels = _sex_shared

        overall_auc = roc_auc_score(y_true, y_proba)
        print(f"  N={len(y_true)}, Overall AUC (aligned cohort): {overall_auc:.4f}")

        _thr = _YOUDEN_THRESHOLDS_20[display_name]
        print(f"  Classification threshold (Youden-optimal): {_thr:.6f}")
        sex_metrics     = calculate_fairness_metrics(
            y_true, y_proba, sex_labels, threshold=_thr)
        sex_disparities = calculate_fairness_disparities(sex_metrics)

        all_sex_metrics[display_name] = sex_metrics

        fairness_row = {
            'Model'             : display_name,
            'Overall_AUC'       : overall_auc,
            'Equal_Opportunity' : sex_disparities.get('Equal_Opportunity',  np.nan),
            'Demographic_Parity': sex_disparities.get('Demographic_Parity', np.nan),
            'PPV_Diff'          : sex_disparities.get('PPV_Difference',     np.nan),
            'NPV_Diff'          : sex_disparities.get('NPV_Difference',     np.nan),
            'AUC_Diff_Sex'      : sex_disparities.get('AUC_Difference',     np.nan),
        }
        all_fairness_results.append(fairness_row)
        print(f"  Fairness metrics calculated at Youden threshold={_thr}")

    df_fairness = pd.DataFrame(all_fairness_results)
    # NOTE: Composite Fairness Score not computed (no statistical basis).

    print("\n" + "=" * 80)
    print("Fairness Summary (individual metric disparities)")
    print("=" * 80)
    print(df_fairness[['Model', 'Overall_AUC',
                        'Equal_Opportunity', 'Demographic_Parity',
                        'PPV_Diff', 'NPV_Diff',
                        'AUC_Diff_Sex']].to_string(index=False))

    # =========================================================================
    # Step 4: Figure Generation
    # =========================================================================
    print("\n" + "=" * 80)
    print("STEP 4: Publication-Quality Figure Generation")
    print("=" * 80)

    saved_files  = []
    FIGSIZE_BAR  = (10, 6)
    BAR_WIDTH    = 0.6
    ALPHA_BAR    = 0.85

    def create_metric_comparison(metric_name, metric_col, ylabel, filename):
        """
        Bar chart for one fairness metric.
        No threshold reference lines (Acceptable/Warning removed:
        no statistical basis per Methods §8.13).
        """
        fig, ax = plt.subplots(figsize=FIGSIZE_BAR)
        models  = df_fairness['Model'].values
        values  = df_fairness[metric_col].values
        colors  = [COLOR_MLRA if 'MLRA' in m else COLOR_XGBOOST for m in models]

        bars = ax.barh(range(len(models)), values, height=BAR_WIDTH,
                       color=colors, alpha=ALPHA_BAR,
                       edgecolor='black', linewidth=1.5)

        _max_val = max(values)
        _offset  = _max_val * 0.04   # relative offset: 4% of max (scale-invariant)
        for i, (bar, val) in enumerate(zip(bars, values)):
            ax.text(val + _offset, i, f'{val:.4f}',
                    va='center', fontsize=10, fontweight='bold')

        ax.set_yticks(range(len(models)))
        ax.set_yticklabels(models, fontsize=11, fontweight='bold')
        ax.set_xlabel(ylabel, fontsize=12, fontweight='bold')
        ax.set_title(f'{metric_name}\nMLRA vs XGBoost Standard',
                     fontsize=13, fontweight='bold', pad=15)
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        ax.set_axisbelow(True)
        ax.set_xlim(0, _max_val * 1.50)  # wider margin to accommodate label
        plt.tight_layout()
        plt.savefig(f'{filename}.jpg', dpi=600, bbox_inches='tight',
                    format='jpeg', pil_kwargs={'quality': 95})
        saved_files.append(f'{filename}.jpg')
        print(f"✓ {filename}.jpg")
        plt.show()
        plt.close()

    print("\nGenerating Figure 1: Equal Opportunity...")
    create_metric_comparison(
        'Equal Opportunity (Sex-based)', 'Equal_Opportunity',
        'Difference in True Positive Rate (Sensitivity)',
        'Figure1_Equal_Opportunity')

    print("Generating Figure 2: Demographic Parity...")
    create_metric_comparison(
        'Demographic Parity (Sex-based)', 'Demographic_Parity',
        'Difference in Positive Prediction Rate',
        'Figure2_Demographic_Parity')

    print("Generating Figure 3: PPV Difference...")
    create_metric_comparison(
        'Positive Predictive Value Difference (Sex-based)', 'PPV_Diff',
        'Difference in PPV (Precision)',
        'Figure3_PPV_Difference')

    print("Generating Figure 4: NPV Difference...")
    create_metric_comparison(
        'Negative Predictive Value Difference (Sex-based)', 'NPV_Diff',
        'Difference in NPV',
        'Figure4_NPV_Difference')

    # Figure 5: AUC vs Equal Opportunity scatter (no threshold lines)
    print("Generating Figure 5: Performance-Fairness scatter...")
    fig, ax = plt.subplots(figsize=(10, 8))

    for i, row in df_fairness.iterrows():
        model  = row['Model']
        color  = COLOR_MLRA if 'MLRA' in model else COLOR_XGBOOST
        marker = 'D' if 'MLRA' in model else 'o'
        size   = 400 if 'MLRA' in model else 350
        ax.scatter(row['Overall_AUC'], row['Equal_Opportunity'],
                   c=color, s=size, alpha=0.85, marker=marker,
                   edgecolors='black', linewidth=2.0, label=model, zorder=5)

    label_offsets = {'MLRA': (10, -15), 'XGBoost Standard': (10, 10)}
    for i, row in df_fairness.iterrows():
        model  = row['Model']
        offset = label_offsets.get(model, (10, 10))
        ax.annotate(model,
                    (row['Overall_AUC'], row['Equal_Opportunity']),
                    xytext=offset, textcoords='offset points',
                    fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                              edgecolor='black', alpha=0.8, linewidth=1.0),
                    zorder=6)

    ax.set_xlabel('Discrimination (AUC)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Equal Opportunity\n(|TPR_Female − TPR_Male|)',
                  fontsize=13, fontweight='bold')
    ax.set_title('Performance vs Equal Opportunity\nMLRA vs XGBoost Standard',
                 fontsize=14, fontweight='bold', pad=20)
    ax.grid(True, alpha=0.3, linestyle='--', zorder=1)
    ax.set_axisbelow(True)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(),
              loc='upper right', fontsize=9,
              frameon=True, shadow=True, fancybox=True, markerscale=0.6)
    ax.set_xlim(df_fairness['Overall_AUC'].min() - 0.01,
                df_fairness['Overall_AUC'].max() + 0.01)
    ax.set_ylim(0, df_fairness['Equal_Opportunity'].max() * 1.35)
    plt.tight_layout()
    plt.savefig('Figure5_Performance_Fairness_Tradeoff.jpg', dpi=600,
                bbox_inches='tight', format='jpeg', pil_kwargs={'quality': 95})
    saved_files.append('Figure5_Performance_Fairness_Tradeoff.jpg')
    print("✓ Figure5_Performance_Fairness_Tradeoff.jpg")
    plt.show()
    plt.close()

    # Figure 6: Integrated 2×3 grid
    print("Generating Figure 6: Integrated layout for journal submission...")

    import matplotlib.gridspec as gridspec

    fig = plt.figure(figsize=(20, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3,
                            left=0.08, right=0.98, top=0.95, bottom=0.06)
    models = df_fairness['Model'].tolist()
    colors = [COLOR_MLRA if 'MLRA' in m else COLOR_XGBOOST for m in models]

    def _panel_bar(ax, metric_col, xlabel, title_label):
        """Horizontal bar panel — no threshold reference lines."""
        values   = df_fairness[metric_col].tolist()
        _max_val = max(values)
        _offset  = _max_val * 0.04   # relative offset: 4% of max (scale-invariant)
        bars     = ax.barh(models, values, color=colors,
                           alpha=0.85, edgecolor='black', linewidth=1.5)
        for i, (bar, val) in enumerate(zip(bars, values)):
            ax.text(val + _offset, i, f'{val:.4f}',
                    va='center', fontsize=10, fontweight='bold')
        ax.set_xlabel(xlabel, fontsize=11, fontweight='bold')
        ax.set_title(title_label, fontsize=12, fontweight='bold',
                     loc='left', pad=10)
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        ax.set_xlim(0, _max_val * 1.50)  # wider margin to accommodate label

    ax1 = fig.add_subplot(gs[0, 0])
    _panel_bar(ax1, 'Equal_Opportunity',
               'Equal Opportunity Difference', 'A. Equal Opportunity')

    ax2 = fig.add_subplot(gs[0, 1])
    _panel_bar(ax2, 'Demographic_Parity',
               'Demographic Parity Difference', 'B. Demographic Parity')

    ax3 = fig.add_subplot(gs[0, 2])
    _panel_bar(ax3, 'PPV_Diff',
               'PPV Difference', 'C. PPV Difference')

    ax4 = fig.add_subplot(gs[1, 0])
    _panel_bar(ax4, 'NPV_Diff',
               'NPV Difference', 'D. NPV Difference')

    # Panel E: AUC vs Equal Opportunity scatter (no threshold lines)
    ax5 = fig.add_subplot(gs[1, 1:])
    for i, row in df_fairness.iterrows():
        model  = row['Model']
        color  = COLOR_MLRA if 'MLRA' in model else COLOR_XGBOOST
        marker = 'D' if 'MLRA' in model else 'o'
        size   = 350 if 'MLRA' in model else 300
        ax5.scatter(row['Overall_AUC'], row['Equal_Opportunity'],
                    c=color, s=size, alpha=0.85, marker=marker,
                    edgecolors='black', linewidth=1.8, label=model, zorder=5)

    label_offsets_int = {'MLRA': (10, -15), 'XGBoost Standard': (10, 10)}
    for i, row in df_fairness.iterrows():
        model  = row['Model']
        offset = label_offsets_int.get(model, (10, 10))
        ax5.annotate(model,
                     (row['Overall_AUC'], row['Equal_Opportunity']),
                     xytext=offset, textcoords='offset points',
                     fontsize=10, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                               edgecolor='black', alpha=0.8, linewidth=1.0),
                     zorder=6)

    ax5.set_xlabel('Overall AUC (Discrimination)', fontsize=11, fontweight='bold')
    ax5.set_ylabel('Equal Opportunity\n(|TPR_Female − TPR_Male|)',
                   fontsize=11, fontweight='bold')
    ax5.set_title('E. Performance vs Equal Opportunity',
                  fontsize=12, fontweight='bold', loc='left', pad=10)
    ax5.grid(True, alpha=0.3, linestyle='--', zorder=1)
    ax5.set_axisbelow(True)
    handles, labels = ax5.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax5.legend(unique.values(), unique.keys(),
               loc='upper right', fontsize=9,
               frameon=True, shadow=True, fancybox=True, markerscale=0.6)
    ax5.set_xlim(df_fairness['Overall_AUC'].min() - 0.01,
                 df_fairness['Overall_AUC'].max() + 0.01)
    ax5.set_ylim(0, df_fairness['Equal_Opportunity'].max() * 1.35)

    fig.suptitle(' ', fontsize=15, fontweight='bold', y=0.98)
    plt.savefig('S8_Fig_Integrated_Fairness.jpg', dpi=600,
                bbox_inches='tight', format='jpeg', pil_kwargs={'quality': 95})
    saved_files.append('S8_Fig_Integrated_Fairness.jpg')
    print("✓ S8_Fig_Integrated_Fairness.jpg")
    plt.show()
    plt.close()

    # =========================================================================
    # Step 5: Export Results
    # =========================================================================
    print("\n" + "=" * 80)
    print("STEP 5: Export Results")
    print("=" * 80)

    df_fairness.to_csv('Fairness_Summary_MLRA_vs_XGBoost.csv',
                       index=False, encoding='utf-8-sig')
    saved_files.append('Fairness_Summary_MLRA_vs_XGBoost.csv')
    print("✓ Fairness_Summary_MLRA_vs_XGBoost.csv")

    with pd.ExcelWriter('Sex_based_Metrics_Detailed.xlsx', engine='openpyxl') as writer:
        for model_name, metrics_df in all_sex_metrics.items():
            sheet_name = model_name.replace(' ', '_')[:31]
            metrics_df.to_excel(writer, sheet_name=sheet_name)
    saved_files.append('Sex_based_Metrics_Detailed.xlsx')
    print("✓ Sex_based_Metrics_Detailed.xlsx")

    # ── S20 Table: Integrated Fairness Analysis ───────────────────────────
    # Reconstructed from deleted Step 5 Download cell.
    # Uses runtime all_sex_metrics (Youden-optimal thresholds) and
    # all_disparities computed above.
    _MODELS_S17  = ['MLRA', 'XGBoost Standard']
    _GRP_LABEL   = {0: 'Female', 1: 'Male'}

    def _f4(v):
        try:
            return f'{float(v):.4f}' if not np.isnan(float(v)) else 'N/A'
        except Exception:
            return 'N/A'

    def _fi(v):
        try:
            return f'{int(v):,}'
        except Exception:
            return 'N/A'

    def _grp(model, grp_val, col):
        try:
            return all_sex_metrics[model].loc[grp_val, col]
        except Exception:
            return np.nan

    s17_rows = []

    def _add(cat, metric, vals):
        row = {'Category': cat, 'Metric': metric}
        for m in _MODELS_S17:
            row[m] = vals.get(m, 'N/A')
        s17_rows.append(row)

    # Section A: Overall Performance
    s17_rows.append({'Category': 'Overall Performance', 'Metric': '',
                     **{m: '' for m in _MODELS_S17}})
    _add('', 'Overall AUC',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m, 'Overall_AUC'].values[0])
          for m in _MODELS_S17})
    _add('', 'Classification Threshold (Youden-optimal)',
         {m: f"{_YOUDEN_THRESHOLDS_20[m]:.3f}" for m in _MODELS_S17})
    _add('', 'Total Sample Size (Aligned Cohort)',
         {m: _fi(len(_y_true_shared)) for m in _MODELS_S17})

    # Section B: Sex-stratified Subgroup Metrics
    for grp_val, grp_label in [(0, 'Female Subgroup'), (1, 'Male Subgroup')]:
        s17_rows.append({'Category': grp_label, 'Metric': '',
                         **{m: '' for m in _MODELS_S17}})
        _add('', 'Sample Size',
             {m: _fi(_grp(m, grp_val, 'N')) for m in _MODELS_S17})
        _add('', 'Number of Events',
             {m: _fi(_grp(m, grp_val, 'Events')) for m in _MODELS_S17})
        _add('', 'Event Rate',
             {m: _f4(_grp(m, grp_val, 'Events') /
                     _grp(m, grp_val, 'N')) for m in _MODELS_S17})
        _add('', 'True Positive Rate (Sensitivity)',
             {m: _f4(_grp(m, grp_val, 'TPR')) for m in _MODELS_S17})
        _add('', 'False Positive Rate',
             {m: _f4(_grp(m, grp_val, 'FPR')) for m in _MODELS_S17})
        _add('', 'True Negative Rate (Specificity)',
             {m: _f4(1.0 - _grp(m, grp_val, 'FPR')) for m in _MODELS_S17})
        _add('', 'Positive Predictive Value',
             {m: _f4(_grp(m, grp_val, 'PPV')) for m in _MODELS_S17})
        _add('', 'Negative Predictive Value',
             {m: _f4(_grp(m, grp_val, 'NPV')) for m in _MODELS_S17})
        _add('', 'Positive Prediction Rate',
             {m: _f4(_grp(m, grp_val, 'Positive_Rate')) for m in _MODELS_S17})
        _add('', 'Area Under ROC Curve',
             {m: _f4(_grp(m, grp_val, 'AUC')) for m in _MODELS_S17})

    # Section C: Fairness Disparities
    s17_rows.append({'Category': 'Fairness Disparities', 'Metric': '',
                     **{m: '' for m in _MODELS_S17}})
    _add('', 'Equal Opportunity (|TPR_Female − TPR_Male|)',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m,
                                  'Equal_Opportunity'].values[0])
          for m in _MODELS_S17})
    _add('', 'Demographic Parity (|PosRate_Female − PosRate_Male|)',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m,
                                  'Demographic_Parity'].values[0])
          for m in _MODELS_S17})
    _add('', 'PPV Difference (|PPV_Female − PPV_Male|)',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m,
                                  'PPV_Diff'].values[0])
          for m in _MODELS_S17})
    _add('', 'NPV Difference (|NPV_Female − NPV_Male|)',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m,
                                  'NPV_Diff'].values[0])
          for m in _MODELS_S17})
    _add('', 'AUC Difference (|AUC_Female − AUC_Male|)',
         {m: _f4(df_fairness.loc[df_fairness['Model'] == m,
                                  'AUC_Diff_Sex'].values[0])
          for m in _MODELS_S17})
    # NOTE: Composite fairness score and acceptance threshold removed
    # per Methods §8.13. Fairness evaluated by individual disparities only.

    df_s17 = pd.DataFrame(s17_rows,
                           columns=['Category', 'Metric'] + _MODELS_S17)

    # Inline display
    from IPython.display import display as _disp20
    print('\n[S20 Table — Integrated Fairness Analysis]')
    _disp20(df_s17)

    # Export
    S17_XLSX = 'S20_Table_Fairness_Analysis.xlsx'
    with pd.ExcelWriter(S17_XLSX, engine='openpyxl') as writer:
        df_s17.to_excel(writer, sheet_name='S20_Table', index=False)
        ws = writer.sheets['S20_Table']
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = (
                max(len(str(c.value or '')) for c in col) + 4)
    saved_files.append(S17_XLSX)
    print(f'✓ {S17_XLSX}')

    with open('Fairness_Analysis_Report.txt', 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("FAIRNESS ANALYSIS REPORT\n")
        f.write("MLRA vs XGBoost Standard\n")
        f.write("=" * 80 + "\n\n")
        f.write("Classification threshold: Youden-optimal per model\n")
        f.write(f"  MLRA: {_YOUDEN_THRESHOLDS_20['MLRA']:.3f}\n")
        f.write(f"  XGBoost Standard: {_YOUDEN_THRESHOLDS_20['XGBoost Standard']:.3f}\n\n")
        f.write("Fairness metrics: absolute sex-based disparities per domain.\n")
        f.write("No composite score or aggregate threshold applied.\n\n")
        f.write("Model Performance Summary\n\n")
        for i, row in df_fairness.iterrows():
            f.write(f"{row['Model']}:\n")
            f.write(f"  Overall AUC         : {row['Overall_AUC']:.4f}\n")
            f.write(f"  Equal Opportunity   : {row['Equal_Opportunity']:.4f}\n")
            f.write(f"  Demographic Parity  : {row['Demographic_Parity']:.4f}\n")
            f.write(f"  PPV Difference      : {row['PPV_Diff']:.4f}\n")
            f.write(f"  NPV Difference      : {row['NPV_Diff']:.4f}\n")
            f.write(f"  AUC Diff (Sex)      : {row['AUC_Diff_Sex']:.4f}\n\n")
        f.write("Sex-based Detailed Metrics\n\n")
        for model_name, metrics_df in all_sex_metrics.items():
            f.write(f"{model_name}:\n")
            f.write(metrics_df.to_string())
            f.write("\n\n")
    saved_files.append('Fairness_Analysis_Report.txt')
    print("✓ Fairness_Analysis_Report.txt")

    # =========================================================================
    # Step 6: Automatic Download
    # =========================================================================
    print("\n" + "=" * 80)
    print("STEP 6: Automatic File Download")
    print("=" * 80)

    download_success = 0
    download_failed  = 0
    for filename in saved_files:
        try:
            files.download(filename)
            download_success += 1
            print(f"  ✓ {filename}")
        except Exception as e:
            print(f"  ✗ {filename}: {e}")
            download_failed += 1

    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE")
    print("=" * 80)
    print(f"  Downloads: {download_success}/{len(saved_files)} files")
    if download_failed > 0:
        print(f"  Failed: {download_failed} files")

print("Section 20 complete.")
# [Reproducibility] Aligned N = 1,066 (MLRA complete-case)
# [Reproducibility] Threshold: Youden-optimal, computed from the predictions
# [Reproducibility] Composite Fairness Score: NOT computed (Methods §8.13)
# [Reproducibility] Threshold lines Acceptable/Warning: REMOVED (no statistical basis)

## Section 22 · Outlier Influence Analysis
**Prerequisite**: Sections 1–7 (train_df, val_df, X_train, all_results in memory).

- Outliers: complete cases on the 4 MLRA predictors, then sequential ±3 SD removal (development, 30; validation, 56)
- Gradient boosting models are retrained on the outlier-excluded data with the final hyperparameters, the same 43 predictors, and native missing-value handling; the refit on the original data must reproduce the final XGBoost Standard predictions (assert)
- AUC: bootstrap mean with 95% percentile interval (500 iterations, seed 42)
- Only XGBoost Standard is reported in the manuscript (S19 Table, Fig 7F); HistGradientBoosting and CatBoost Ordered are supplementary checks
- Output: `S19_Table_source_Outlier_GB.xlsx`

In [ ]:
# ============================================================
# Section 22: Outlier Influence Analysis
# Source: 62_Outlier_Analysis.ipynb (FINAL CORRECT VERSION, 2025-01-18)
# [FIXED] BUG-4: eval_set=(X_val, y_val) was data leakage from validation cohort.
#   CatBoost Ordered now trains without eval_set, consistent with Cells 24-30.
# Data loading replaced with in-memory train_df / val_df
# ============================================================
print("=" * 70)
print("Section 22: Outlier Influence Analysis")
print("=" * 70)

from catboost import CatBoostClassifier as _CBCls22

_RS22 = RANDOM_STATE  # = 42

# ── Data preparation (uses train_df, val_df from Section 3) ──
def _prep_s22(df):
    out = df.copy()
    for col in out.columns:
        if col != "Outcome":
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

df_tr22 = _prep_s22(train_df)
df_va22 = _prep_s22(val_df)
features22 = [c for c in df_tr22.columns if c != "Outcome"]
X_tr22 = df_tr22[list(X_train.columns)]   # same predictors/order as the final model; native missing-value handling
y_tr22 = df_tr22["Outcome"].values
X_va22 = df_va22[list(X_train.columns)]
y_va22 = df_va22["Outcome"].values
print(f"  Training: {X_tr22.shape}, events={y_tr22.sum()}")
print(f"  Validation: {X_va22.shape}, events={y_va22.sum()}")

# =============================================================================
# SECTION 4: Outlier Detection (MLRA-Identical Method)
# =============================================================================

print("\n" + "="*80)
print("SECTION 4: MLRA-Identical Outlier Detection")
print("="*80)

mlra_features = ['Total protein (g/dL)',
                 'Direct bilirubin (mg/dL)',
                 'White blood cells (103/μL)',
                 'C-reactive protein (mg/dL)']

print(f"\nMLRA selected features ({len(mlra_features)} variables):")
for feat in mlra_features:
    print(f"  - {feat}")

def identify_outliers_mlra_method(df, mlra_features):
    """
    Identify outliers using the MLRA-identical two-step procedure:
      Step 1: Listwise deletion on the 4 MLRA predictor variables.
      Step 2: Sequential ±3 SD removal (each feature in order).
    """
    # Step 1: Listwise deletion
    df_complete = df.dropna(subset=mlra_features)
    n_missing = len(df) - len(df_complete)

    print(f"  Step 1 - Listwise deletion:")
    print(f"    Original samples: {len(df)}")
    print(f"    Complete cases: {len(df_complete)}")
    print(f"    Missing data removed: {n_missing}")

    # Step 2: Sequential +/-3 SD outlier removal
    df_no_outliers = df_complete.copy()

    for feature in mlra_features:
        mean_val = df_no_outliers[feature].mean()
        std_val = df_no_outliers[feature].std()
        mask = (df_no_outliers[feature] >= mean_val - 3*std_val) & \
               (df_no_outliers[feature] <= mean_val + 3*std_val)
        df_no_outliers = df_no_outliers[mask]

    n_outliers = len(df_complete) - len(df_no_outliers)

    print(f"  Step 2 - Sequential outlier removal:")
    print(f"    After Step 1: {len(df_complete)} samples")
    print(f"    After outlier removal: {len(df_no_outliers)} samples")
    print(f"    Outliers removed: {n_outliers}")

    remaining_indices = df_no_outliers.index
    outlier_mask = ~df.index.isin(remaining_indices)

    total_removed = outlier_mask.sum()
    print(f"  Total removed: {total_removed} ({n_missing} missing + {n_outliers} outliers)")

    return outlier_mask, n_outliers, n_missing

# [FIXED] What was wrong : identify_outliers_mlra_method was called with
#                          df_train and df_val (undefined variables).
#                          The preprocessed DataFrames prepared in this cell
#                          are named df_tr22 and df_va22.
# [FIXED] What was changed: df_train → df_tr22, df_val → df_va22.
print("\nTraining Cohort:")
train_outliers, n_tr22_out, n_train_missing = identify_outliers_mlra_method(
    df_tr22, mlra_features)
print(f"  Final dataset: {(~train_outliers).sum()} samples")

print("\nValidation Cohort:")
val_outliers, n_va22_out, n_val_missing = identify_outliers_mlra_method(
    df_va22, mlra_features)
print(f"  Final dataset: {(~val_outliers).sum()} samples")

X_tr22_clean = X_tr22[~train_outliers]
y_tr22_clean = y_tr22[~train_outliers]
X_va22_clean = X_va22[~val_outliers]
y_va22_clean = y_va22[~val_outliers]

print(f"\nVerification against MLRA analysis results:")
print(f"  Training outliers: {n_tr22_out} (expected: 30)")
print(f"  Validation outliers: {n_va22_out} (expected: 56)")

if n_tr22_out == 30 and n_va22_out == 56:
    print("\n  ✓✓✓ Exact match! ✓✓✓")

print("\n✓ Outlier identification complete")

# =============================================================================
# SECTION 5: Bootstrap AUC with 95% CI
# =============================================================================

print("\n" + "="*80)
print("SECTION 5: Bootstrap Function Setup")
print("="*80)

def bootstrap_auc_ci(y_true, y_pred, n_bootstrap=500, random_state=42):
    """Calculate AUC with 95% CI using bootstrap (500 iterations)."""
    n = len(y_true)
    auc_scores = []

    np.random.seed(random_state)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        try:
            auc = roc_auc_score(y_true[indices], y_pred[indices])
            auc_scores.append(auc)
        except:
            continue

    if len(auc_scores) == 0:
        return 0.5, 0.5, 0.5

    auc_mean = np.mean(auc_scores)
    auc_ci_lower = np.percentile(auc_scores, 2.5)
    auc_ci_upper = np.percentile(auc_scores, 97.5)

    return auc_mean, auc_ci_lower, auc_ci_upper

print("✓ Bootstrap function ready (500 iterations)")

# =============================================================================
# SECTION 6: Model Training and Evaluation
# (Exact hyperparameters + fixed iterations for CatBoost Ordered)
# =============================================================================

print("\n" + "="*80)
print("SECTION 6: Model Training with CORRECT Hyperparameters + fixed iterations")
print("="*80)

results = {}

# ---------------------------------------------------------------------------
# Model 1: XGBoost Standard
# ---------------------------------------------------------------------------

print("\n" + "="*80)
print("MODEL 1: XGBoost Standard")
print("="*80)

xgb_params = {
    'n_estimators': 80,
    'max_depth': 3,
    'learning_rate': 0.08,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'min_child_weight': 10,
    'gamma': 0.2,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'tree_method': 'hist',
    'random_state': _RS22,
    'n_jobs': 1,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss'
}

print("\n[1/2] Training on original full cohort...")
model_xgb = xgb.XGBClassifier(**xgb_standard_params)
model_xgb.fit(X_train, y_train)
pred_train = model_xgb.predict_proba(X_train)[:, 1]
pred_val = model_xgb.predict_proba(X_val)[:, 1]
assert (np.allclose(pred_train, all_results['XGBoost_Standard']['train_proba'], atol=1e-6) and
        np.allclose(pred_val, all_results['XGBoost_Standard']['val_proba'], atol=1e-6)), \
    "Outlier analysis: XGBoost does not reproduce the final model — HALT"

auc_train, ci_l_train, ci_u_train = bootstrap_auc_ci(y_train, pred_train)
auc_val, ci_l_val, ci_u_val = bootstrap_auc_ci(y_val, pred_val)

print(f"  Training:   AUC {auc_train:.3f} (95% CI: {ci_l_train:.3f}-{ci_u_train:.3f})")
print(f"  Validation: AUC {auc_val:.3f} (95% CI: {ci_l_val:.3f}-{ci_u_val:.3f})")

print("\n[2/2] Training on outlier-excluded cohort...")
model_xgb_clean = xgb.XGBClassifier(**xgb_standard_params)
model_xgb_clean.fit(X_tr22_clean, y_tr22_clean)
pred_train_clean = model_xgb_clean.predict_proba(X_tr22_clean)[:, 1]
pred_val_clean = model_xgb_clean.predict_proba(X_va22_clean)[:, 1]

auc_train_clean, ci_l_train_clean, ci_u_train_clean = bootstrap_auc_ci(y_tr22_clean, pred_train_clean)
auc_val_clean, ci_l_val_clean, ci_u_val_clean = bootstrap_auc_ci(y_va22_clean, pred_val_clean)

print(f"  Training:   AUC {auc_train_clean:.3f} (95% CI: {ci_l_train_clean:.3f}-{ci_u_train_clean:.3f})")
print(f"  Validation: AUC {auc_val_clean:.3f} (95% CI: {ci_l_val_clean:.3f}-{ci_u_val_clean:.3f})")
print(f"\n  ΔAUC Training:   {auc_train_clean - auc_train:+.3f}")
print(f"  ΔAUC Validation: {auc_val_clean - auc_val:+.3f}")

results['XGBoost Standard'] = {
    'Training_Original': {'AUC': auc_train, 'CI_Lower': ci_l_train, 'CI_Upper': ci_u_train, 'N': len(y_train)},
    'Training_Excluded': {'AUC': auc_train_clean, 'CI_Lower': ci_l_train_clean, 'CI_Upper': ci_u_train_clean, 'N': len(y_tr22_clean), 'N_removed': n_tr22_out},
    'Validation_Original': {'AUC': auc_val, 'CI_Lower': ci_l_val, 'CI_Upper': ci_u_val, 'N': len(y_val)},
    'Validation_Excluded': {'AUC': auc_val_clean, 'CI_Lower': ci_l_val_clean, 'CI_Upper': ci_u_val_clean, 'N': len(y_va22_clean), 'N_removed': n_va22_out}
}

# ---------------------------------------------------------------------------
# Model 2: HistGradientBoosting
# ---------------------------------------------------------------------------

print("\n" + "="*80)
print("MODEL 2: HistGradientBoosting")
print("="*80)

hist_params = {
    'max_iter': 50,
    'max_depth': 3,
    'learning_rate': 0.05,
    'min_samples_leaf': 50,
    'l2_regularization': 3.0,
    'max_bins': 128,
    'max_leaf_nodes': 15,
    'early_stopping': False,
    'random_state': _RS22
}

print("\n[1/2] Training on original full cohort...")
model_hist = HistGradientBoostingClassifier(**hist_params)
model_hist.fit(X_train, y_train)
pred_train = model_hist.predict_proba(X_train)[:, 1]
pred_val = model_hist.predict_proba(X_val)[:, 1]

auc_train, ci_l_train, ci_u_train = bootstrap_auc_ci(y_train, pred_train)
auc_val, ci_l_val, ci_u_val = bootstrap_auc_ci(y_val, pred_val)

print(f"  Training:   AUC {auc_train:.3f} (95% CI: {ci_l_train:.3f}-{ci_u_train:.3f})")
print(f"  Validation: AUC {auc_val:.3f} (95% CI: {ci_l_val:.3f}-{ci_u_val:.3f})")
print(f"  Expected Training AUC: 0.983")

print("\n[2/2] Training on outlier-excluded cohort...")
model_hist_clean = HistGradientBoostingClassifier(**hist_params)
model_hist_clean.fit(X_tr22_clean, y_tr22_clean)
pred_train_clean = model_hist_clean.predict_proba(X_tr22_clean)[:, 1]
pred_val_clean = model_hist_clean.predict_proba(X_va22_clean)[:, 1]

auc_train_clean, ci_l_train_clean, ci_u_train_clean = bootstrap_auc_ci(y_tr22_clean, pred_train_clean)
auc_val_clean, ci_l_val_clean, ci_u_val_clean = bootstrap_auc_ci(y_va22_clean, pred_val_clean)

print(f"  Training:   AUC {auc_train_clean:.3f} (95% CI: {ci_l_train_clean:.3f}-{ci_u_train_clean:.3f})")
print(f"  Validation: AUC {auc_val_clean:.3f} (95% CI: {ci_l_val_clean:.3f}-{ci_u_val_clean:.3f})")
print(f"\n  ΔAUC Training:   {auc_train_clean - auc_train:+.3f}")
print(f"  ΔAUC Validation: {auc_val_clean - auc_val:+.3f}")

results['HistGradientBoosting'] = {
    'Training_Original': {'AUC': auc_train, 'CI_Lower': ci_l_train, 'CI_Upper': ci_u_train, 'N': len(y_train)},
    'Training_Excluded': {'AUC': auc_train_clean, 'CI_Lower': ci_l_train_clean, 'CI_Upper': ci_u_train_clean, 'N': len(y_tr22_clean), 'N_removed': n_tr22_out},
    'Validation_Original': {'AUC': auc_val, 'CI_Lower': ci_l_val, 'CI_Upper': ci_u_val, 'N': len(y_val)},
    'Validation_Excluded': {'AUC': auc_val_clean, 'CI_Lower': ci_l_val_clean, 'CI_Upper': ci_u_val_clean, 'N': len(y_va22_clean), 'N_removed': n_va22_out}
}

# ---------------------------------------------------------------------------
# Model 3: CatBoost Ordered (CRITICAL: WITHOUT eval_set)
# ---------------------------------------------------------------------------

print("\n" + "="*80)
print("MODEL 3: CatBoost Ordered (WITHOUT eval_set)")
print("="*80)

cat_params = {
    'iterations': 100,
    'depth': 4,
    'learning_rate': 0.1,
    'l2_leaf_reg': 3.0,
    'subsample': 0.8,
    'random_strength': 1.0,
    'border_count': 128,
    'grow_policy': 'SymmetricTree',
    'bootstrap_type': 'Bernoulli',
    'sampling_frequency': 'PerTree',
    'boosting_type': 'Ordered',
    'random_seed': _RS22,
    'thread_count': 1,
    'verbose': False
}

print("\nHyperparameters:")
print(f"  iterations: {cat_params['iterations']}")
print(f"  depth: {cat_params['depth']}")
print(f"  learning_rate: {cat_params['learning_rate']}")
print(f"  l2_leaf_reg: {cat_params['l2_leaf_reg']}")
print(f"  subsample: {cat_params['subsample']}")

print("\n[1/2] Training on original full cohort...")
# [FIXED] BUG-4: eval_set=(X_val, y_val) constitutes data leakage from the
# external validation cohort. Removed to be consistent with Cells 24-30.
# CatBoost Ordered is trained with fixed iterations (no early stopping).
model_cat = _CBCls22(**cat_params)
model_cat.fit(X_train, y_train,
              verbose=False)
pred_train = model_cat.predict_proba(X_train)[:, 1]
pred_val = model_cat.predict_proba(X_val)[:, 1]

auc_train, ci_l_train, ci_u_train = bootstrap_auc_ci(y_train, pred_train)
auc_val, ci_l_val, ci_u_val = bootstrap_auc_ci(y_val, pred_val)

print(f"  Training:   AUC {auc_train:.3f} (95% CI: {ci_l_train:.3f}-{ci_u_train:.3f})")
print(f"  Validation: AUC {auc_val:.3f} (95% CI: {ci_l_val:.3f}-{ci_u_val:.3f})")
print(f"  Note: Training AUC may differ from prior run (eval_set leakage removed)")

print("\n[2/2] Training on outlier-excluded cohort...")
# [FIXED] BUG-4: eval_set referencing validation data removed.
model_cat_clean = _CBCls22(**cat_params)
model_cat_clean.fit(X_tr22_clean, y_tr22_clean,
                   verbose=False)
pred_train_clean = model_cat_clean.predict_proba(X_tr22_clean)[:, 1]
pred_val_clean = model_cat_clean.predict_proba(X_va22_clean)[:, 1]

auc_train_clean, ci_l_train_clean, ci_u_train_clean = bootstrap_auc_ci(y_tr22_clean, pred_train_clean)
auc_val_clean, ci_l_val_clean, ci_u_val_clean = bootstrap_auc_ci(y_va22_clean, pred_val_clean)

print(f"  Training:   AUC {auc_train_clean:.3f} (95% CI: {ci_l_train_clean:.3f}-{ci_u_train_clean:.3f})")
print(f"  Validation: AUC {auc_val_clean:.3f} (95% CI: {ci_l_val_clean:.3f}-{ci_u_val_clean:.3f})")
print(f"\n  ΔAUC Training:   {auc_train_clean - auc_train:+.3f}")
print(f"  ΔAUC Validation: {auc_val_clean - auc_val:+.3f}")

results['CatBoost Ordered'] = {
    'Training_Original': {'AUC': auc_train, 'CI_Lower': ci_l_train, 'CI_Upper': ci_u_train, 'N': len(y_train)},
    'Training_Excluded': {'AUC': auc_train_clean, 'CI_Lower': ci_l_train_clean, 'CI_Upper': ci_u_train_clean, 'N': len(y_tr22_clean), 'N_removed': n_tr22_out},
    'Validation_Original': {'AUC': auc_val, 'CI_Lower': ci_l_val, 'CI_Upper': ci_u_val, 'N': len(y_val)},
    'Validation_Excluded': {'AUC': auc_val_clean, 'CI_Lower': ci_l_val_clean, 'CI_Upper': ci_u_val_clean, 'N': len(y_va22_clean), 'N_removed': n_va22_out}
}

print("\n" + "="*80)
print("✓ ALL MODELS TRAINED WITH CORRECT SETTINGS")
print("="*80)

# =============================================================================
# SECTION 8: Figure Creation and File Export
# =============================================================================

print("\n" + "="*80)
print("SECTION 8: Creating Figures and Saving Results")
print("="*80)

plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1.5

def convert_results_for_figure(model_results):
    """Convert results structure for figure generation."""
    return {
        'Training': {
            'Original': model_results['Training_Original'],
            'Excluded': model_results['Training_Excluded']
        },
        'Validation': {
            'Original': model_results['Validation_Original'],
            'Excluded': model_results['Validation_Excluded']
        }
    }

def create_outlier_figure(model_name, data, fig_num):
    """Create outlier influence analysis figure."""

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))

    cohorts = ['Development\n(Training)', 'Validation']
    cohort_keys = ['Training', 'Validation']
    x_positions = [0, 1]
    bar_width = 0.35

    colors = {'Original': '#87CEEB', 'Excluded': '#FFB6C1'}

    all_aucs = []
    for cohort_key in cohort_keys:
        cohort_data = data[cohort_key]
        for condition_key in ['Original', 'Excluded']:
            condition_data = cohort_data[condition_key]
            all_aucs.extend([condition_data['CI_Lower'], condition_data['CI_Upper']])

    y_min = max(0.80, min(all_aucs) - 0.02)
    y_max = min(1.00, max(all_aucs) + 0.08)

    for i, (cohort, cohort_key) in enumerate(zip(cohorts, cohort_keys)):
        cohort_data = data[cohort_key]

        original = cohort_data['Original']
        x_orig = x_positions[i] - bar_width/2

        ax.bar(x_orig, original['AUC'], bar_width,
               color=colors['Original'], edgecolor='black', linewidth=1.5,
               label='Original (Full cohort)' if i == 0 else "")

        ax.errorbar(x_orig, original['AUC'],
                   yerr=[[original['AUC'] - original['CI_Lower']],
                         [original['CI_Upper'] - original['AUC']]],
                   fmt='none', ecolor='black', capsize=5, capthick=2, linewidth=2)

        excluded = cohort_data['Excluded']
        x_excl = x_positions[i] + bar_width/2

        ax.bar(x_excl, excluded['AUC'], bar_width,
               color=colors['Excluded'], edgecolor='black', linewidth=1.5,
               label='Outlier-excluded (±3SD)' if i == 0 else "")

        ax.errorbar(x_excl, excluded['AUC'],
                   yerr=[[excluded['AUC'] - excluded['CI_Lower']],
                         [excluded['CI_Upper'] - excluded['AUC']]],
                   fmt='none', ecolor='black', capsize=5, capthick=2, linewidth=2)

        delta_auc = excluded['AUC'] - original['AUC']
        mid_x = x_positions[i]
        mid_y = 0.96  # Fixed at Y = 0.96

        text_label = f"ΔAUC={delta_auc:+.3f}\n(n={excluded['N_removed']} excluded)"
        box_color = 'lightyellow' if delta_auc < 0 else 'lightgreen'

        ax.text(mid_x, mid_y, text_label, ha='center', va='bottom', fontsize=11,
               bbox=dict(boxstyle='round,pad=0.5', facecolor=box_color,
                        edgecolor='black', linewidth=2, alpha=0.9))

        auc_y_position = y_min + (y_max - y_min) * 0.15

        ax.text(x_orig, auc_y_position, f'{original["AUC"]:.3f}',
               ha='center', va='center', fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                        edgecolor='black', linewidth=1.0, alpha=0.9))
        ax.text(x_excl, auc_y_position, f'{excluded["AUC"]:.3f}',
               ha='center', va='center', fontsize=11, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                        edgecolor='black', linewidth=1.0, alpha=0.9))

    ax.set_ylabel('Area Under the ROC Curve (AUC)', fontsize=14, fontweight='bold')
    ax.set_title(f'Outlier Influence Analysis ({model_name})',
                fontsize=15, fontweight='bold', pad=15)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(cohorts, fontsize=13, fontweight='bold')
    ax.set_ylim(y_min, y_max)
    ax.grid(axis='y', alpha=0.3, linestyle=':', linewidth=1.0)
    ax.set_axisbelow(True)
    ax.legend(loc='lower right', frameon=True, framealpha=0.95,
             fontsize=11, edgecolor='black', fancybox=False)

    plt.tight_layout()

    return fig


# Results table creation
results_list = []
for model_name, model_results in results.items():
    for key, values in model_results.items():
        cohort, condition = key.split('_', 1)
        delta_auc = 0.0
        if condition == 'Excluded':
            original_auc = results[model_name][f'{cohort}_Original']['AUC']
            delta_auc = values['AUC'] - original_auc

        results_list.append({
            'Model': model_name,
            'Cohort': cohort,
            'Condition': condition,
            'AUC': values['AUC'],
            'CI_Lower': values['CI_Lower'],
            'CI_Upper': values['CI_Upper'],
            'N': values['N'],
            'N_removed': values.get('N_removed', 0),
            'ΔAUC': delta_auc
        })

df_results = pd.DataFrame(results_list)

with pd.ExcelWriter('S19_Table_source_Outlier_GB.xlsx', engine='openpyxl') as writer:
    df_results.to_excel(writer, sheet_name='All Results', index=False)

    df_xgb = df_results[df_results['Model'] == 'XGBoost Standard']
    df_xgb.to_excel(writer, sheet_name='XGBoost Standard', index=False)

    df_hist = df_results[df_results['Model'] == 'HistGradientBoosting']
    df_hist.to_excel(writer, sheet_name='HistGradientBoosting', index=False)

    df_cat = df_results[df_results['Model'] == 'CatBoost Ordered']
    df_cat.to_excel(writer, sheet_name='CatBoost Ordered', index=False)

    summary_list = []
    for model_name in ['XGBoost Standard', 'HistGradientBoosting', 'CatBoost Ordered']:
        train_original = results[model_name]['Training_Original']['AUC']
        train_excluded = results[model_name]['Training_Excluded']['AUC']
        val_original = results[model_name]['Validation_Original']['AUC']
        val_excluded = results[model_name]['Validation_Excluded']['AUC']

        summary_list.append({
            'Model': model_name,
            'Training_ΔAUC': train_excluded - train_original,
            'Training_N_removed': results[model_name]['Training_Excluded']['N_removed'],
            'Validation_ΔAUC': val_excluded - val_original,
            'Validation_N_removed': results[model_name]['Validation_Excluded']['N_removed']
        })
    df_summary = pd.DataFrame(summary_list)
    df_summary.to_excel(writer, sheet_name='ΔAUC Summary', index=False)

print("\n✓ Results saved: S19_Table_source_Outlier_GB.xlsx")

# Display final summary
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print("\n" + df_results.to_string(index=False))

files.download('S19_Table_source_Outlier_GB.xlsx')


print("Section 22 complete.")
# [Reproducibility] Training outliers = 30, Validation outliers = 56
# [Reproducibility] XGBoost Standard Training AUC ~ 0.940
# [Reproducibility] HistGradientBoosting Training AUC ~ 0.983
# [Reproducibility] CatBoost Ordered Training AUC ~0.995 (WITHOUT eval_set; leaked run was 0.983)

## Section 21 · Outlier results for MLRA, and Fig 7 / S18 Table

**MLRA outlier influence cell** — MLRA AUC before and after outlier removal, computed with the same outlier masks and bootstrap procedure as Section 22. Output: `S19_Table_source_Outlier_MLRA.xlsx` (S19 Table, MLRA rows).

**Fig 7 cell** — Panels A–E: sensitivity, specificity, F1-score, PPV, and NPV of MLRA and XGBoost Standard at thresholds 0.05–0.70 (validation cohort); Panel F: AUC before and after outlier removal; Panel G: overall and sex-stratified AUC (unified cohort, n = 1,066). All values are computed in this notebook.

Outputs: `Fig7_Sensitivity_Analyses_MLRA_vs_XGBoost.png` (Fig 7), `S18_Table_Threshold_Metrics_and_Sex_AUC.xlsx` (S18 Table; the sex-AUC sheet is the source of Fig 7G).

In [ ]:
# ============================================================
# MLRA outlier influence analysis (computed; replaces hard-coded values)
#   Same outlier masks (train_outliers / val_outliers) and the same
#   bootstrap_auc_ci() (500 iterations, seed 42) as XGBoost in Section 21.
#   Original  = MLRA complete cases (664 / 1,066)
#   Excluded  = complete cases after sequential ±3 SD removal (634 / 1,010)
# Outputs: inline display + MLRA_outlier_analysis.xlsx (downloaded)
# ============================================================
import numpy as np, pandas as pd
from google.colab import files

rows_mlra_out = []
for coh, df_c, y_c, mask_out in [('Development', df_tr22, y_tr22, train_outliers),
                                 ('Validation',  df_va22, y_va22, val_outliers)]:
    y_arr   = np.asarray(y_c).astype(int)
    p_arr   = np.asarray(apply_mlra_formula(df_c), dtype=float)
    cc_mask = df_c[mlra_features].notna().all(axis=1).to_numpy()
    ex_mask = ~np.asarray(mask_out)
    assert not np.isnan(p_arr[cc_mask]).any(), f"{coh}: NaN prediction in complete cases — HALT"
    res = {}
    for cond, m in [('Original', cc_mask), ('Excluded', ex_mask)]:
        auc_m, lo, hi = bootstrap_auc_ci(y_arr[m], p_arr[m], n_bootstrap=500, random_state=42)
        res[cond] = auc_m
        rows_mlra_out.append({'Model': 'MLRA', 'Cohort': coh, 'Condition': cond,
                              'AUC_bootstrap_mean': auc_m, 'CI_Lower': lo, 'CI_Upper': hi,
                              'N': int(m.sum()), 'Events': int(y_arr[m].sum())})
    rows_mlra_out[-1]['dAUC'] = res['Excluded'] - res['Original']

mlra_out_df = pd.DataFrame(rows_mlra_out)
pd.set_option('display.width', 200)
print(mlra_out_df.round(4).to_string(index=False))

out_x = 'S19_Table_source_Outlier_MLRA.xlsx'
mlra_out_df.to_excel(out_x, index=False)
files.download(out_x)

In [ ]:
# ============================================================
# Section 21 — Fig 7: Sensitivity Analyses (MLRA vs XGBoost Standard)
#
# Panel layout (3 rows × 3 columns):
#   Row 1: [A] Sensitivity  | [B] Specificity    | [C] F1-Score
#   Row 2: [D] PPV          | [E] NPV             | [G] Sex-Stratified AUC
#   Row 3: [F] Outlier Influence Analysis (full width)
#
# Data sources (all computed in this notebook; no fixed values):
#   Threshold metrics : mlra_results, all_results (Sections 5, 7)
#   Outlier AUC       : results['XGBoost Standard'] (Section 22) and mlra_out_df
#   Sex-Stratified AUC: df_fairness and all_sex_metrics (Section 20)
#
# Outputs:
#   Fig7_Sensitivity_Analyses_MLRA_vs_XGBoost.png      (600 DPI; Fig 7)
#   S18_Table_Threshold_Metrics_and_Sex_AUC.xlsx       (S18 Table; sheet 3 = Fig 7G source)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.image as _mpimg_s7
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix as _cm_s7, f1_score as _f1_s7
from IPython.display import display as _disp_s7
from google.colab import files as _files_s7

# ── Diagnostic: Check prerequisite variables ──────────────────────────────
_diag_vars = ['mlra_results', 'all_results', 'ml_results',
              'train_df', 'val_df', 'df_fairness']
print('=== Variable Availability Check ===')
for _v in _diag_vars:
    try:
        _obj = eval(_v)
        if hasattr(_obj, '__len__'):
            print(f'  ✓ {_v}: type={type(_obj).__name__}, len={len(_obj)}')
        else:
            print(f'  ✓ {_v}: type={type(_obj).__name__}')
    except NameError:
        print(f'  ❌ {_v}: NOT DEFINED')
print()

# ── Publication matplotlib settings ──────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 600, 'savefig.dpi': 600,
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'axes.linewidth': 1.2, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'lines.linewidth': 1.8, 'lines.markersize': 6,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.12,
})

_COL_MLRA_S7 = '#2B2B2B'
_COL_XGB_S7  = '#2F6FAE'

_THRESHOLDS_S7 = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]
_FIG_S7_PATH   = 'Fig7_Sensitivity_Analyses_MLRA_vs_XGBoost.png'
_TBL_S7_PATH   = 'S18_Table_Threshold_Metrics_and_Sex_AUC.xlsx'

print('=' * 70)
print('Fig 7: Sensitivity + Sex-Stratified AUC Analyses')
print('MLRA vs XGBoost Standard — External Validation Cohort')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1: Retrieve validation probabilities
# ═══════════════════════════════════════════════════════════════════════════
_mlra_s7_true  = np.asarray(mlra_results.get('val_true',  []))
_mlra_s7_proba = np.asarray(mlra_results.get('val_proba', []))

_xgb_s7_res = {}
try:
    _xgb_s7_res = all_results.get('XGBoost_Standard', {})
    if not _xgb_s7_res:
        _xgb_s7_res = all_results.get('XGBoost Standard', {})
except NameError:
    try:
        _xgb_s7_res = ml_results.get('XGBoost_Standard', {})
        if not _xgb_s7_res:
            _xgb_s7_res = ml_results.get('XGBoost Standard', {})
    except NameError:
        print('  ❌ all_results / ml_results not found — run Section 7 first.')

_xgb_s7_true  = np.asarray(_xgb_s7_res.get('val_true',  []))
_xgb_s7_proba = np.asarray(_xgb_s7_res.get('val_proba', []))

_prereq_ok = True
for _nm, _yt, _yp in [('MLRA',             _mlra_s7_true, _mlra_s7_proba),
                       ('XGBoost_Standard', _xgb_s7_true,  _xgb_s7_proba)]:
    if len(_yt) == 0 or len(_yp) == 0:
        print(f'  ❌ {_nm}: val_true/val_proba missing — run Sections 5 & 7 first.')
        _prereq_ok = False
    else:
        print(f'  ✓ {_nm}: N={len(_yt)}, events={int(_yt.sum())}')

if not _prereq_ok:
    raise RuntimeError('Prerequisites missing — run Sections 5 and 7 first.')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2: Threshold metrics (full range 0.05–0.70)
# ═══════════════════════════════════════════════════════════════════════════
def _threshold_metrics_s7(y_true, y_proba, thresholds):
    """Compute Sensitivity, Specificity, PPV, NPV, F1 at each threshold."""
    out = {m: [] for m in ['Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1']}
    for thr in thresholds:
        yhat = (y_proba >= thr).astype(int)
        if yhat.sum() == 0 or yhat.sum() == len(yhat):
            for m in out: out[m].append(np.nan)
            continue
        cm = _cm_s7(y_true, yhat)
        if cm.shape != (2, 2):
            for m in out: out[m].append(np.nan)
            continue
        tn, fp, fn, tp = cm.ravel()
        out['Sensitivity'].append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        out['Specificity'].append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
        out['PPV'].append(tp / (tp + fp) if (tp + fp) > 0 else np.nan)
        out['NPV'].append(tn / (tn + fn) if (tn + fn) > 0 else np.nan)
        out['F1'].append(_f1_s7(y_true, yhat, zero_division=0))
    return out

_mlra_m = _threshold_metrics_s7(_mlra_s7_true, _mlra_s7_proba, _THRESHOLDS_S7)
_xgb_m  = _threshold_metrics_s7(_xgb_s7_true,  _xgb_s7_proba,  _THRESHOLDS_S7)
print(f'  ✓ Threshold metrics computed ({len(_THRESHOLDS_S7)} thresholds × 5 metrics)')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 3: Outlier influence data
#   Primary : results['XGBoost Standard'] from Section 21
#   MLRA    : mlra_out_df (MLRA outlier influence cell)
# ═══════════════════════════════════════════════════════════════════════════
# [REPLACED 2026-10-24] No hard-coded values: both models are read from
#   computed results (XGBoost: Section 22 'results'; MLRA: 'mlra_out_df').
assert 'results' in dir() and 'XGBoost Standard' in results, \
    "Run Section 22 (Outlier Influence Analysis) first — HALT"
assert 'mlra_out_df' in dir(), "Run the MLRA outlier influence cell first — HALT"

_OUTLIER_S7 = {'MLRA': {}, 'XGBoost Standard': {}}
_NREM = {'Development': int(n_tr22_out), 'Validation': int(n_va22_out)}
for _ds, _ok, _ek in [('Development', 'Training_Original',   'Training_Excluded'),
                      ('Validation',  'Validation_Original', 'Validation_Excluded')]:
    _o, _e = results['XGBoost Standard'][_ok], results['XGBoost Standard'][_ek]
    _OUTLIER_S7['XGBoost Standard'][_ds] = {
        'orig_auc': _o['AUC'], 'orig_lo': _o['CI_Lower'], 'orig_hi': _o['CI_Upper'],
        'excl_auc': _e['AUC'], 'excl_lo': _e['CI_Lower'], 'excl_hi': _e['CI_Upper'],
        'n_removed': _NREM[_ds], 'delta': _e['AUC'] - _o['AUC']}
    _m = mlra_out_df[mlra_out_df['Cohort'] == _ds].set_index('Condition')
    _OUTLIER_S7['MLRA'][_ds] = {
        'orig_auc': _m.loc['Original', 'AUC_bootstrap_mean'],
        'orig_lo':  _m.loc['Original', 'CI_Lower'], 'orig_hi': _m.loc['Original', 'CI_Upper'],
        'excl_auc': _m.loc['Excluded', 'AUC_bootstrap_mean'],
        'excl_lo':  _m.loc['Excluded', 'CI_Lower'], 'excl_hi': _m.loc['Excluded', 'CI_Upper'],
        'n_removed': int(_m.loc['Original', 'N'] - _m.loc['Excluded', 'N']),
        'delta': _m.loc['Excluded', 'AUC_bootstrap_mean'] - _m.loc['Original', 'AUC_bootstrap_mean']}
    assert _OUTLIER_S7['MLRA'][_ds]['n_removed'] == _NREM[_ds], f"{_ds}: outlier counts differ — HALT"

print('  ✓ Outlier AUC loaded from computed results (MLRA and XGBoost Standard)')
_out_tab = pd.DataFrame([{'Model': _mdl, 'Cohort': _ds, **_v}
                         for _mdl, _d in _OUTLIER_S7.items() for _ds, _v in _d.items()])
print(_out_tab.round(6).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════
# STEP 4: Sex-stratified AUC data for Panel G
#   Primary : df_fairness (Section 20)
#   (no fallback values)
#   [FIXED] Fairness_Score removed; PPV_Diff / Equal_Opportunity updated to
#           Youden-threshold values; AUC_Male / AUC_Female added for bar chart
# ═══════════════════════════════════════════════════════════════════════════
# [REPLACED 2026-10-24] Fallback values removed; df_fairness (Section 20) is required.
assert 'df_fairness' in dir() and df_fairness is not None and len(df_fairness) > 0, \
    "Run Section 20 (Extended Fairness Analysis) first — HALT"
_df_fair = df_fairness.copy()
# Sex-specific AUCs are not stored in df_fairness; take them from the per-sex
# metrics computed in Section 20 (all_sex_metrics: index = sex code, column 'AUC').
assert 'all_sex_metrics' in dir(), "all_sex_metrics (Section 20) not found — HALT"
def _sex_auc(metrics_df, which):
    idx = list(metrics_df.index)
    keys = {'Male': [1, 1.0, '1', 'M', 'Male'], 'Female': [0, 0.0, '0', 'F', 'Female']}[which]
    hit = [k for k in idx if k in keys]
    assert len(hit) == 1, f"Cannot identify {which} row in {idx} — HALT"
    return float(metrics_df.loc[hit[0], 'AUC'])
for _col, _which in [('AUC_Male', 'Male'), ('AUC_Female', 'Female')]:
    _df_fair[_col] = _df_fair['Model'].map(
        {m: _sex_auc(all_sex_metrics[m], _which) for m in all_sex_metrics})
assert _df_fair[['AUC_Male', 'AUC_Female']].notna().all().all(), "Sex-specific AUC missing — HALT"
# Definition check: |AUC_Female - AUC_Male| must equal AUC_Diff_Sex from Section 20
assert np.allclose((_df_fair['AUC_Female'] - _df_fair['AUC_Male']).abs(),
                   _df_fair['AUC_Diff_Sex'], atol=1e-9), "AUC_Diff_Sex inconsistent — HALT"
print(f'  ✓ df_fairness loaded from Section 20 (N={len(_df_fair)} models); '
      f'sex-specific AUCs from all_sex_metrics')
print(_df_fair[['Model', 'Overall_AUC', 'AUC_Male', 'AUC_Female', 'AUC_Diff_Sex']].round(4).to_string(index=False))

_df_fair = _df_fair[
    _df_fair['Model'].isin(['MLRA', 'XGBoost Standard'])
].reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 5: Build Fig 7
#   Row 1: [A] Sensitivity | [B] Specificity    | [C] F1-Score
#   Row 2: [D] PPV         | [E] NPV             | [G] Sex-Stratified AUC
#   Row 3: [F] Outlier Influence Analysis (full width)
# ═══════════════════════════════════════════════════════════════════════════
fig_s7 = plt.figure(figsize=(17, 15))
gs_s7  = gridspec.GridSpec(
    3, 3, figure=fig_s7,
    hspace=0.52, wspace=0.34,
    height_ratios=[1, 1, 1.15],
)
ax_A = fig_s7.add_subplot(gs_s7[0, 0])
ax_B = fig_s7.add_subplot(gs_s7[0, 1])
ax_C = fig_s7.add_subplot(gs_s7[0, 2])
ax_D = fig_s7.add_subplot(gs_s7[1, 0])
ax_E = fig_s7.add_subplot(gs_s7[1, 1])
ax_G = fig_s7.add_subplot(gs_s7[1, 2])
ax_F = fig_s7.add_subplot(gs_s7[2, :])

# ── Threshold line-panel helper ───────────────────────────────────────────
def _line_panel(ax, panel_label, title, mlra_vals, xgb_vals, ylabel, thresholds):
    ax.plot(thresholds, mlra_vals, 's-', color=_COL_MLRA_S7,
            label='MLRA', linewidth=1.8, markersize=5)
    ax.plot(thresholds, xgb_vals, 'D-', color=_COL_XGB_S7,
            label='XGBoost Standard', linewidth=1.8, markersize=5)
    ax.set_xlim(thresholds[0] - 0.025, thresholds[-1] + 0.025)
    ax.set_ylim(-0.02, 1.06)
    ax.set_xticks([t for t in thresholds
                   if t in [0.05, 0.10, 0.20, 0.30, 0.50, 0.70]])
    ax.set_xlabel('Probability Threshold', fontsize=9.5, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=9.5, fontweight='bold')
    ax.set_title(f'Panel {panel_label} — {title}',
                 fontsize=10.5, fontweight='bold', pad=7)
    ax.legend(fontsize=8.5, frameon=True, framealpha=0.88, loc='best')
    ax.spines[['top', 'right']].set_visible(False)
    ax.axvspan(thresholds[0] - 0.01, 0.305,
               alpha=0.06, color='#DDDDDD', zorder=0)
    ax.axvline(0.30, color='#BBBBBB', linewidth=0.8, linestyle=':', zorder=1)

_line_panel(ax_A, 'A', 'Sensitivity',
            _mlra_m['Sensitivity'], _xgb_m['Sensitivity'],
            'Sensitivity', _THRESHOLDS_S7)
_line_panel(ax_B, 'B', 'Specificity',
            _mlra_m['Specificity'], _xgb_m['Specificity'],
            'Specificity', _THRESHOLDS_S7)
_line_panel(ax_C, 'C', 'F1-Score',
            _mlra_m['F1'], _xgb_m['F1'],
            'F1-Score', _THRESHOLDS_S7)
_line_panel(ax_D, 'D', 'Positive Predictive Value',
            _mlra_m['PPV'], _xgb_m['PPV'],
            'PPV', _THRESHOLDS_S7)
_line_panel(ax_E, 'E', 'Negative Predictive Value',
            _mlra_m['NPV'], _xgb_m['NPV'],
            'NPV', _THRESHOLDS_S7)

# ── Panel G: Sex-Stratified AUC Bar Chart ────────────────────────────────
_models_g    = ['MLRA', 'XGBoost Standard']
_labels_g    = ['MLRA', 'XGBoost\nStandard']
_x_g         = np.arange(len(_models_g))
_BAR_W_G     = 0.22
_col_male    = '#4472C4'
_col_female  = '#ED7D31'
_col_overall = '#595959'

_auc_male, _auc_female, _auc_overall = [], [], []
for _mdl in _models_g:
    _row = _df_fair[_df_fair['Model'] == _mdl]
    if _row.empty:
        _auc_male.append(np.nan); _auc_female.append(np.nan); _auc_overall.append(np.nan)
    else:
        _auc_male.append(float(_row.iloc[0].get('AUC_Male',    np.nan)))
        _auc_female.append(float(_row.iloc[0].get('AUC_Female', np.nan)))
        _auc_overall.append(float(_row.iloc[0].get('Overall_AUC', np.nan)))

ax_G.bar(_x_g - _BAR_W_G, _auc_male,    _BAR_W_G, color=_col_male,
         edgecolor='#333', linewidth=0.8, label='Male',    zorder=3)
ax_G.bar(_x_g,             _auc_overall, _BAR_W_G, color=_col_overall,
         edgecolor='#333', linewidth=0.8, label='Overall', zorder=3)
ax_G.bar(_x_g + _BAR_W_G, _auc_female,  _BAR_W_G, color=_col_female,
         edgecolor='#333', linewidth=0.8, label='Female',  zorder=3)

for _xi, (_vm, _vo, _vf) in enumerate(zip(_auc_male, _auc_overall, _auc_female)):
    for _xoff, _val in [(-_BAR_W_G, _vm), (0, _vo), (_BAR_W_G, _vf)]:
        if not np.isnan(_val):
            ax_G.text(_xi + _xoff, _val + 0.004, f'{_val:.3f}',
                      ha='center', va='bottom', fontsize=7.5, color='#111')

for _xi, _mdl in enumerate(_models_g):
    _row = _df_fair[_df_fair['Model'] == _mdl]
    if not _row.empty:
        _diff = float(_row.iloc[0].get('AUC_Diff_Sex', np.nan))
        if not np.isnan(_diff):
            _ymax = max(v for v in [_auc_male[_xi], _auc_female[_xi], _auc_overall[_xi]]
                        if not np.isnan(v))
            ax_G.text(_xi, _ymax + 0.022, f'|ΔAUC|={_diff:.4f}',
                      ha='center', va='bottom', fontsize=7.5,
                      color='#111', style='italic')

ax_G.set_xticks(_x_g)
ax_G.set_xticklabels(_labels_g, fontsize=9.5, fontweight='bold')
ax_G.set_ylim(0.82, 1.02)
ax_G.set_ylabel('AUC', fontsize=9.5, fontweight='bold')
ax_G.set_title('Panel G — Overall and Sex-Stratified AUC',
               fontsize=10.5, fontweight='bold', pad=7)
ax_G.legend(fontsize=8.0, frameon=True, framealpha=0.88,
            loc='lower right', ncol=1)
ax_G.spines[['top', 'right']].set_visible(False)

# ── Panel F: Outlier Influence Analysis (full-width) ─────────────────────
_BAR_W        = 0.28
_COLOR_MLRA_O = '#BEBEBE'; _COLOR_MLRA_E = '#696969'
_COLOR_XGB_O  = '#AEC6E8'; _COLOR_XGB_E  = '#2F6FAE'
_X = {'MLRA_Dev': 0.65, 'XGB_Dev': 1.40, 'MLRA_Val': 2.70, 'XGB_Val': 3.45}

for _mk, _dk, _xc in [
    ('MLRA',             'Development', _X['MLRA_Dev']),
    ('XGBoost Standard', 'Development', _X['XGB_Dev']),
    ('MLRA',             'Validation',  _X['MLRA_Val']),
    ('XGBoost Standard', 'Validation',  _X['XGB_Val']),
]:
    _d  = _OUTLIER_S7[_mk][_dk]
    _co = _COLOR_MLRA_O if 'MLRA' in _mk else _COLOR_XGB_O
    _ce = _COLOR_MLRA_E if 'MLRA' in _mk else _COLOR_XGB_E
    _xo, _xe = _xc - _BAR_W / 2, _xc + _BAR_W / 2
    ax_F.bar(_xo, _d['orig_auc'], _BAR_W, color=_co,
             edgecolor='#444', linewidth=0.8, zorder=3)
    ax_F.errorbar(_xo, _d['orig_auc'],
                  yerr=[[_d['orig_auc'] - _d['orig_lo']],
                        [_d['orig_hi'] - _d['orig_auc']]],
                  fmt='none', color='#222', capsize=4, linewidth=1.2, zorder=4)
    ax_F.bar(_xe, _d['excl_auc'], _BAR_W, color=_ce,
             edgecolor='#444', linewidth=0.8, zorder=3)
    ax_F.errorbar(_xe, _d['excl_auc'],
                  yerr=[[_d['excl_auc'] - _d['excl_lo']],
                        [_d['excl_hi'] - _d['excl_auc']]],
                  fmt='none', color='#222', capsize=4, linewidth=1.2, zorder=4)
    ax_F.text(_xc, max(_d['orig_hi'], _d['excl_hi']) + 0.013,
              f"N removal={_d['n_removed']}\nΔAUC={_d['delta']:+.3f}",
              ha='center', va='bottom', fontsize=7.5, color='#111',
              linespacing=1.4)

ax_F.set_xticks([(_X['MLRA_Dev'] + _X['XGB_Dev']) / 2,
                  (_X['MLRA_Val'] + _X['XGB_Val']) / 2])
ax_F.set_xticklabels(['Development', 'Validation'],
                      fontsize=11, fontweight='bold')
ax_F.set_xlim(0.25, 3.90)
ax_F.set_ylim(0.50, 1.04)
ax_F.set_ylabel('AUC', fontsize=10.5, fontweight='bold')
ax_F.set_title('Panel F — Outlier Influence Analysis (Model Comparison)',
               fontsize=10.5, fontweight='bold', pad=9)
ax_F.spines[['top', 'right']].set_visible(False)
ax_F.axvline(2.05, color='#aaa', linestyle='--', linewidth=0.8, alpha=0.6)
ax_F.legend(
    handles=[
        Patch(facecolor=_COLOR_MLRA_O, edgecolor='#444', label='MLRA Original'),
        Patch(facecolor=_COLOR_MLRA_E, edgecolor='#444', label='MLRA After outlier removal'),
        Patch(facecolor=_COLOR_XGB_O,  edgecolor='#444', label='XGBoost Standard Original'),
        Patch(facecolor=_COLOR_XGB_E,  edgecolor='#444', label='XGBoost Standard After outlier removal'),
    ],
    loc='lower right', fontsize=8.5, frameon=True, framealpha=0.88, ncol=2,
)

# [REMOVED 2026-10-24] Figure title deleted: the figure is Fig 7 in the manuscript,
#   and PLOS ONE figure files should not contain titles (the legend carries them).

# ── Save 600 DPI ──────────────────────────────────────────────────────────
plt.savefig(_FIG_S7_PATH, dpi=600, bbox_inches='tight',
            facecolor='white', format='png')
print(f'\n✓ Saved: {_FIG_S7_PATH} (600 DPI, publication quality)')
plt.close(fig_s7)

# ── Inline preview at 120 DPI (avoids Colab "Output hidden" at 600 DPI) ──
print('\n[Fig 7 — inline preview (120 DPI display)]')
_fig_inline, _ax_inline = plt.subplots(figsize=(14, 12), dpi=120)
_img_data = _mpimg_s7.imread(_FIG_S7_PATH)
_ax_inline.imshow(_img_data)
_ax_inline.axis('off')
_fig_inline.tight_layout(pad=0)
plt.show()
plt.close(_fig_inline)
del _img_data

_files_s7.download(_FIG_S7_PATH)

# ═══════════════════════════════════════════════════════════════════════════
# TABLE S7: Threshold Sensitivity + Sex-Stratified AUC Metrics
#   Sheet S7A_Development : threshold metrics, development cohort
#   Sheet S7B_Validation  : threshold metrics, validation cohort
#   Sheet S7_Sex_AUC      : sex-stratified AUC and fairness metrics
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('S18 Table: Threshold Sensitivity + Sex-Stratified AUC Metrics')
print('=' * 70)

# ── Retrieve development (train) probabilities ────────────────────────────
_mlra_tr_true  = np.asarray(mlra_results.get('train_true',  []))
_mlra_tr_proba = np.asarray(mlra_results.get('train_proba', []))

_xgb_tr_res = {}
try:
    _xgb_tr_res = all_results.get('XGBoost_Standard', {})
    if not _xgb_tr_res:
        _xgb_tr_res = all_results.get('XGBoost Standard', {})
except NameError:
    try:
        _xgb_tr_res = ml_results.get('XGBoost_Standard', {})
        if not _xgb_tr_res:
            _xgb_tr_res = ml_results.get('XGBoost Standard', {})
    except NameError:
        pass

_xgb_tr_true  = np.asarray(_xgb_tr_res.get('train_true',  []))
_xgb_tr_proba = np.asarray(_xgb_tr_res.get('train_proba', []))

_mlra_m_dev = _threshold_metrics_s7(_mlra_tr_true, _mlra_tr_proba, _THRESHOLDS_S7)
_xgb_m_dev  = _threshold_metrics_s7(_xgb_tr_true,  _xgb_tr_proba,  _THRESHOLDS_S7)

for _nm, _yt in [('MLRA Dev', _mlra_tr_true), ('XGBoost Dev', _xgb_tr_true)]:
    if len(_yt) > 0:
        print(f'  ✓ {_nm}: N={len(_yt)}, events={int(_yt.sum())}')
    else:
        print(f'  ⚠  {_nm}: empty — Development metrics will be N/A')

def _fv(v):
    try:
        return f'{float(v):.3f}' if not np.isnan(float(v)) else 'N/A'
    except Exception:
        return 'N/A'

def _acc(y_true, y_proba, thr):
    if len(y_true) == 0 or len(y_proba) == 0:
        return np.nan
    yhat = (y_proba >= thr).astype(int)
    return float(np.mean(yhat == y_true))

# Build threshold table (Development + Validation)
_tbl_thresh = []
for _cohort_lbl, _mm_mlra, _mm_xgb, _yt_mlra, _yp_mlra, _yt_xgb, _yp_xgb in [
    ('Development',
     _mlra_m_dev, _xgb_m_dev,
     _mlra_tr_true, _mlra_tr_proba,
     _xgb_tr_true,  _xgb_tr_proba),
    ('Validation',
     _mlra_m, _xgb_m,
     _mlra_s7_true, _mlra_s7_proba,
     _xgb_s7_true,  _xgb_s7_proba),
]:
    for _i, _thr in enumerate(_THRESHOLDS_S7):
        for _mn, _mm, _yt, _yp in [
            ('MLRA',             _mm_mlra, _yt_mlra, _yp_mlra),
            ('XGBoost Standard', _mm_xgb,  _yt_xgb,  _yp_xgb),
        ]:
            _tbl_thresh.append({
                'Cohort':      _cohort_lbl,
                'Model':       _mn,
                'Cutoff':      f'{_thr:.2f}',
                'Accuracy':    _fv(_acc(_yt, _yp, _thr)),
                'Sensitivity': _fv(_mm['Sensitivity'][_i]),
                'Specificity': _fv(_mm['Specificity'][_i]),
                'PPV':         _fv(_mm['PPV'][_i]),
                'NPV':         _fv(_mm['NPV'][_i]),
                'F1':          _fv(_mm['F1'][_i]),
            })

df_thresh_s7 = pd.DataFrame(_tbl_thresh, columns=[
    'Cohort', 'Model', 'Cutoff',
    'Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1',
])


# Sheet 2: Sex-Stratified AUC Metrics
_fair_cols = ['Model', 'Overall_AUC',
              'AUC_Male', 'AUC_Female', 'AUC_Diff_Sex',
              'Equal_Opportunity', 'Demographic_Parity',
              'PPV_Diff', 'NPV_Diff']
df_fair_tbl = _df_fair[[c for c in _fair_cols if c in _df_fair.columns]].copy()
for _fc in [c for c in _fair_cols if c not in df_fair_tbl.columns]:
    df_fair_tbl[_fc] = np.nan
df_fair_tbl = df_fair_tbl[_fair_cols]
for _fc in ['Overall_AUC', 'AUC_Male', 'AUC_Female', 'AUC_Diff_Sex',
            'Equal_Opportunity', 'Demographic_Parity', 'PPV_Diff', 'NPV_Diff']:
    df_fair_tbl[_fc] = df_fair_tbl[_fc].apply(
        lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')

# ── Styled inline display ─────────────────────────────────────────────────
def _style_s7t(df, caption):
    return (
        df.style
        .set_table_styles([
            {'selector': 'thead th',
             'props': [('background-color', '#2E4057'), ('color', 'white'),
                       ('font-weight', 'bold'), ('text-align', 'center'),
                       ('padding', '7px 10px')]},
            {'selector': 'tbody td',
             'props': [('padding', '5px 10px'), ('text-align', 'center'),
                       ('border-bottom', '1px solid #ddd')]},
            {'selector': 'tbody tr:nth-child(even)',
             'props': [('background-color', '#f5f8fc')]},
        ])
        .set_caption(caption)
        .hide(axis='index')
    )

print('\n[Sheet 1A — Development Cohort]')
_disp_s7(_style_s7t(
    df_thresh_s7[df_thresh_s7['Cohort'] == 'Development']
    .drop(columns='Cohort').reset_index(drop=True),
    'S18 Table Sheet 1A. Development Cohort — Threshold sensitivity (0.05–0.70). '
    'Columns: Model | Cutoff | Accuracy | Sensitivity | Specificity | PPV | NPV | F1.',
))

print('\n[Sheet 1B — Validation Cohort]')
_disp_s7(_style_s7t(
    df_thresh_s7[df_thresh_s7['Cohort'] == 'Validation']
    .drop(columns='Cohort').reset_index(drop=True),
    'S18 Table Sheet 1B. Validation Cohort — Threshold sensitivity (0.05–0.70). '
    'Columns: Model | Cutoff | Accuracy | Sensitivity | Specificity | PPV | NPV | F1.',
))

print('\n[Sheet 2 — Sex-Stratified AUC and Fairness Metrics]')
print('Threshold basis: Youden index (Section 15, N=1,066 aligned cohort).')
print('Equal_Opportunity = |Sensitivity_Male − Sensitivity_Female|; '
      'Demographic_Parity = |Positive_Rate_Male − Positive_Rate_Female|.')
_disp_s7(_style_s7t(
    df_fair_tbl,
    'S18 Table Sheet 2. Sex-stratified AUC and fairness metrics — Youden-threshold basis. '
    'AUC_Male / AUC_Female: from Section 20 per-sex metrics (all_sex_metrics).',
))

# ── Export to Excel (3 sheets) + download ────────────────────────────────
with pd.ExcelWriter(_TBL_S7_PATH, engine='openpyxl') as _wtr:
    (df_thresh_s7[df_thresh_s7['Cohort'] == 'Development']
     .drop(columns='Cohort').reset_index(drop=True)
     .to_excel(_wtr, sheet_name='S7A_Development', index=False))
    (df_thresh_s7[df_thresh_s7['Cohort'] == 'Validation']
     .drop(columns='Cohort').reset_index(drop=True)
     .to_excel(_wtr, sheet_name='S7B_Validation', index=False))
    df_fair_tbl.to_excel(_wtr, sheet_name='S7_Sex_AUC', index=False)
    for _sn in ['S7A_Development', 'S7B_Validation', 'S7_Sex_AUC']:
        _ws = _wtr.sheets[_sn]
        for _col in _ws.columns:
            _ws.column_dimensions[_col[0].column_letter].width = (
                max(len(str(_c.value or '')) for _c in _col) + 4)

print(f'\n✓ Saved: {_TBL_S7_PATH} '
      '(3 sheets: S7A_Development, S7B_Validation, S7_Sex_AUC)')
_files_s7.download(_TBL_S7_PATH)
print(f'✓ Downloads: {_FIG_S7_PATH}, {_TBL_S7_PATH}')

# ── Verification ──────────────────────────────────────────────────────────
# [Reproducibility] Figure: 600 DPI saved; 120 DPI shown inline (VS Code + Colab)
# [Reproducibility] Tables: styled display(df) → text/html (VS Code + Colab)
# [Reproducibility] Threshold range 0.05–0.70; MLRA N≈1,066; XGBoost N≈1,243
# [Reproducibility] Outlier AUC: computed in this notebook (Section 22 and the
#   MLRA outlier influence cell); no hard-coded values.
# [Reproducibility] Fairness: Youden-threshold values (Section 15, N=1,066)
#   MLRA  EO=0.0174 PPV_Diff=0.0040; XGBoost EO=0.0243 PPV_Diff=0.0183
# [NOTE] No model retraining; all metrics derived from precomputed probabilities.

## **Section 23 · IDI & NRI — XGBoost Standard vs Comparator Models (S15 Table, Fig 5)**

In [ ]:
# =============================================================================
# SECTION 8-F: Integrated Discrimination Improvement (IDI) &
#              Net Reclassification Improvement (NRI) — XGBoost Standard vs comparators
#              Reference model: XGBoost Standard
#              Fig 5: Five-panel vertical layout, 600 DPI
# =============================================================================

# **Cell B — IDI & NRI: Statistical Functions**

In [ ]:
import string
from scipy import stats

# ─────────────────────────────────────────────────────────────────────────────
# IDI Calculation
# Integrated Discrimination Improvement between a new and a reference model.
# Formula: IDI = (mean_new_events - mean_old_events)
#              - (mean_new_non_events - mean_old_non_events)
# Reference: Pencina et al., Stat Med 2008; 27(2):157-172
# ─────────────────────────────────────────────────────────────────────────────
def calculate_idi(y_true, prob_new, prob_old):
    """
    Compute IDI, standard error, z-score, p-value, and 95% CI.

    Parameters
    ----------
    y_true   : array-like, binary outcome (1=event, 0=non-event)
    prob_new : array-like, predicted probabilities from the NEW model
    prob_old : array-like, predicted probabilities from the REFERENCE model

    Returns
    -------
    dict with keys: IDI, SE, Z_score, p_value, CI_lower, CI_upper,
                    improvement_events, improvement_non_events
    """
    y_true   = np.array(y_true)
    prob_new = np.array(prob_new)
    prob_old = np.array(prob_old)

    events     = y_true == 1
    non_events = y_true == 0

    improvement_events     = np.mean(prob_new[events])     - np.mean(prob_old[events])
    improvement_non_events = np.mean(prob_new[non_events]) - np.mean(prob_old[non_events])
    idi = improvement_events - improvement_non_events

    n_events     = np.sum(events)
    n_non_events = np.sum(non_events)

    diff_events     = prob_new[events]     - prob_old[events]
    diff_non_events = prob_new[non_events] - prob_old[non_events]

    var_events     = np.var(diff_events,     ddof=1) / n_events
    var_non_events = np.var(diff_non_events, ddof=1) / n_non_events

    se_idi  = np.sqrt(var_events + var_non_events)
    z_score = idi / se_idi if se_idi > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

    return {
        'IDI':                    idi,
        'SE':                     se_idi,
        'Z_score':                z_score,
        'p_value':                p_value,
        'CI_lower':               idi - 1.96 * se_idi,
        'CI_upper':               idi + 1.96 * se_idi,
        'improvement_events':     improvement_events,
        'improvement_non_events': improvement_non_events,
    }


# ─────────────────────────────────────────────────────────────────────────────
# NRI Calculation
# Categorical NRI using two risk thresholds (default: 10% and 20%).
# Reference: Pencina et al., Stat Med 2008; 27(2):157-172
# ─────────────────────────────────────────────────────────────────────────────
def calculate_nri(y_true, prob_new, prob_old, thresholds=None):
    """
    Compute categorical NRI, standard error, z-score, p-value, and 95% CI.

    Parameters
    ----------
    y_true     : array-like, binary outcome
    prob_new   : array-like, predicted probabilities from the NEW model
    prob_old   : array-like, predicted probabilities from the REFERENCE model
    thresholds : list of float, risk category boundaries (default [0.1, 0.2])

    Returns
    -------
    dict with NRI components and inference statistics
    """
    if thresholds is None:
        thresholds = [0.1, 0.2]

    y_true   = np.array(y_true)
    prob_new = np.array(prob_new)
    prob_old = np.array(prob_old)

    def categorize(probs, thresh_list):
        cats = np.zeros(len(probs), dtype=int)
        for i, t in enumerate(thresh_list):
            cats[probs >= t] = i + 1
        return cats

    cat_new = categorize(prob_new, thresholds)
    cat_old = categorize(prob_old, thresholds)

    events     = y_true == 1
    non_events = y_true == 0

    up_ev   = np.sum(cat_new[events]     > cat_old[events])
    down_ev = np.sum(cat_new[events]     < cat_old[events])
    n_ev    = np.sum(events)

    up_nev   = np.sum(cat_new[non_events] > cat_old[non_events])
    down_nev = np.sum(cat_new[non_events] < cat_old[non_events])
    n_nev    = np.sum(non_events)

    nri_events     = (up_ev   - down_ev)  / n_ev
    nri_non_events = (down_nev - up_nev)  / n_nev
    nri = nri_events + nri_non_events

    se_ev  = np.sqrt((up_ev   + down_ev)  / n_ev**2)
    se_nev = np.sqrt((up_nev  + down_nev) / n_nev**2)
    se_nri = np.sqrt(se_ev**2 + se_nev**2)

    z_score = nri / se_nri if se_nri > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

    return {
        'NRI':               nri,
        'NRI_events':        nri_events,
        'NRI_non_events':    nri_non_events,
        'SE':                se_nri,
        'Z_score':           z_score,
        'p_value':           p_value,
        'CI_lower':          nri - 1.96 * se_nri,
        'CI_upper':          nri + 1.96 * se_nri,
        'upward_events':     up_ev,
        'downward_events':   down_ev,
        'upward_non_events': up_nev,
        'downward_non_events': down_nev,
    }

print("✓ IDI / NRI functions defined.")

# **Cell C — IDI & NRI: Data Preparation (in-memory predictions, n = 1,066)**

In [ ]:
# Cell 76─────────────────────────────────────────────────────────────────────────────
# Data preparation — IDI & NRI (XGBoost Standard vs comparators)
# [REPLACED 2026-10-24] Predictions are taken directly from the in-memory models
#   on the MLRA complete-case validation subset (n = 1,066; 50 events).
#   The previous version uploaded CSV files from the local disk, which could
#   overwrite the CSVs written by Section 13 with stale predictions.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import roc_auc_score

_idx = (mlra_results['val_true'].index if hasattr(mlra_results['val_true'], 'index')
        else np.arange(len(mlra_results['val_true'])))
y_true_idi = np.asarray(mlra_results['val_true']).astype(int)
assert len(y_true_idi) == 1066 and y_true_idi.sum() == 50, "Common sample is not 1,066 / 50 — HALT"

def _common(key):
    if key == 'MLRA':
        return np.asarray(mlra_results['val_proba'], dtype=float)
    p = np.asarray(all_results[key]['val_proba'], dtype=float)
    return p[np.asarray(_idx)] if len(p) != len(y_true_idi) else p

_IDI_MODELS = {                      # publication name : internal key
    'XGBoost Standard'    : 'XGBoost_Standard',     # reference (index) model
    'MLRA'                : 'MLRA',
    'XGBoost DART'        : 'XGBoost_DART',
    'LightGBM GBDT'       : 'LightGBM_GBDT',
    'LightGBM DART'       : 'LightGBM_DART',
    'LightGBM GOSS'       : 'LightGBM_GOSS',
    'CatBoost Standard'   : 'CatBoost_Standard',
    'CatBoost Ordered'    : 'CatBoost_Ordered',
    'HistGradientBoosting': 'HistGradientBoosting',
}
model_preds_idi = {pub: _common(key) for pub, key in _IDI_MODELS.items()}

print("=" * 70)
print("IDI & NRI — XGBoost Standard vs comparators (in-memory predictions)")
print("=" * 70)
print(f"Common sample: n = {len(y_true_idi)}, events = {int(y_true_idi.sum())} "
      f"({y_true_idi.mean() * 100:.2f}%)")
for pub, p in model_preds_idi.items():
    assert len(p) == len(y_true_idi), f"{pub}: length mismatch — HALT"
    print(f"  {pub:<22} AUC (common sample) = {roc_auc_score(y_true_idi, p):.4f}")

# **Cell D — IDI & NRI: Statistical Analysis**

In [ ]:
# Cell 78─────────────────────────────────────────────────────────────────────────────
# IDI & NRI pairwise comparison: XGBoost Standard (reference) vs all others
# Risk thresholds for categorical NRI: 10% and 20%
# ─────────────────────────────────────────────────────────────────────────────
RISK_THRESHOLDS = [0.1, 0.2]   # 10% and 20% risk cut-offs

prob_ref  = model_preds_idi['XGBoost Standard']
auc_ref   = roc_auc_score(y_true_idi, prob_ref)
idi_nri_results = []

print("=" * 70)
print("Running IDI & NRI analyses ...")
print("=" * 70)

for model_name, prob_model in model_preds_idi.items():
    if model_name == 'XGBoost Standard':
        continue

    auc_model = roc_auc_score(y_true_idi, prob_model)
    delta_auc = auc_ref - auc_model   # positive = XGBoost Superior

    idi_res = calculate_idi(y_true_idi, prob_ref, prob_model)
    nri_res = calculate_nri(y_true_idi, prob_ref, prob_model,
                            thresholds=RISK_THRESHOLDS)

    idi_nri_results.append({
        'Comparison_Model': model_name,
        'AUC_XGBoost_Std':  auc_ref,
        'AUC_Model':        auc_model,
        'Delta_AUC':        delta_auc,
        'IDI':              idi_res['IDI'],
        'IDI_SE':           idi_res['SE'],
        'IDI_p_value':      idi_res['p_value'],
        'IDI_CI_lower':     idi_res['CI_lower'],
        'IDI_CI_upper':     idi_res['CI_upper'],
        'NRI':              nri_res['NRI'],
        'NRI_events':       nri_res['NRI_events'],
        'NRI_non_events':   nri_res['NRI_non_events'],
        'NRI_SE':           nri_res['SE'],
        'NRI_p_value':      nri_res['p_value'],
        'NRI_CI_lower':     nri_res['CI_lower'],
        'NRI_CI_upper':     nri_res['CI_upper'],
    })
    print(f"\n  {model_name}")
    print(f"    IDI = {idi_res['IDI']:+.4f}  (95% CI: {idi_res['CI_lower']:+.4f} – {idi_res['CI_upper']:+.4f})  p = {idi_res['p_value']:.4f}")
    print(f"    NRI = {nri_res['NRI']:+.4f}  (95% CI: {nri_res['CI_lower']:+.4f} – {nri_res['CI_upper']:+.4f})  p = {nri_res['p_value']:.4f}")
    print(f"    ΔAUC = {delta_auc:+.4f}")

# ─── Assemble results DataFrame ───────────────────────────────────────────
idi_nri_df = pd.DataFrame(idi_nri_results)
idi_nri_df = idi_nri_df.sort_values('Delta_AUC', ascending=False).reset_index(drop=True)
idi_nri_df['IDI_Significant'] = idi_nri_df['IDI_p_value'] < 0.05
idi_nri_df['NRI_Significant'] = idi_nri_df['NRI_p_value'] < 0.05
idi_nri_df['Label']           = list(string.ascii_uppercase[:len(idi_nri_df)])

print("\n" + "=" * 70)
print("Significance summary")
print("=" * 70)
print(f"  IDI significant (p<0.05): {idi_nri_df['IDI_Significant'].sum()} / {len(idi_nri_df)}")
print(f"  NRI significant (p<0.05): {idi_nri_df['NRI_Significant'].sum()} / {len(idi_nri_df)}")
print(f"  Both significant        : {(idi_nri_df['IDI_Significant'] & idi_nri_df['NRI_Significant']).sum()} / {len(idi_nri_df)}")

# Reproducibility checkpoint
# ── Expected: IDI and NRI values should match within ±0.0001 on re-run ──

# **Cell E — IDI & NRI: Fig 5 — Five-Panel Visualization (600 DPI)**

In [ ]:
# Cell 80─────────────────────────────────────────────────────────────────────────────
# Fig 5: IDI & NRI comparison — five-panel vertical layout
#   Panel A: IDI horizontal bar chart with 95% CI error bars
#   Panel B: NRI horizontal bar chart with 95% CI error bars
#   Panel C: ΔAUC horizontal bar chart
#   Panel D: IDI vs NRI scatter plot (alphabet-labelled markers)
#   Panel E: Legend table (alphabet → model name mapping)
# Color code: Red = both IDI & NRI significant | Orange = one significant
#             Gray = neither significant
# ─────────────────────────────────────────────────────────────────────────────

FIG8_FILENAME = 'Fig5_IDI_NRI_XGBoost_Standard_vs_Comparators.jpg'

fig = plt.figure(figsize=(12, 24))
gs  = fig.add_gridspec(5, 1,
                        height_ratios=[1, 1, 1, 1.2, 0.8],
                        hspace=0.45)

models_list  = idi_nri_df['Comparison_Model']
sig_color    = lambda row: (
    'red'    if (row['IDI_Significant'] and row['NRI_Significant']) else
    'orange' if (row['IDI_Significant'] or  row['NRI_Significant']) else
    'gray'
)

# ── Panel A: IDI ──────────────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
idi_vals = idi_nri_df['IDI']
idi_err  = 1.96 * idi_nri_df['IDI_SE']
bar_colors_idi = ['red' if p < 0.05 else 'gray'
                  for p in idi_nri_df['IDI_p_value']]

ax_a.barh(range(len(models_list)), idi_vals, xerr=idi_err,
          color=bar_colors_idi, alpha=0.7, capsize=5)
ax_a.set_yticks(range(len(models_list)))
ax_a.set_yticklabels(models_list, fontsize=11)
ax_a.set_xlabel('IDI (Integrated Discrimination Improvement)',
                fontsize=13, fontweight='bold')
ax_a.set_title('A. IDI Comparison: XGBoost Standard vs Comparator Models\n(Red: p<0.05)',
               fontsize=14, fontweight='bold')
ax_a.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
ax_a.grid(axis='x', alpha=0.3)

# ── Panel B: NRI ──────────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[1, 0])
nri_vals = idi_nri_df['NRI']
nri_err  = 1.96 * idi_nri_df['NRI_SE']
bar_colors_nri = ['red' if p < 0.05 else 'gray'
                  for p in idi_nri_df['NRI_p_value']]

ax_b.barh(range(len(models_list)), nri_vals, xerr=nri_err,
          color=bar_colors_nri, alpha=0.7, capsize=5)
ax_b.set_yticks(range(len(models_list)))
ax_b.set_yticklabels(models_list, fontsize=11)
ax_b.set_xlabel('NRI (Net Reclassification Improvement)',
                fontsize=13, fontweight='bold')
ax_b.set_title('B. NRI Comparison: XGBoost Standard vs Comparator Models\n(Red: p<0.05)',
               fontsize=14, fontweight='bold')
ax_b.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
ax_b.grid(axis='x', alpha=0.3)

# ── Panel C: ΔAUC ─────────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[2, 0])
dauc_vals  = idi_nri_df['Delta_AUC']
dauc_colors = ['green' if v > 0 else 'orange' for v in dauc_vals]

ax_c.barh(range(len(models_list)), dauc_vals, color=dauc_colors, alpha=0.7)
ax_c.set_yticks(range(len(models_list)))
ax_c.set_yticklabels(models_list, fontsize=11)
ax_c.set_xlabel('ΔAUC (XGBoost Standard − Comparison Model)',
                fontsize=13, fontweight='bold')
ax_c.set_title('C. AUC Difference: XGBoost Standard vs Comparator Models\n'
               '(Green: XGBoost Standard superior)',
               fontsize=14, fontweight='bold')
ax_c.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
ax_c.grid(axis='x', alpha=0.3)

# ── Panel D: IDI vs NRI scatter ───────────────────────────────────────────
ax_d = fig.add_subplot(gs[3, 0])
scatter_colors_d = [sig_color(idi_nri_df.loc[i])
                    for i in range(len(idi_nri_df))]

ax_d.scatter(idi_nri_df['IDI'], idi_nri_df['NRI'],
             c=scatter_colors_d, s=250, alpha=0.85,
             edgecolors='none', zorder=3)  # [FIXED] Removed black circle borders per request

for i, lbl in enumerate(idi_nri_df['Label']):
    ax_d.text(idi_nri_df.loc[i, 'IDI'], idi_nri_df.loc[i, 'NRI'],
              lbl, fontsize=13, ha='center', va='center',
              fontweight='bold', color='black', zorder=4)  # [FIXED] White → Black for visibility

ax_d.set_xlabel('IDI', fontsize=13, fontweight='bold')
ax_d.set_ylabel('NRI', fontsize=13, fontweight='bold')
ax_d.set_title('D. IDI vs NRI Scatter Plot\n'
               '(See legend table below for model identification)',
               fontsize=14, fontweight='bold')
ax_d.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax_d.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax_d.grid(True, alpha=0.3, linestyle=':', linewidth=0.8)

_xr = idi_nri_df['IDI'].max() - idi_nri_df['IDI'].min()
_yr = idi_nri_df['NRI'].max() - idi_nri_df['NRI'].min()
ax_d.set_xlim(idi_nri_df['IDI'].min() - 0.2 * _xr,
              idi_nri_df['IDI'].max() + 0.2 * _xr)
ax_d.set_ylim(idi_nri_df['NRI'].min() - 0.2 * _yr,
              idi_nri_df['NRI'].max() + 0.2 * _yr)

# ── Panel E: Legend table ─────────────────────────────────────────────────
ax_e = fig.add_subplot(gs[4, 0])
ax_e.axis('off')

N_COLS  = 3
n_items = len(idi_nri_df)
n_rows  = int(np.ceil(n_items / N_COLS))

table_data = []
for r in range(n_rows):
    row = []
    for c in range(N_COLS):
        idx = r + c * n_rows
        if idx < n_items:
            row.extend([idi_nri_df.loc[idx, 'Label'],
                        idi_nri_df.loc[idx, 'Comparison_Model']])
        else:
            row.extend(['', ''])
    table_data.append(row)

col_labels_e = ['ID', 'Model Name'] * N_COLS

tbl = ax_e.table(cellText=table_data, colLabels=col_labels_e,
                 cellLoc='left', loc='center',
                 colWidths=[0.04, 0.28] * N_COLS)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 2.5)

for j in range(len(col_labels_e)):                       # Header row
    cell = tbl[(0, j)]
    cell.set_facecolor('#4CAF50')
    cell.set_text_props(weight='bold', color='black')  # [FIXED] white→black for visibility, fontsize=11

for r in range(1, n_rows + 1):                           # Data rows
    for j in range(len(col_labels_e)):
        cell = tbl[(r, j)]
        if r % 2 == 0:
            cell.set_facecolor('#f0f0f0')
        if j % 2 == 0:
            cell.set_text_props(weight='bold', ha='center')

ax_e.set_title('E. Legend: Alphabet Label — Model Name Reference',
               fontsize=13, fontweight='bold', pad=10)
ax_e.text(0.5, -0.05,
          'Color code:  Red = Both IDI & NRI significant (p<0.05)  |  '
          'Orange = One significant  |  Gray = Neither significant',
          ha='center', va='top', fontsize=10,
          transform=ax_e.transAxes, style='italic')

# ── Save figure ───────────────────────────────────────────────────────────
plt.savefig(FIG8_FILENAME, dpi=600, bbox_inches='tight', facecolor='white',
            format='jpeg', pil_kwargs={'quality': 95})
print(f"✓ Fig 5 saved: {FIG8_FILENAME}  (600 DPI, 5-panel vertical, JPEG quality=95)")
plt.show()

# **Cell F — IDI & NRI: S15 Table Export**

In [ ]:
# Cell 82─────────────────────────────────────────────────────────────────────────────
# S15 Table: Discrimination and Reclassification Metrics
#            XGBoost Standard vs Comparator Models
# Export: CSV + S15 Table XLSX with in-cell display
# Reference: Pencina et al. Statistics in Medicine 2008; 2011.
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import display as _disp13
import pandas as pd, numpy as np
from google.colab import files as colab_files

IDI_NRI_CSV  = 'S15_Table_source_IDI_NRI_raw.csv'
S13_XLSX     = 'S15_Table_IDI_NRI_XGBoost_Standard_vs_Comparators.xlsx'

idi_nri_df.to_csv(IDI_NRI_CSV, index=False, encoding='utf-8-sig')
print(f'\u2713 Results table saved: {IDI_NRI_CSV}')

# ── Build S15 Table in publication format ───────────────────────────────────
def _p_fmt(v):
    if pd.isna(v): return 'N/A'
    return '<0.001' if float(v) < 0.001 else f'{float(v):.3f}'

def _ci_fmt(v, lo, hi):
    if pd.isna(v): return 'N/A'
    if pd.notna(lo) and pd.notna(hi):
        return f'{float(v):.4f} ({float(lo):.4f}, {float(hi):.4f})'
    return f'{float(v):.4f}'

rows_s13 = []
for _, row in idi_nri_df.iterrows():
    mname = row.get('Comparison_Model', row.get('Model', ''))
    # ΔAUC
    delta_auc = row.get('Delta_AUC', row.get('ΔAUC', row.get('dAUC', np.nan)))
    # IDI
    idi_v  = row.get('IDI', np.nan)
    idi_lo = row.get('IDI_CI_lower', row.get('IDI_lower', np.nan))
    idi_hi = row.get('IDI_CI_upper', row.get('IDI_upper', np.nan))
    idi_p  = row.get('IDI_p_value', row.get('IDI_p', np.nan))
    # NRI (total or categorical)
    nri_v  = row.get('NRI', row.get('NRI_Total', row.get('NRI_Categorical', np.nan)))
    nri_p  = row.get('NRI_p_value', row.get('NRI_p', np.nan))

    rows_s13.append({
        'Model'         : mname,
        '\u0394AUC\u2020': (f'{float(delta_auc):+.4f}' if pd.notna(delta_auc) else 'N/A'),
        'IDI (95% CI)'  : _ci_fmt(idi_v, idi_lo, idi_hi),
        'P value (IDI)' : _p_fmt(idi_p),
        'NRI\u00a7'     : (f'{float(nri_v):.4f}' if pd.notna(nri_v) else 'N/A'),
        'P value (NRI)' : _p_fmt(nri_p),
    })

df_s13 = pd.DataFrame(rows_s13)

print('\n' + '=' * 70)
print('S15 Table: Integrated Discrimination Improvement & Net Reclassification')
print('           XGBoost Standard vs Comparator Models')
print('=' * 70)
print('\n')
_disp13(df_s13)

with pd.ExcelWriter(S13_XLSX, engine='openpyxl') as writer:
    df_s13.to_excel(writer, sheet_name='S15_Table', index=False)
    idi_nri_df.to_excel(writer, sheet_name='Raw_IDI_NRI_Data', index=False)
print(f'\n\u2713 Saved: {S13_XLSX}')

# Summary model reference
print('\nModel reference (label \u2192 comparator):')
for _, row in idi_nri_df.iterrows():
    status = ('Both significant'
              if (row.get('IDI_Significant', False) and row.get('NRI_Significant', False))
              else 'IDI or NRI significant'
              if (row.get('IDI_Significant', False) or row.get('NRI_Significant', False))
              else 'Not significant')
    lbl = row.get('Label', '')
    cmp = row.get('Comparison_Model', row.get('Model', ''))
    print(f'  {lbl}: {cmp}  [{status}]')

# ── Download ─────────────────────────────────────────────────────────────────
for fpath in [IDI_NRI_CSV, S13_XLSX]:
    try:
        colab_files.download(fpath)
        print(f'  \u2713 Downloaded: {fpath}')
    except Exception as exc:
        print(f'  \u26a0  Failed: {fpath} \u2014 {exc}')

# [Reproducibility] ΔAUC = AUC(XGBoost Standard) − AUC(comparator); positive = XGBoost higher.
# IDI and NRI computed on common MLRA subset (listwise deletion, N\u2248common).
# NRI threshold: 10% and 20% risk categories (cardiovascular convention).


# **Cell G — IDI & NRI: Fig 5 Download**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Download: Fig 5 (JPG) and supplementary results table (CSV)
# ─────────────────────────────────────────────────────────────────────────────
for fpath in [FIG8_FILENAME]:   # the CSV is downloaded in the previous cell
    try:
        colab_files.download(fpath)
        print(f"✓ Downloaded: {fpath}")
        # [FIXED] Removed duplicate inline display here.
        # Fig 5 is already shown inline by plt.show() in the generating cell.
        # Re-displaying the same PNG in this download cell would produce redundant output.
    except Exception as exc:
        print(f"⚠  Download failed — {fpath}: {exc}")

print("\n" + "=" * 70)
print("Section 8-F complete.")
print("  Fig 5    : " + FIG8_FILENAME)
print("  Table    : " + IDI_NRI_CSV)
print("=" * 70)

## **Section 24 · Bootstrap Internal Validation (n = 500, Harrell's optimism correction)**

In [ ]:
# ============================================================
# Section 8 (Cell 88 replacement): Bootstrap internal validation
# Harrell's optimism correction (Harrell FE Jr, Lee KL, Mark DB. Stat Med 1996)
#
#   apparent_m       = metric(final model, development data)
#   optimism_i       = metric(model_i, bootstrap sample_i) - metric(model_i, development data)
#   optimism         = mean_i(optimism_i)
#   corrected_m      = apparent_m - optimism
#
# Models refitted in every bootstrap sample:
#   MLRA : coefficients of the 4 retained predictors re-estimated by unpenalized ML
#          (statsmodels Logit) in complete cases. Variable reduction and predictor
#          transformation are NOT repeated.
#   GB   : retrained with identical hyperparameters and native missing-data handling.
#          Hyperparameter optimization is NOT repeated.
#   -> Optimism from variable selection, transformation, and tuning is not captured.
# Apparent performance = final model predictions (all_results / MLRA formula).
# Threshold-dependent metrics: Youden threshold derived in each bootstrap sample.
# Scope: development cohort only; the validation cohort is never accessed.
# No confidence intervals are reported for corrected estimates (no standard method).
# ============================================================
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             accuracy_score, precision_score, f1_score)
from google.colab import files

N_BOOT  = 500
METRICS = ['auc', 'accuracy', 'sensitivity', 'specificity', 'precision', 'f1']

MLRA_TP, MLRA_DB = 'Total protein (g/dL)', 'Direct bilirubin (mg/dL)'
MLRA_WBC, MLRA_CRP = 'White blood cells (103/μL)', 'C-reactive protein (mg/dL)'
MLRA_COEF_FORMULA = np.array([2.053, -0.8872, 0.7692, 0.5063, 0.3229])  # from apply_mlra_formula


def find_optimal_threshold(y_true, y_proba):
    fpr, tpr, thr = roc_curve(np.asarray(y_true, float), np.asarray(y_proba, float))
    return float(thr[np.argmax(tpr - fpr)])


def _calc_metrics(y_true, y_proba, threshold):
    yt = np.asarray(y_true, int)
    yp = np.asarray(y_proba, float)
    if len(np.unique(yt)) < 2:
        return None
    y_pred = (yp >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, y_pred, labels=[0, 1]).ravel()
    return {'auc': roc_auc_score(yt, yp),
            'accuracy': accuracy_score(yt, y_pred),
            'sensitivity': tp / (tp + fn) if (tp + fn) else 0.0,
            'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
            'precision': precision_score(yt, y_pred, zero_division=0),
            'f1': f1_score(yt, y_pred, zero_division=0)}


def _mlra_design(df):
    X = pd.DataFrame({
        'TP'    : df[MLRA_TP].values,
        'logDB' : np.log(df[MLRA_DB].values  + EPSILON),
        'logWBC': np.log(df[MLRA_WBC].values + EPSILON),
        'logCRP': np.log(df[MLRA_CRP].values + EPSILON)})
    return sm.add_constant(X, has_constant='add')


def _fit_mlra(df):
    """Unpenalized ML logistic regression on the 4 retained predictors. None if not converged."""
    try:
        res = sm.Logit(df['Outcome'].values.astype(float), _mlra_design(df)).fit(disp=0, maxiter=200)
    except Exception:
        return None
    if not res.mle_retvals.get('converged', False):
        return None
    return res


SEED_KEYS = ('random_state', 'seed', 'random_seed', 'feature_fraction_seed',
             'bagging_seed', 'data_random_seed', 'drop_seed')


def _ml_params(model_class, model_params, seed):
    """Final-model hyperparameters (Section 6) with only the existing seed keys replaced.
    With seed = RANDOM_STATE, the parameters are identical to those of the final model."""
    p = model_params.copy()
    for k in SEED_KEYS:
        if k in p:
            p[k] = seed
    if 'CatBoost' in str(model_class):
        p['allow_writing_files'] = False   # file output only; no effect on the model
    return p


def harrell_bootstrap(name, n_boot=N_BOOT):
    """Returns per-iteration optimism arrays and summary for one model."""
    t0 = time.time()
    is_mlra = (name == 'MLRA')

    # ---- Apparent performance: FINAL model predictions ----
    if is_mlra:
        data = train_mlra.reset_index(drop=True)
        y = data['Outcome'].values.astype(int)
        p_app = np.asarray(apply_mlra_formula(data), float)
        # Check (definition): ML refit on the full complete-case data reproduces the final formula
        full_fit = _fit_mlra(data)
        assert full_fit is not None, 'MLRA refit on full data did not converge'
        p_refit = full_fit.predict(_mlra_design(data))
        assert np.max(np.abs(p_refit - p_app)) < 1e-3, \
            f'MLRA refit differs from final formula (max |dp| = {np.max(np.abs(p_refit - p_app)):.2e})'
        print(f'  MLRA full-data refit coefficients: {np.round(full_fit.params.values, 4)} '
              f'(formula: {MLRA_COEF_FORMULA})')
        n_s = len(data)
    else:
        model_cls, params = ML_CLASS_MAP[name]
        X = X_train.reset_index(drop=True) if hasattr(X_train, 'reset_index') else X_train
        y = np.asarray(y_train, int)
        p_app = np.asarray(all_results[name]['train_proba'], float)
        assert len(p_app) == len(y) == 756
        # Check (final-model identity): refit with the bootstrap settings on full data
        # must reproduce the final model's predictions.
        set_global_seed(RANDOM_STATE)
        mdl_full = model_cls(**_ml_params(model_cls, params, RANDOM_STATE))
        mdl_full.fit(X, y)
        dmax = np.max(np.abs(mdl_full.predict_proba(X)[:, 1] - p_app))
        assert dmax < 1e-6, (f'{name}: refit with bootstrap settings does not reproduce the final '
                             f'model (max |dp| = {dmax:.2e}); align _ml_params with Section 7.')
        n_s = len(X)

    thr_app = find_optimal_threshold(y, p_app)
    apparent = _calc_metrics(y, p_app, thr_app)
    if is_mlra:
        assert abs(apparent['auc'] - roc_auc_score(all_results['MLRA']['train_true'],
                                                   all_results['MLRA']['train_proba'])) < 1e-9

    # ---- Bootstrap iterations ----
    rng = np.random.RandomState(RANDOM_STATE)
    idx_list = [rng.choice(n_s, size=n_s, replace=True) for _ in range(n_boot)]
    opt = {m: [] for m in METRICS}
    n_skip = 0
    for i, idx in enumerate(idx_list):
        seed = RANDOM_STATE + i
        set_global_seed(seed)
        y_b = y[idx]
        if len(np.unique(y_b)) < 2:
            n_skip += 1; continue
        if is_mlra:
            d_b = data.iloc[idx].reset_index(drop=True)
            fit_b = _fit_mlra(d_b)
            if fit_b is None:
                n_skip += 1; continue
            p_b = fit_b.predict(_mlra_design(d_b))
            p_o = fit_b.predict(_mlra_design(data))
        else:
            X_b = X.iloc[idx] if hasattr(X, 'iloc') else X[idx]
            mdl = model_cls(**_ml_params(model_cls, params, seed))
            mdl.fit(X_b, y_b)
            p_b = mdl.predict_proba(X_b)[:, 1]
            p_o = mdl.predict_proba(X)[:, 1]
        thr_b = find_optimal_threshold(y_b, p_b)
        m_b = _calc_metrics(y_b, p_b, thr_b)
        m_o = _calc_metrics(y, p_o, thr_b)
        if m_b is None or m_o is None:
            n_skip += 1; continue
        for m in METRICS:
            opt[m].append(m_b[m] - m_o[m])

    summary = {'Model': name, 'N': int(n_s), 'Events': int(y.sum()),
               'Valid iterations': n_boot - n_skip}
    for m in METRICS:
        o = np.asarray(opt[m])
        corrected = apparent[m] - o.mean()
        # Check (definition): corrected = apparent - mean(boot_i - orig_i)
        assert np.isclose(corrected, apparent[m] - np.mean(o))
        summary[f'{m}_apparent'] = apparent[m]
        summary[f'{m}_optimism'] = o.mean()
        summary[f'{m}_corrected'] = corrected
    print(f'  {name}: {n_boot - n_skip}/{n_boot} valid iterations, '
          f'AUC apparent {apparent["auc"]:.4f}, optimism {summary["auc_optimism"]:.4f}, '
          f'corrected {summary["auc_corrected"]:.4f} ({time.time() - t0:.1f}s)')
    return summary


ML_CLASS_MAP = {
    'XGBoost_Standard'    : (xgb.XGBClassifier,             xgb_standard_params),
    'XGBoost_DART'        : (xgb.XGBClassifier,             xgb_dart_params),
    'LightGBM_GBDT'       : (lgb.LGBMClassifier,            lgb_gbdt_params),
    'LightGBM_DART'       : (lgb.LGBMClassifier,            lgb_dart_params),
    'LightGBM_GOSS'       : (lgb.LGBMClassifier,            lgb_goss_params),
    'CatBoost_Standard'   : (cb.CatBoostClassifier,         catboost_standard_params),
    'CatBoost_Ordered'    : (cb.CatBoostClassifier,         catboost_ordered_params),
    'HistGradientBoosting': (HistGradientBoostingClassifier, hist_gb_params),
}

assert len(train_mlra) == 664 and int(train_mlra['Outcome'].sum()) == 45

print('=' * 70)
print("Bootstrap internal validation -- Harrell's optimism correction (n = 500)")
print('=' * 70)
set_global_seed(RANDOM_STATE)
boot_rows = [harrell_bootstrap('MLRA')]
for _name in ML_CLASS_MAP:
    boot_rows.append(harrell_bootstrap(_name))
boot_df = pd.DataFrame(boot_rows)

# ---- S12 Table ----
s12 = pd.DataFrame({
    'Model'                         : boot_df['Model'].str.replace('_', ' '),
    'N'                             : boot_df['N'],
    'Events'                        : boot_df['Events'],
    'Apparent AUC'                  : boot_df['auc_apparent'].map('{:.3f}'.format),
    'Optimism'                      : boot_df['auc_optimism'].map('{:.3f}'.format),
    'Optimism-corrected AUC'        : boot_df['auc_corrected'].map('{:.3f}'.format),
    'Corrected accuracy'            : boot_df['accuracy_corrected'].map('{:.3f}'.format),
    'Corrected sensitivity'         : boot_df['sensitivity_corrected'].map('{:.3f}'.format),
    'Corrected specificity'         : boot_df['specificity_corrected'].map('{:.3f}'.format),
    'Corrected PPV'                 : boot_df['precision_corrected'].map('{:.3f}'.format),
    'Corrected F1-score'            : boot
    _df['f1_corrected'].map('{:.3f}'.format),
})
s12 = s12.iloc[np.argsort(-boot_df['auc_corrected'].values)].reset_index(drop=True)
print('\nS12 Table: Bootstrap internal validation (development cohort)')
from IPython.display import display as _disp88
_disp88(s12)

# Independent check: apparent AUC of each GB model equals the final-model AUC in Table 4 source
for _name in ML_CLASS_MAP:
    _a = boot_df.loc[boot_df['Model'] == _name, 'auc_apparent'].item()
    assert abs(_a - roc_auc_score(y_train, all_results[_name]['train_proba'])) < 1e-9

S12_XLSX, S12_CSV = 'S12_Table_bootstrap_internal_validation.xlsx', 'S12_Table_bootstrap_internal_validation.csv'
with pd.ExcelWriter(S12_XLSX, engine='openpyxl') as w:
    s12.to_excel(w, sheet_name='S12_Table', index=False)
    boot_df.to_excel(w, sheet_name='Unrounded', index=False)
s12.to_csv(S12_CSV, index=False, encoding='utf-8-sig')
files.download(S12_XLSX)
files.download(S12_CSV)
print(f'\n✓ Saved: {S12_XLSX}, {S12_CSV}')

## Final · Export of manuscript figures as PLOS ONE TIFF files

In [ ]:
# ============================================================
# Final cell: export all manuscript figures as PLOS ONE TIFF files
#   RGB (no alpha), LZW compression, 300 dpi tag,
#   fitted within 2,250 x 2,625 px (7.5 x 8.75 in at 300 dpi), aspect ratio preserved.
# Fig 8 is already saved as Fig8.tif by its own cell.
# PACE (https://pacev2.apexcovantage.com) should still be used for the final check.
# ============================================================
import os
from PIL import Image
from google.colab import files

PLOS_DPI = 300
MAX_W_PX, MAX_H_PX = 2250, 2625          # 7.5 x 8.75 in at 300 dpi
MIN_W_PX = 789                           # 2.63 in at 300 dpi

EXPORT = [                               # (source file in this notebook, PLOS file name)
    ('Fig3_ROC_curves.jpg',                              'Fig3.tif'),
    ('Fig4_Calibration_plots.jpg',                       'Fig4.tif'),
    ('Fig5_IDI_NRI_XGBoost_Standard_vs_Comparators.jpg', 'Fig5.tif'),
    ('Fig6_DCA_XGBoost_Standard_vs_MLRA.jpg',            'Fig6.tif'),
    ('Fig7_Sensitivity_Analyses_MLRA_vs_XGBoost.png',    'Fig7.tif'),
    ('S3_Fig_Missing_Data_Patterns.jpg',                 'S3_Fig.tif'),
    ('S6_Fig_Subgroup_Forest_Plot.jpg',                  'S6_Fig.tif'),
    ('S7_Fig_Imputation_Comparison.jpg',                 'S7_Fig.tif'),
    ('S8_Fig_Integrated_Fairness.jpg',                   'S8_Fig.tif'),
    ('S9_Fig_SHAP_dependence_XGBoost_Standard.jpg',      'S9_Fig.tif'),
]

for src, dst in EXPORT:
    if not os.path.exists(src):
        print(f'  ✗ Not found (run the generating cell first): {src}')
        continue
    _im = Image.open(src)
    src_dpi = (_im.info.get('dpi') or (600, 600))[0]
    img = _im.convert('RGB')
    w, h = img.size
    src_w_in = w / src_dpi                                  # width at which the text was drawn
    scale = min(MAX_W_PX / w, MAX_H_PX / h, 1.0)
    new = (round(w * scale), round(h * scale))
    if scale < 1.0:
        img = img.resize(new, Image.LANCZOS)
    img.save(dst, format='TIFF', compression='tiff_lzw', dpi=(PLOS_DPI, PLOS_DPI))
    w_in, h_in = new[0] / PLOS_DPI, new[1] / PLOS_DPI
    shrink = w_in / src_w_in                                # text is printed at this fraction of its size
    note = (f'  ← text printed at {shrink:.0%} of its drawn size: fonts drawn at 10 pt become '
            f'{10 * shrink:.1f} pt (PLOS: 8–12 pt); redraw at 7.5 in width' if shrink < 0.8 else '')
    print(f'  ✓ {dst}: {new[0]} x {new[1]} px ({w_in:.2f} x {h_in:.2f} in){note}')
    files.download(dst)